# Foveate on Kaggle: profile VLM inference, then spend compute only where it is needed

**Before you run:** *Settings → Accelerator → GPU T4 x2* and *Settings → Internet → On*.
Then set `MODE` below and use *Run All*. Results land in `/kaggle/working/foveate_results`
(kept after the session ends) and are zipped at the end for download.

| Stage | What it does | Theory |
|---|---|---|
| 1 profile | where batch-1 latency goes (decode, ViT, prefill, decode) | `theory/01` |
| 2 grid | every question × every frames/resolution/model config, cached | `theory/02` |
| 3 replay | static vs rules vs cascade vs oracle, held-out | `theory/03` |
| 4 shift | fog/night/blur: does the cascade spend more on harder inputs? | `theory/04` |
| 5 serve | vLLM (3B on GPU0, 7B-AWQ on GPU1) + gateway under Poisson load | `theory/05` |

`MODE = "pilot"` runs a fixed-seed 30% of the questions (stratified by task, same
questions every time). `MODE = "full"` runs everything. Every GPU step is resumable:
if the session dies, re-run the notebook and finished questions are skipped (as long
as `/kaggle/working/foveate_results` was kept, e.g. via *Save Version*).

In [ ]:
# ---- settings -------------------------------------------------------------
MODE = "pilot"          # "pilot" (fixed 30%) or "full"
TASKS = "auto"          # "all", "lite" (skips the 11 GB Perception zip), or "auto" (lite if disk < 45 GB free)
RUN_PROFILE = True
RUN_7B = True           # the escalation model (needs vLLM)
RUN_SHIFT = True        # stage 4; the longest step in full mode
RUN_LINGOQA = True      # driving demo (Google Drive download; skipped on failure)
RUN_SERVING = True      # stage 5
GPUS = None             # None = all visible GPUs

import os, shutil, subprocess
WORK = "/kaggle/working/foveate" if os.path.isdir("/kaggle") else os.path.abspath("foveate")
os.environ["FOVEATE_RESULTS"] = "/kaggle/working/foveate_results" if os.path.isdir("/kaggle") else os.path.abspath("foveate_results")
os.environ["FOVEATE_DATA"] = "/tmp/foveate_data"        # big, not kept after the session
os.environ["FOVEATE_CACHE"] = "/tmp/foveate_data/frame_cache"
os.makedirs(os.environ["FOVEATE_DATA"], exist_ok=True)
free_gb = shutil.disk_usage("/tmp").free / 2**30
if TASKS == "auto":
    TASKS = "all" if free_gb > 45 else "lite"
if GPUS is None:
    GPUS = max(1, len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()))
print(f"MODE={MODE} TASKS={TASKS} GPUS={GPUS} free disk in /tmp: {free_gb:.0f} GB")

In [ ]:
# Optional: a Hugging Face token from Kaggle secrets (Add-ons -> Secrets, name HF_TOKEN)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception:
    print("no HF token (fine for public models and MVBench)")

## 0 · Write the package and install it
The next cell writes the Foveate source (identical to the GitHub repo) into `/kaggle/working/foveate`.

In [ ]:
FILES = {
"pyproject.toml": "[build-system]\nrequires = [\"setuptools>=68\", \"wheel\"]\nbuild-backend = \"setuptools.build_meta\"\n\n[project]\nname = \"foveate\"\nversion = \"0.1.0\"\ndescription = \"Profile VLM inference and spend visual compute (frames, resolution, model size) only where the question needs it.\"\nreadme = \"README.md\"\nrequires-python = \">=3.10\"\nlicense = { text = \"MIT\" }\nauthors = [{ name = \"Vivek Gangwani\" }]\nkeywords = [\"vlm\", \"inference\", \"vllm\", \"video-qa\", \"profiling\", \"cascade\"]\n# Core deps are CPU-only so tests, replay and reports run anywhere.\ndependencies = [\n  \"numpy>=1.26\",\n  \"pandas>=2.1\",\n  \"pyarrow>=14\",\n  \"pillow>=10\",\n  \"pyyaml>=6\",\n  \"matplotlib>=3.8\",\n  \"tqdm>=4.66\",\n]\n\n[project.optional-dependencies]\n# Video decoding + dataset download.\ndata = [\"av>=12\", \"huggingface_hub>=0.24\", \"gdown>=5\"]\n# Stage-level profiler (Transformers backend).\nhf = [\"torch>=2.4\", \"transformers>=4.51\", \"accelerate>=0.33\"]\n# Serving gateway and load generator (talks to `vllm serve`).\nserve = [\"fastapi>=0.110\", \"uvicorn>=0.29\", \"httpx>=0.27\"]\n# vLLM is installed separately on the GPU box (see requirements-gpu.txt),\n# because it pins its own torch build.\ndev = [\"pytest>=8\", \"ruff>=0.5\"]\n\n[project.scripts]\nfoveate = \"foveate.cli:main\"\n\n[tool.setuptools.packages.find]\nwhere = [\"src\"]\n\n[tool.setuptools.package-data]\nfoveate = [\"configs/*.yaml\"]\n\n[tool.pytest.ini_options]\ntestpaths = [\"tests\"]\n\n[tool.ruff]\nline-length = 100\n",
"README.md": "# Foveate\n\n**Spend visual compute only where the question needs it.**\n\nYour eye's fovea sees sharply only where you look, and your brain moves it only when\nsomething needs a closer look. Foveate does the same for vision-language models: it\nanswers every video question cheaply first (few frames, low resolution, small model)\nand escalates to more frames, more pixels or a bigger model only when the cheap\nanswer looks unsure. It also measures where VLM inference time actually goes, so\nevery decision rests on profiler numbers rather than guesses.\n\n```\n            ┌──────────── Stage 1: profile ────────────┐\n question   │ load │ preprocess │ ViT │ prefill │ decode │  where does the time go?\n + video ──►└───────────────────────────────────────────┘\n            Stage 2: grid    every question × every (frames, resolution, model) config, cached\n            Stage 3: policy  static vs rules vs confidence cascade vs oracle, held-out evaluation\n            Stage 4: shift   fog / night / blur: does the cascade spend more where inputs get harder?\n            Stage 5: serve   gateway + vLLM (one model per GPU) under Poisson load: p50/p95/p99\n```\n\n| | |\n|---|---|\n| Models | Qwen2.5-VL-3B-Instruct (cheap), Qwen2.5-VL-7B-Instruct-AWQ (escalation) |\n| Main benchmark | **MVBench**: 19 temporal tasks, 3,800 multiple-choice questions (published Qwen2.5-VL scores: 3B 67.0, 7B 69.6) |\n| Driving demo | **LingoQA** evaluation set: 500 driving-scene questions, scored with Lingo-Judge |\n| Target hardware | Kaggle **2× T4** (16 GB each, fp16). Any CUDA GPU works. |\n| Engines | vLLM for throughput and serving; Hugging Face Transformers for per-stage timing |\n\n**Why MVBench:** it is in the video table of almost every recent VLM paper, so your\nharness can be checked against a published number, and each answer is a single\noption letter, so a full pass over one config takes minutes. See\n[theory/02](theory/02_benchmarks_confidence_and_statistics.md).\n\n---\n\n## Two ways to run it\n\n1. **Kaggle notebook (easiest):** open [`notebooks/foveate_kaggle.ipynb`](notebooks/foveate_kaggle.ipynb)\n   on Kaggle, set the accelerator to **GPU T4 x2** and **Internet on**, choose\n   `MODE = \"pilot\"` or `\"full\"` in the first cell, and *Run All*. The notebook embeds\n   the whole package, so it needs nothing else.\n2. **This repo (any GPU machine):** the commands below.\n\n### Pilot vs full\n\nEvery command that touches data takes `--pilot` (default) or `--full`.\n**Pilot = a fixed-seed 30% of the questions, stratified by task** (≈1,140 MVBench\nquestions, 150 LingoQA questions). The same seed always selects the same questions,\nso pilot runs on different days are directly comparable.\n\n---\n\n## Quick start (repo)\n\n```bash\ngit clone https://github.com/<you>/foveate && cd foveate\nbash scripts/setup_gpu.sh          # picks a vLLM build that matches your NVIDIA driver\nfoveate env                        # GPU, driver, library versions\n\n# Everything, end to end (pilot, both GPUs):\nMODE=pilot bash scripts/run_all.sh\n# ...then read results/REPORT.md and results/figures/\n```\n\nData goes to `./data` (override with `FOVEATE_DATA`), results to `./results`\n(`FOVEATE_RESULTS`), decoded-frame cache to `data/frame_cache` (`FOVEATE_CACHE`).\n\n### Stage by stage\n\n```bash\n# Stage 1: batch-1 stage timing (synthetic frames: no dataset needed)\nfoveate profile --model 3b --source synthetic\n\n# Stage 2: data + quality grid (one process per GPU with --gpus 2)\nfoveate download mvbench --tasks all          # ~17 GB; --tasks lite skips the 11 GB Perception zip\nfoveate grid --dataset mvbench --pilot --model 3b --gpus 2\nfoveate grid --dataset mvbench --pilot --model 7b --gpus 2\n\n# Stage 4 data: rerun the ladder on fog / night / blur copies\nfoveate shift --dataset mvbench --pilot --model 3b --gpus 2\nfoveate shift --dataset mvbench --pilot --model 7b --gpus 2\n\n# Stages 3-4: replay every policy from the cache (CPU, seconds)\nfoveate replay --dataset mvbench --pilot\n\n# Driving demo\nfoveate download lingoqa\nfoveate grid  --dataset lingoqa --pilot --model 3b --gpus 2\nfoveate grid  --dataset lingoqa --pilot --model 7b --gpus 2\nfoveate judge --dataset lingoqa --pilot\nfoveate replay --dataset lingoqa --pilot\n\n# Stage 5: vLLM servers (3B on GPU 0, 7B-AWQ on GPU 1) + gateway + load test\nMODE=pilot bash scripts/serve_and_load.sh\n\n# Figures and the summary\nfoveate report\n```\n\nEvery grid run is **resumable**: if a Kaggle session dies, rerun the same command\nand it skips questions already in the output files.\n\n### Rough GPU time on Kaggle 2× T4\n\nThese are planning estimates from FLOP arithmetic, not measurements. Your Stage 1\nprofile replaces them with real numbers.\n\n| Step | Pilot (30%) | Full |\n|---|---|---|\n| setup + MVBench download | 20–30 min | same |\n| profile | ~15 min | same |\n| grid, 3B (7 configs) | ~45 min | ~2.5 h |\n| grid, 7B-AWQ (2 configs) | ~25 min | ~1.5 h |\n| shift (3 corruptions × 3 configs) | ~60 min | ~3.5 h |\n| LingoQA (both models + judge) | ~10 min | ~30 min |\n| serving load tests | ~30 min | same |\n\nKaggle sessions stop after 12 hours and give 30 GPU-hours a week, so run the pilot\nfirst, then the full grid, and the full shift study in a second session if needed.\nSet `RUN_LINGOQA=0`, `RUN_SERVING=0` or `TASKS=lite` to trim.\n\n---\n\n## What you get\n\n`results/REPORT.md` with tables and figures, all generated from result files:\n\n- **Stage 1:** latency breakdown per (frames, resolution); TTFT vs visual tokens with a\n  linear fit; prefill vs decode; achieved TFLOP/s vs the GPU's spec peak.\n- **Stage 2–3:** accuracy (bootstrap 95% CI) vs cost for every static config, the\n  learned rules, the cascade, rules+cascade and the oracle, on a held-out half, with\n  paired accuracy differences; confidence AUROC/ECE per config; per-task heatmap;\n  ablations.\n- **Stage 4:** escalation rate and accuracy clean vs shifted; a clean→fog stream with\n  a drift alarm; cost of a drift-aware cascade.\n- **Stage 5:** throughput, p50/p95/p99 latency and TTFT for the cascade vs the best\n  fixed config under the same Poisson trace.\n\n## Theory, one note per stage\n\n1. [Anatomy of VLM inference](theory/01_anatomy_of_vlm_inference.md): visual tokens, prefill vs decode, roofline, KV cache, timing correctly, T4 caveats\n2. [Benchmarks, confidence and statistics](theory/02_benchmarks_confidence_and_statistics.md): MVBench, logit scoring, AUROC/ECE, pilot precision, paired bootstrap\n3. [Adaptive compute and cascades](theory/03_adaptive_compute_and_cascades.md): the cost model, held-out thresholds, rules, oracle gap, related work\n4. [Distribution shift and drift](theory/04_distribution_shift_and_drift.md): covariate shift, a free detector, why drift ≠ failure\n5. [Serving VLMs under load](theory/05_serving_vlms_under_load.md): continuous batching, PagedAttention, prefix/encoder caches, queueing, goodput\n\n## Repository layout\n\n```\nsrc/foveate/\n  tokens.py            visual-token, FLOP and KV-cache arithmetic (+ GPU spec table)\n  data/                MVBench, LingoQA, synthetic loaders; frame sampling/cache; corruptions\n  backends/            hf_backend.py (stage timing via CUDA events), vllm_backend.py (fast path)\n  profile.py           Stage 1 sweep          grid.py     Stage 2 runner (sharded, resumable)\n  policy.py            Stage 3 policies       drift.py    Stage 4 shift + stream detector\n  serve/               Stage 5 gateway (FastAPI) and Poisson load generator\n  report.py            figures + REPORT.md    cli.py      `foveate ...`\n  configs/default.yaml models, grids, ladder, shift and profile settings\ntheory/                one note per stage\nnotebooks/             the self-contained Kaggle notebook (built by tools/build_notebook.py)\ntests/                 CPU tests, incl. a tiny random Qwen2.5-VL and a fake-model pipeline\n```\n\nTo change what runs, edit `src/foveate/configs/default.yaml` or pass\n`--config my.yaml`. Configs are written `MODEL:FRAMESxRES`, e.g. `3b:8x448`.\n\n## Troubleshooting\n\n- **vLLM will not install or start.** The grid's default `--backend auto` falls back to\n  Transformers (batch 1, several times slower, same results format). The 7B rung needs\n  vLLM because it is an AWQ checkpoint; without it, run the 3B grid only and the\n  cascade uses the 3B rungs. You can force a vLLM version with\n  `VLLM_OVERRIDE=\"vllm==0.11.0\" bash scripts/setup_gpu.sh`.\n- **NaN confidences or one repeated answer.** That is fp16 overflow on a GPU without\n  bf16. Rerun with `--dtype float32 --tp 2` (3B only; it then spans both T4s).\n- **Out of memory at start-up.** Add `--enforce-eager` and/or `--gpu-mem 0.85`.\n- **LingoQA download fails.** Google Drive rate-limits shared folders. Download the\n  evaluation folder manually from the link in the\n  [LingoQA repo](https://github.com/wayveai/lingoqa) into `data/lingoqa/` and unzip it.\n- **Disk.** MVBench needs ~17 GB after extraction (zips are deleted as they are\n  extracted). Use `--tasks lite` (~6 GB) when space is tight.\n\n## Honest limits\n\n- MVBench accuracy at 4–16 frames will sit below Qwen's published 67.0 / 69.6, which\n  used up to 768 frames. The comparison that matters is between policies on the same\n  questions.\n- One NTU-based MVBench task (`fine_grained_pose`) is not redistributable and is skipped.\n- LingoQA scoring depends on a learned judge.\n- Synthetic corruptions are a robustness study, not evidence of on-road safety. This is\n  not a driving system.\n- Novelty: cascades, routing and adaptive frame selection all have prior work (see\n  theory/03). The contribution here is a measured, held-out cost/accuracy trade-off on a\n  real serving stack.\n\n## Licences\n\nCode: MIT. MVBench is distributed under MIT on the Hugging Face Hub (its source\nvideos keep their original licences). LingoQA has its own licence; read it before\nredistributing anything derived from it. Model weights follow the Qwen licences.\n",
"LICENSE": "MIT License\n\nCopyright (c) 2026 Vivek Gangwani\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the \"Software\"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED \"AS IS\", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n",
"src/foveate/__init__.py": "\"\"\"Foveate: profile VLM inference and spend visual compute only where the question needs it.\"\"\"\n\n__version__ = \"0.1.0\"\n",
"src/foveate/backends/__init__.py": "",
"src/foveate/backends/hf_backend.py": "\"\"\"Transformers backend: batch-1 inference with per-stage GPU timing.\n\nvLLM is faster but does not expose per-request vision-encoder time. Here we\nown the model object, so we can bracket the vision tower with CUDA events\n(forward pre/post hooks) and timestamp every generation step with a logits\nprocessor, which runs once per generated token, right after each forward pass.\n\n    start ─ H2D ─┬─ [vision encoder] ─┬─ LLM prefill ─ step1 ─ decode ... ─ stepN\n                 v0                    v1               (TTFT on GPU)\n\nTimes come from CUDA events, which are recorded in the GPU stream, so they\nmeasure GPU execution, not Python dispatch. We synchronize once per request,\nafter generation, never inside the timed region.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport math\nimport time\nfrom dataclasses import dataclass\n\nfrom ..data import Sample\nfrom ..data.frames import FrameBatch\nfrom ..prompts import chat_messages, mc_summary, option_letters, softmax_over_options\n\nlog = logging.getLogger(__name__)\n\n# Frames arrive pre-resized to multiples of 28 px, so the processor must not\n# resize again: allow everything from one token to a huge frame, no re-sampling.\nNO_RESIZE_VIDEO_KWARGS = {\n    \"size\": {\"shortest_edge\": 28 * 28, \"longest_edge\": 28 * 28 * 16384},\n    \"do_sample_frames\": False,\n    \"cap_pixels_per_frame\": False,\n}\n\n\n@dataclass\nclass StageTiming:\n    preprocess_ms: float\n    h2d_ms: float\n    vision_ms: float\n    prefill_ms: float\n    ttft_gpu_ms: float\n    decode_ms: float\n    tpot_ms: float\n    gpu_total_ms: float\n    out_tokens: int\n    prompt_tokens: int\n    visual_tokens: int\n    peak_mem_gb: float\n\n\nclass HFBackend:\n    def __init__(\n        self,\n        model_id: str,\n        dtype: str = \"auto\",\n        attn_implementation: str = \"sdpa\",\n        device: str = \"cuda\",\n        model=None,\n        processor=None,\n    ):\n        import torch\n        from transformers import AutoProcessor\n\n        from ..utils import default_dtype\n\n        self.torch = torch\n        self.device = device\n        self.dtype_name = default_dtype() if dtype == \"auto\" else dtype\n        torch_dtype = getattr(torch, self.dtype_name)\n        if model is None:\n            from transformers import Qwen2_5_VLForConditionalGeneration\n\n            kw = dict(attn_implementation=attn_implementation, device_map=device)\n            try:\n                model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n                    model_id, dtype=torch_dtype, **kw)\n            except TypeError:  # transformers < 4.56 calls it torch_dtype\n                model = Qwen2_5_VLForConditionalGeneration.from_pretrained(\n                    model_id, torch_dtype=torch_dtype, **kw)\n        self.model = model.eval()\n        self.processor = processor or AutoProcessor.from_pretrained(model_id)\n        self.tok = self.processor.tokenizer\n        self.video_token_id = self.processor.video_token_id\n        inner = getattr(self.model, \"model\", self.model)\n        self.visual = getattr(inner, \"visual\", None) or getattr(self.model, \"visual\")\n        self._events: dict[str, list] = {\"v0\": [], \"v1\": [], \"steps\": []}\n        self.visual.register_forward_pre_hook(lambda m, a: self._mark(\"v0\"))\n        self.visual.register_forward_hook(lambda m, a, o: self._mark(\"v1\"))\n        self.letter_ids = {}\n        for L in \"ABCDEFGH\":\n            ids = self.tok.encode(L, add_special_tokens=False)\n            if len(ids) == 1:\n                self.letter_ids[L] = ids[0]\n\n    # -- timing helpers ---------------------------------------------------\n    @property\n    def _cuda(self) -> bool:\n        return self.device.startswith(\"cuda\") and self.torch.cuda.is_available()\n\n    def _event(self):\n        if self._cuda:\n            e = self.torch.cuda.Event(enable_timing=True)\n            e.record()\n            return e\n        return time.perf_counter()  # CPU fallback (tests)\n\n    @staticmethod\n    def _ms(a, b) -> float:\n        if isinstance(a, float):\n            return (b - a) * 1e3\n        return a.elapsed_time(b)\n\n    def _mark(self, key: str) -> None:\n        self._events[key].append(self._event())\n\n    def _step_processor(self):\n        backend = self\n\n        class _StepTimer:  # a LogitsProcessor that only timestamps\n            def __call__(self, input_ids, scores):\n                backend._mark(\"steps\")\n                return scores\n\n        return _StepTimer()\n\n    # -- inputs -------------------------------------------------------------\n    def build_inputs(self, s: Sample, fb: FrameBatch):\n        text = self.processor.apply_chat_template(\n            chat_messages(s), tokenize=False, add_generation_prompt=True\n        )\n        try:\n            return self.processor(\n                text=[text], videos=[fb.frames], video_metadata=[fb.meta],\n                videos_kwargs=NO_RESIZE_VIDEO_KWARGS, return_tensors=\"pt\",\n            )\n        except (TypeError, ValueError):\n            # Older transformers: no video_metadata / size dict; pass pixel bounds and fps.\n            fps = len(fb.frames) / fb.meta[\"duration\"] if fb.meta.get(\"duration\") else 2.0\n            return self.processor(\n                text=[text], videos=[fb.frames], fps=fps, return_tensors=\"pt\",\n                min_pixels=28 * 28, max_pixels=28 * 28 * 16384,\n            )\n\n    # -- profiling ----------------------------------------------------------\n    def profile_one(self, s: Sample, fb: FrameBatch, out_tokens: int = 32) -> StageTiming:\n        torch = self.torch\n        for k in self._events:\n            self._events[k].clear()\n        if self._cuda:\n            torch.cuda.reset_peak_memory_stats()\n        t0 = time.perf_counter()\n        inputs = self.build_inputs(s, fb)\n        pre_ms = (time.perf_counter() - t0) * 1e3\n        if self._cuda:\n            torch.cuda.synchronize()\n        e0 = self._event()\n        inputs = inputs.to(self.model.device)\n        e_h2d = self._event()\n        with torch.inference_mode():\n            out = self.model.generate(\n                **inputs, max_new_tokens=out_tokens, min_new_tokens=out_tokens,\n                do_sample=False, logits_processor=[self._step_processor()],\n                pad_token_id=self.tok.pad_token_id or self.tok.eos_token_id,\n            )\n        e_end = self._event()\n        if self._cuda:\n            torch.cuda.synchronize()\n        steps, v0, v1 = self._events[\"steps\"], self._events[\"v0\"], self._events[\"v1\"]\n        vision = self._ms(v0[0], v1[-1]) if v0 and v1 else float(\"nan\")\n        ttft_gpu = self._ms(e0, steps[0])\n        h2d = self._ms(e0, e_h2d)\n        decode = self._ms(steps[0], steps[-1]) if len(steps) > 1 else 0.0\n        n_out = int(out.shape[1] - inputs[\"input_ids\"].shape[1])\n        ids = inputs[\"input_ids\"]\n        return StageTiming(\n            preprocess_ms=pre_ms,\n            h2d_ms=h2d,\n            vision_ms=vision,\n            prefill_ms=ttft_gpu - h2d - (0 if math.isnan(vision) else vision),\n            ttft_gpu_ms=ttft_gpu,\n            decode_ms=decode,\n            tpot_ms=decode / (len(steps) - 1) if len(steps) > 1 else float(\"nan\"),\n            gpu_total_ms=self._ms(e0, e_end),\n            out_tokens=n_out,\n            prompt_tokens=int(ids.shape[1]),\n            visual_tokens=int((ids == self.video_token_id).sum()),\n            peak_mem_gb=(torch.cuda.max_memory_allocated() / 2**30) if self._cuda else 0.0,\n        )\n\n    # -- prediction (fallback grid backend; vLLM is the fast path) ----------\n    def predict(self, s: Sample, fb: FrameBatch, max_new_tokens: int = 48) -> dict:\n        torch = self.torch\n        inputs = self.build_inputs(s, fb).to(self.model.device)\n        n_prompt = int(inputs[\"input_ids\"].shape[1])\n        n_vis = int((inputs[\"input_ids\"] == self.video_token_id).sum())\n        t0 = time.perf_counter()\n        with torch.inference_mode():\n            out = self.model.generate(\n                **inputs, max_new_tokens=1 if s.is_mc else max_new_tokens, do_sample=False,\n                output_logits=True, return_dict_in_generate=True,\n                pad_token_id=self.tok.pad_token_id or self.tok.eos_token_id,\n            )\n        if self._cuda:\n            torch.cuda.synchronize()\n        latency = (time.perf_counter() - t0) * 1e3\n        gen_ids = out.sequences[0, n_prompt:]\n        text = self.tok.decode(gen_ids, skip_special_tokens=True)\n        row = {\"text\": text, \"prompt_tokens\": n_prompt, \"visual_tokens\": n_vis, \"latency_ms\": latency}\n        if s.is_mc:\n            lp = torch.log_softmax(out.logits[0][0].float(), dim=-1)\n            letters = option_letters(s)\n            letter_lp = {L: float(lp[self.letter_ids[L]]) for L in letters if L in self.letter_ids}\n            row.update(mc_summary(softmax_over_options(letter_lp, letters)))\n        else:\n            lps = []\n            for t, logits in enumerate(out.logits):\n                if t >= len(gen_ids):\n                    break\n                lps.append(float(torch.log_softmax(logits[0].float(), -1)[gen_ids[t]]))\n            row[\"pred\"] = text.strip()\n            row[\"conf\"] = math.exp(sum(lps) / max(1, len(lps)))  # geometric-mean token prob\n        return row\n",
"src/foveate/backends/vllm_backend.py": "\"\"\"vLLM offline engine: the fast path for running every question through a config.\n\nFrames are sampled and resized by Foveate (see data/frames.py) and handed to\nvLLM as a pre-sampled video plus its metadata, so vLLM neither re-samples nor\nre-sizes them. Multiple-choice answers are scored from next-token log-probs\nover the option letters (one decode step per question).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport math\nimport time\n\nfrom ..data import Sample\nfrom ..data.frames import FrameBatch\nfrom ..prompts import (\n    chat_messages,\n    clean_token,\n    mc_summary,\n    option_letters,\n    parse_letter,\n    softmax_over_options,\n)\n\nlog = logging.getLogger(__name__)\n\n\nclass VLLMBackend:\n    def __init__(\n        self,\n        model_id: str,\n        *,\n        quantization: str | None = None,\n        dtype: str = \"auto\",\n        max_res: int = 672,\n        max_model_len: int = 8192,\n        gpu_memory_utilization: float = 0.90,\n        enforce_eager: bool = False,\n        tensor_parallel_size: int = 1,\n        max_num_seqs: int | None = None,\n        seed: int = 0,\n    ):\n        from transformers import AutoProcessor\n        from vllm import LLM\n\n        from ..utils import default_dtype\n\n        self.model_id = model_id\n        dtype = default_dtype() if dtype == \"auto\" else dtype\n        if quantization == \"awq\" and dtype == \"bfloat16\":\n            dtype = \"float16\"  # AWQ kernels run in fp16\n        self.processor = AutoProcessor.from_pretrained(model_id)\n        self.tok = self.processor.tokenizer\n        kwargs = dict(\n            model=model_id,\n            dtype=dtype,\n            quantization=quantization,\n            max_model_len=max_model_len,\n            gpu_memory_utilization=gpu_memory_utilization,\n            enforce_eager=enforce_eager,\n            tensor_parallel_size=tensor_parallel_size,\n            limit_mm_per_prompt={\"video\": 1, \"image\": 0},\n            # Frames arrive already resized; this range only has to contain them.\n            # Keeping max_pixels near the real maximum also keeps vLLM's start-up\n            # memory profiling (which uses a worst-case dummy video) realistic.\n            mm_processor_kwargs={\"min_pixels\": 28 * 28, \"max_pixels\": int(max_res * max_res)},\n            seed=seed,\n        )\n        if max_num_seqs:\n            kwargs[\"max_num_seqs\"] = max_num_seqs\n        log.info(\"starting vLLM: %s\", {k: v for k, v in kwargs.items() if k != \"model\"})\n        self.llm = LLM(**kwargs)\n        self.dtype = dtype\n        self.video_metadata = True  # flips off automatically on older vLLM\n        self.letter_ids = {}\n        for L in \"ABCDEFGH\":\n            ids = self.tok.encode(L, add_special_tokens=False)\n            if len(ids) == 1:\n                self.letter_ids[L] = ids[0]\n        self.video_token_id = self.tok.convert_tokens_to_ids(\"<|video_pad|>\")\n        import inspect\n\n        from vllm import SamplingParams\n\n        self._has_token_id_logprobs = (\n            \"logprob_token_ids\" in inspect.signature(SamplingParams.__init__).parameters\n            or \"logprob_token_ids\" in getattr(SamplingParams, \"__struct_fields__\", ())\n        )\n\n    def _prompt(self, s: Sample, fb: FrameBatch) -> dict:\n        text = self.processor.apply_chat_template(\n            chat_messages(s), tokenize=False, add_generation_prompt=True\n        )\n        if not self.video_metadata:\n            # Older vLLM: plain frame array (temporal positions assume the default fps).\n            return {\"prompt\": text, \"multi_modal_data\": {\"video\": fb.as_array()}}\n        meta = dict(fb.meta)\n        meta[\"do_sample_frames\"] = False\n        meta[\"height\"], meta[\"width\"] = fb.height, fb.width\n        return {\"prompt\": text, \"multi_modal_data\": {\"video\": (fb.as_array(), meta)}}\n\n    def _params(self, s: Sample, max_new_tokens: int):\n        from vllm import SamplingParams\n\n        if s.is_mc:\n            ids = [self.letter_ids[L] for L in option_letters(s) if L in self.letter_ids]\n            if self._has_token_id_logprobs and len(ids) == len(option_letters(s)):\n                # Exact log-probs for exactly the option letters (newer vLLM).\n                return SamplingParams(temperature=0.0, max_tokens=1, logprob_token_ids=ids)\n            return SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)\n        return SamplingParams(temperature=0.0, max_tokens=max_new_tokens, logprobs=1)\n\n    def _parse(self, s: Sample, out) -> dict:\n        co = out.outputs[0]\n        row = {\n            \"text\": co.text,\n            \"prompt_tokens\": len(out.prompt_token_ids or []),\n            \"visual_tokens\": sum(1 for t in (out.prompt_token_ids or []) if t == self.video_token_id),\n        }\n        if s.is_mc:\n            letters = option_letters(s)\n            first = co.logprobs[0] if co.logprobs else {}\n            id2letter = {v: k for k, v in self.letter_ids.items()}\n            letter_lp: dict[str, float] = {}\n            for tid, lp in first.items():\n                L = id2letter.get(tid) or clean_token(lp.decoded_token or \"\")\n                if L in letters:\n                    letter_lp[L] = max(letter_lp.get(L, -math.inf), lp.logprob)\n            if letter_lp:\n                row.update(mc_summary(softmax_over_options(letter_lp, letters)))\n            else:  # nothing usable in top-k: fall back to the generated text\n                row.update({\"pred\": parse_letter(co.text, letters), \"conf\": 0.0,\n                            \"margin\": 0.0, \"entropy\": math.log(len(letters))})\n        else:\n            n = max(1, len(co.token_ids))\n            row[\"pred\"] = co.text.strip()\n            cum = co.cumulative_logprob\n            row[\"conf\"] = math.exp(cum / n) if cum is not None else float(\"nan\")\n        return row\n\n    def predict_batch(\n        self, samples: list[Sample], batches: list[FrameBatch], max_new_tokens: int = 48\n    ) -> tuple[list[dict], float]:\n        params = [self._params(s, max_new_tokens) for s in samples]\n        prompts = [self._prompt(s, fb) for s, fb in zip(samples, batches)]\n        t0 = time.perf_counter()\n        try:\n            outs = self.llm.generate(prompts, params, use_tqdm=False)\n        except (TypeError, ValueError, AssertionError) as e:\n            if not self.video_metadata:\n                raise\n            log.warning(\"vLLM rejected (frames, metadata) video input (%s); \"\n                        \"retrying with plain frame arrays\", str(e).splitlines()[0][:120])\n            self.video_metadata = False\n            prompts = [self._prompt(s, fb) for s, fb in zip(samples, batches)]\n            t0 = time.perf_counter()\n            outs = self.llm.generate(prompts, params, use_tqdm=False)\n        wall = time.perf_counter() - t0\n        return [self._parse(s, o) for s, o in zip(samples, outs)], wall\n\n    def latency_probe(\n        self, samples: list[Sample], batches: list[FrameBatch], max_new_tokens: int = 48\n    ) -> list[float]:\n        \"\"\"Batch-1 end-to-end latency (ms) inside vLLM: processor + encoder + prefill + decode.\"\"\"\n        lat = []\n        for s, fb in zip(samples, batches):\n            p, sp = self._prompt(s, fb), self._params(s, max_new_tokens)\n            t0 = time.perf_counter()\n            self.llm.generate([p], sp, use_tqdm=False)\n            lat.append((time.perf_counter() - t0) * 1e3)\n        return lat\n",
"src/foveate/cli.py": "\"\"\"Command-line entry point: `foveate <command> ...`.\n\nEvery command that touches data takes --subset pilot (fixed-seed 30%, the\ndefault) or --subset full (also spelled --pilot / --full).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport logging\nimport os\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nfrom . import config as C\nfrom .utils import capture_env, setup_logging\n\nlog = logging.getLogger(\"foveate\")\n\n\ndef _subset_args(p: argparse.ArgumentParser) -> None:\n    g = p.add_mutually_exclusive_group()\n    g.add_argument(\"--subset\", choices=[\"pilot\", \"full\"], default=\"pilot\",\n                   help=\"pilot = fixed-seed 30%% stratified subset (default); full = everything\")\n    g.add_argument(\"--pilot\", dest=\"subset\", action=\"store_const\", const=\"pilot\")\n    g.add_argument(\"--full\", dest=\"subset\", action=\"store_const\", const=\"full\")\n    p.add_argument(\"--tasks\", default=None,\n                   help=\"MVBench tasks: all (default), lite (skip the 11 GB perception zip), \"\n                        \"or a comma list\")\n    p.add_argument(\"--limit\", type=int, default=None, help=\"cap questions (smoke tests)\")\n\n\ndef _samples(args, cfg):\n    from .data import load_samples\n\n    tasks = args.tasks.split(\",\") if args.tasks and \",\" in args.tasks else args.tasks\n    return load_samples(args.dataset, args.subset, fraction=cfg[\"pilot_fraction\"],\n                        seed=cfg[\"seed\"], tasks=tasks, limit=args.limit)\n\n\ndef _store_res(cfg, dataset: str) -> int:\n    return max(C.VisualConfig.parse(s, m).res for m, specs in cfg[\"grid\"][dataset].items() for s in specs)\n\n\n# -- commands ----------------------------------------------------------------\n\n\ndef cmd_env(args, cfg):\n    print(json.dumps(capture_env(), indent=1))\n\n\ndef cmd_download(args, cfg):\n    root = C.data_root()\n    if args.dataset == \"mvbench\":\n        from .data import mvbench\n\n        mvbench.download(root, args.tasks, keep_zips=args.keep_zips)\n    elif args.dataset == \"lingoqa\":\n        from .data import lingoqa\n\n        lingoqa.download(root)\n    elif args.dataset == \"synthetic\":\n        from .data import synthetic\n\n        synthetic.generate(root)\n    log.info(\"data is under %s\", root)\n\n\ndef cmd_profile(args, cfg):\n    from .backends.hf_backend import HFBackend\n    from .data import load_samples\n    from .profile import run_profile, summarize\n\n    pcfg = cfg[\"profile\"]\n    mid = C.model_id(cfg, args.model)\n    samples = load_samples(args.source, \"full\", tasks=args.tasks, limit=1)\n    backend = HFBackend(mid, dtype=args.dtype, attn_implementation=args.attn)\n    out = C.results_root() / \"profile\" / f\"{args.model}_{args.source}.jsonl\"\n    rows = run_profile(\n        backend, mid, samples,\n        frames=args.frames or pcfg[\"frames\"], resolutions=args.res or pcfg[\"res\"],\n        out_tokens=args.out_tokens or pcfg[\"out_tokens\"],\n        decode_sweep=pcfg[\"decode_sweep_out_tokens\"],\n        warmup=args.warmup if args.warmup is not None else pcfg[\"warmup\"],\n        repeats=args.repeats or pcfg[\"repeats\"], out_path=out,\n        cache_dir=C.cache_root() / \"profile\",\n    )\n    summ = summarize(rows, mid, args.peak_tflops, args.mem_bw)\n    out.with_suffix(\".summary.json\").write_text(json.dumps(summ, indent=1))\n    log.info(\"wrote %s and %s\", out, out.with_suffix(\".summary.json\"))\n\n\ndef _spawn_shards(args, n_gpus: int) -> None:\n    \"\"\"One child process per GPU, each running a deterministic shard of the questions.\"\"\"\n    argv = [a for a in sys.argv[1:]]\n    if \"--gpus\" in argv:\n        i = argv.index(\"--gpus\")\n        del argv[i:i + 2]\n    procs = []\n    for g in range(n_gpus):\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))\n        cmd = [sys.executable, \"-m\", \"foveate.cli\", *argv, \"--shard\", str(g), \"--num-shards\", str(n_gpus)]\n        log.info(\"GPU %d: %s\", g, \" \".join(cmd))\n        procs.append(subprocess.Popen(cmd, env=env))\n    codes = [p.wait() for p in procs]\n    if any(codes):\n        raise SystemExit(f\"shard processes exited with {codes}\")\n\n\ndef cmd_grid(args, cfg):\n    if args.gpus > 1 and args.shard is None:\n        return _spawn_shards(args, args.gpus)\n    from .grid import HFGridAdapter, run_grid\n\n    samples = _samples(args, cfg)\n    configs = ([C.VisualConfig.parse(s, args.model) for s in args.configs]\n               if args.configs else C.grid_configs(cfg, args.dataset, args.model))\n    configs = [c for c in configs if c.model == args.model]\n    mid = C.model_id(cfg, args.model)\n    backend = None\n    if args.backend in (\"vllm\", \"auto\"):\n        try:\n            from .backends.vllm_backend import VLLMBackend\n\n            backend = VLLMBackend(\n                mid, quantization=C.model_quantization(cfg, args.model), dtype=args.dtype,\n                max_res=max(c.res for c in configs), max_model_len=args.max_model_len,\n                gpu_memory_utilization=args.gpu_mem, enforce_eager=args.enforce_eager,\n                tensor_parallel_size=args.tp, max_num_seqs=args.max_num_seqs, seed=cfg[\"seed\"],\n            )\n        except Exception as e:  # noqa: BLE001 - any vLLM start-up failure\n            if args.backend == \"vllm\":\n                raise\n            log.warning(\"vLLM unavailable (%s: %s); falling back to the Transformers backend \"\n                        \"(batch 1, slower)\", type(e).__name__, str(e).splitlines()[0][:200])\n            _free_gpu()\n    if backend is None:\n        from .backends.hf_backend import HFBackend\n\n        backend = HFGridAdapter(HFBackend(mid, dtype=args.dtype, attn_implementation=args.attn))\n    corruptions = args.corruption or [None]\n    for corr in corruptions:\n        run_grid(backend, samples, configs, dataset=args.dataset, subset=args.subset,\n                 corruption=corr, shard_id=args.shard or 0, num_shards=args.num_shards,\n                 batch_size=args.batch_size, probe=0 if corr else args.probe,\n                 workers=args.workers, store_res=_store_res(cfg, args.dataset))\n\n\ndef _free_gpu():\n    import gc\n\n    gc.collect()\n    try:\n        import torch\n\n        torch.cuda.empty_cache()\n    except ImportError:\n        pass\n\n\ndef cmd_shift(args, cfg):\n    \"\"\"Re-run this model's shift configs on fog / night / blur copies.\"\"\"\n    sh = cfg[\"shift\"]\n    args.configs = [s for s in sh[\"configs\"][args.dataset] if s.startswith(args.model + \":\")]\n    if not args.configs:\n        log.info(\"no shift configs for model %s\", args.model)\n        return\n    args.corruption = args.corruption or [f\"{c}:{sh['severity']}\" for c in sh[\"corruptions\"]]\n    return cmd_grid(args, cfg)\n\n\ndef cmd_judge(args, cfg):\n    from .grid import run_dir\n    from .judge import judge_dir\n\n    samples = {s.qid: s for s in _samples(args, cfg)}\n    judge_dir(run_dir(args.dataset, args.subset), samples, device=args.device)\n\n\ndef cmd_replay(args, cfg):\n    from .drift import shift_report\n    from .grid import run_dir\n    from .policy import build_table, load_costs, load_rows, replay\n\n    rd = run_dir(args.dataset, args.subset)\n    df = load_rows(rd)\n    costs = load_costs(rd)\n    model_ids = {k: v[\"id\"] for k, v in cfg[\"models\"].items()}\n    t = build_table(df, costs, cost_metric=args.cost, model_ids=model_ids)\n    ladder = args.ladder or cfg[\"ladder\"][args.dataset]\n    rep = replay(t, ladder, ref=args.ref, tol=args.tol, seed=cfg[\"seed\"])\n    out = rd / \"replay\"\n    out.mkdir(parents=True, exist_ok=True)\n    (out / \"summary.json\").write_text(json.dumps(rep, indent=1))\n    policy = {\"dataset\": args.dataset, \"ladder\": rep[\"ladder\"], \"tau\": rep[\"tau\"],\n              \"rules\": rep[\"rules\"], \"costs\": rep[\"costs\"], \"cost_metric\": rep[\"cost_metric\"]}\n    (out / \"policy.json\").write_text(json.dumps(policy, indent=1))\n    for p in rep[\"policies\"]:\n        log.info(\"%-34s acc %.3f [%.3f, %.3f]  cost %.1f\", p[\"policy\"], p[\"accuracy\"],\n                 p[\"acc_lo\"], p[\"acc_hi\"], p[\"mean_cost\"])\n    if (df[\"corruption\"] != \"clean\").any():\n        srep = shift_report(df, costs, rep[\"ladder\"], rep[\"tau\"], model_ids)\n        (out / \"shift.json\").write_text(json.dumps(srep, indent=1))\n        for r in srep[\"per_corruption\"]:\n            log.info(\"%s: escalation %.2f -> %.2f, cascade acc %.3f -> %.3f\", r[\"corruption\"],\n                     r[\"clean_escalation\"], r[\"shifted_escalation\"], r[\"clean_cascade_acc\"],\n                     r[\"shifted_cascade_acc\"])\n    log.info(\"wrote %s\", out)\n\n\ndef cmd_serve(args, cfg):\n    import uvicorn\n\n    from .serve.gateway import Gateway, build_app\n\n    samples = {s.qid: s for s in _samples(args, cfg)}\n    pol = json.loads(Path(args.policy_file).read_text()) if args.policy_file else {}\n    ladder = [C.VisualConfig.parse(s) for s in (pol.get(\"ladder\") or cfg[\"ladder\"][args.dataset])]\n    backends = dict(b.split(\"=\", 1) for b in args.backends)\n    gw = Gateway(samples, backends, {k: v[\"id\"] for k, v in cfg[\"models\"].items()}, ladder,\n                 tau=args.tau if args.tau is not None else pol.get(\"tau\", 0.6),\n                 telemetry=C.results_root() / \"serving\" / \"telemetry.jsonl\",\n                 store_res=_store_res(cfg, args.dataset))\n    uvicorn.run(build_app(gw), host=args.host, port=args.port, log_level=\"warning\")\n\n\ndef cmd_loadtest(args, cfg):\n    import asyncio\n\n    from .serve.loadgen import make_trace, run_trace\n\n    samples = _samples(args, cfg)\n    trace = make_trace(samples, args.n, args.rate, args.repeat, seed=cfg[\"seed\"])\n    tag = args.tag or f\"{args.policy.replace(':', '-')}_r{args.rate}\"\n    out = C.results_root() / \"serving\" / f\"{tag}.jsonl\"\n    summ = asyncio.run(run_trace(args.url, trace, args.policy, out))\n    summ.update({\"rate\": args.rate, \"repeat\": args.repeat})\n    out.with_suffix(\".summary.json\").write_text(json.dumps(summ, indent=1))\n    print(json.dumps(summ, indent=1))\n\n\ndef cmd_report(args, cfg):\n    from .report import build_report\n\n    print(build_report(C.results_root()))\n\n\n# -- parser ------------------------------------------------------------------\n\n\ndef build_parser() -> argparse.ArgumentParser:\n    ap = argparse.ArgumentParser(prog=\"foveate\", description=__doc__)\n    ap.add_argument(\"--config\", default=None, help=\"YAML config (default: packaged default.yaml)\")\n    ap.add_argument(\"-v\", \"--verbose\", action=\"store_true\")\n    sub = ap.add_subparsers(dest=\"cmd\", required=True)\n\n    sub.add_parser(\"env\", help=\"print GPU / library versions\")\n\n    p = sub.add_parser(\"download\", help=\"fetch a dataset into $FOVEATE_DATA (default ./data)\")\n    p.add_argument(\"dataset\", choices=[\"mvbench\", \"lingoqa\", \"synthetic\"])\n    p.add_argument(\"--tasks\", default=\"all\")\n    p.add_argument(\"--keep-zips\", action=\"store_true\")\n\n    p = sub.add_parser(\"profile\", help=\"stage 1: batch-1 stage timing sweep (Transformers)\")\n    p.add_argument(\"--model\", default=\"3b\")\n    p.add_argument(\"--source\", default=\"synthetic\", choices=[\"synthetic\", \"mvbench\", \"lingoqa\"])\n    p.add_argument(\"--tasks\", default=None)\n    p.add_argument(\"--frames\", type=int, nargs=\"*\")\n    p.add_argument(\"--res\", type=int, nargs=\"*\")\n    p.add_argument(\"--out-tokens\", type=int)\n    p.add_argument(\"--repeats\", type=int)\n    p.add_argument(\"--warmup\", type=int)\n    p.add_argument(\"--dtype\", default=\"auto\")\n    p.add_argument(\"--attn\", default=\"sdpa\")\n    p.add_argument(\"--peak-tflops\", type=float)\n    p.add_argument(\"--mem-bw\", type=float)\n\n    def grid_args(p):\n        p.add_argument(\"--dataset\", default=\"mvbench\", choices=[\"mvbench\", \"lingoqa\", \"synthetic\"])\n        _subset_args(p)\n        p.add_argument(\"--model\", default=\"3b\")\n        p.add_argument(\"--configs\", nargs=\"*\", help=\"e.g. 4x224 8x448 (default: from config)\")\n        p.add_argument(\"--corruption\", nargs=\"*\", help=\"e.g. fog:3 night:3 blur:3\")\n        p.add_argument(\"--backend\", choices=[\"auto\", \"vllm\", \"hf\"], default=\"auto\",\n                       help=\"auto = vLLM, falling back to Transformers if vLLM fails to start\")\n        p.add_argument(\"--gpus\", type=int, default=1, help=\"data-parallel: one process per GPU\")\n        p.add_argument(\"--shard\", type=int, default=None)\n        p.add_argument(\"--num-shards\", type=int, default=1)\n        p.add_argument(\"--batch-size\", type=int, default=32)\n        p.add_argument(\"--probe\", type=int, default=16, help=\"batch-1 latency probe size\")\n        p.add_argument(\"--workers\", type=int, default=4, help=\"frame-loading threads\")\n        p.add_argument(\"--dtype\", default=\"auto\")\n        p.add_argument(\"--attn\", default=\"sdpa\")\n        p.add_argument(\"--gpu-mem\", type=float, default=0.90)\n        p.add_argument(\"--max-model-len\", type=int, default=8192)\n        p.add_argument(\"--max-num-seqs\", type=int, default=None)\n        p.add_argument(\"--tp\", type=int, default=1, help=\"tensor parallel size (e.g. fp32 on 2xT4)\")\n        p.add_argument(\"--enforce-eager\", action=\"store_true\",\n                       help=\"skip CUDA graphs (faster start-up, less memory)\")\n\n    grid_args(sub.add_parser(\"grid\", help=\"stage 2: run configs over every question (vLLM)\"))\n    grid_args(sub.add_parser(\"shift\", help=\"stage 4: rerun shift configs on corrupted frames\"))\n\n    p = sub.add_parser(\"judge\", help=\"score LingoQA answers with Lingo-Judge\")\n    p.add_argument(\"--dataset\", default=\"lingoqa\")\n    _subset_args(p)\n    p.add_argument(\"--device\", default=0)\n\n    p = sub.add_parser(\"replay\", help=\"stages 3-4: evaluate policies + shift from the grid cache\")\n    p.add_argument(\"--dataset\", default=\"mvbench\")\n    _subset_args(p)\n    p.add_argument(\"--ladder\", nargs=\"*\")\n    p.add_argument(\"--ref\", default=None, help=\"reference config (default: most accurate)\")\n    p.add_argument(\"--cost\", default=\"latency_b1_ms\", choices=[\"latency_b1_ms\", \"amortized_ms\"])\n    p.add_argument(\"--tol\", type=float, default=0.0, help=\"accuracy the cascade may give up\")\n\n    p = sub.add_parser(\"serve\", help=\"stage 5: policy gateway in front of vllm serve\")\n    p.add_argument(\"--dataset\", default=\"mvbench\")\n    _subset_args(p)\n    p.add_argument(\"--backends\", nargs=\"+\", default=[\"3b=http://localhost:8001\", \"7b=http://localhost:8002\"])\n    p.add_argument(\"--policy-file\", default=None)\n    p.add_argument(\"--tau\", type=float, default=None)\n    p.add_argument(\"--host\", default=\"0.0.0.0\")\n    p.add_argument(\"--port\", type=int, default=8000)\n\n    p = sub.add_parser(\"loadtest\", help=\"stage 5: Poisson load against the gateway\")\n    p.add_argument(\"--dataset\", default=\"mvbench\")\n    _subset_args(p)\n    p.add_argument(\"--url\", default=\"http://localhost:8000\")\n    p.add_argument(\"--policy\", default=\"cascade\", help=\"cascade or static:3b:16x448\")\n    p.add_argument(\"--rate\", type=float, default=1.0, help=\"requests per second\")\n    p.add_argument(\"--n\", type=int, default=200)\n    p.add_argument(\"--repeat\", type=float, default=0.3, help=\"share of repeated-clip requests\")\n    p.add_argument(\"--tag\", default=None)\n\n    sub.add_parser(\"report\", help=\"figures + results/REPORT.md\")\n    return ap\n\n\nCOMMANDS = {\n    \"env\": cmd_env, \"download\": cmd_download, \"profile\": cmd_profile, \"grid\": cmd_grid,\n    \"shift\": cmd_shift, \"judge\": cmd_judge, \"replay\": cmd_replay, \"serve\": cmd_serve,\n    \"loadtest\": cmd_loadtest, \"report\": cmd_report,\n}\n\n\ndef main(argv: list[str] | None = None) -> None:\n    args = build_parser().parse_args(argv)\n    setup_logging(args.verbose)\n    cfg = C.load_config(args.config)\n    COMMANDS[args.cmd](args, cfg)\n\n\nif __name__ == \"__main__\":\n    main()\n",
"src/foveate/config.py": "\"\"\"Experiment configuration: model aliases, visual configs and paths.\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nimport re\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\n\nDEFAULT_CONFIG = Path(__file__).resolve().parent / \"configs\" / \"default.yaml\"\n\n_SPEC = re.compile(r\"^(?:(?P<model>[\\w.\\-]+):)?(?P<frames>\\d+)x(?P<res>\\d+)$\")\n\n\n@dataclass(frozen=True, order=True)\nclass VisualConfig:\n    \"\"\"One point in the compute space: which model, how many frames, what resolution.\"\"\"\n\n    model: str\n    frames: int\n    res: int\n\n    @property\n    def name(self) -> str:\n        return f\"{self.model}:{self.frames}x{self.res}\"\n\n    @classmethod\n    def parse(cls, spec: str, default_model: str | None = None) -> \"VisualConfig\":\n        m = _SPEC.match(spec.strip())\n        if not m:\n            raise ValueError(f\"bad config spec {spec!r}; expected MODEL:FRAMESxRES like 3b:8x448\")\n        model = m.group(\"model\") or default_model\n        if model is None:\n            raise ValueError(f\"config {spec!r} needs a model prefix\")\n        return cls(model, int(m.group(\"frames\")), int(m.group(\"res\")))\n\n    def __str__(self) -> str:  # pragma: no cover\n        return self.name\n\n\ndef load_config(path: str | Path | None = None) -> dict[str, Any]:\n    path = Path(path) if path else DEFAULT_CONFIG\n    with open(path) as f:\n        return yaml.safe_load(f)\n\n\ndef model_id(cfg: dict, alias: str) -> str:\n    if alias in cfg[\"models\"]:\n        return cfg[\"models\"][alias][\"id\"]\n    return alias  # allow raw HF ids\n\n\ndef model_quantization(cfg: dict, alias: str) -> str | None:\n    return cfg[\"models\"].get(alias, {}).get(\"quantization\")\n\n\ndef grid_configs(cfg: dict, dataset: str, model_alias: str) -> list[VisualConfig]:\n    specs = cfg[\"grid\"][dataset][model_alias]\n    return [VisualConfig.parse(s, default_model=model_alias) for s in specs]\n\n\ndef ladder(cfg: dict, dataset: str) -> list[VisualConfig]:\n    return [VisualConfig.parse(s) for s in cfg[\"ladder\"][dataset]]\n\n\ndef data_root() -> Path:\n    return Path(os.environ.get(\"FOVEATE_DATA\", \"data\")).expanduser()\n\n\ndef results_root() -> Path:\n    return Path(os.environ.get(\"FOVEATE_RESULTS\", \"results\")).expanduser()\n\n\ndef cache_root() -> Path:\n    return Path(os.environ.get(\"FOVEATE_CACHE\", str(data_root() / \"frame_cache\"))).expanduser()\n",
"src/foveate/data/__init__.py": "\"\"\"Datasets: a common Sample record, loaders, and the pilot/full subset logic.\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport random\nfrom collections import defaultdict\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Callable\n\nLETTERS = \"ABCDEFGH\"\n\n\n@dataclass\nclass Sample:\n    qid: str\n    dataset: str\n    task: str\n    question: str\n    answer: str  # gold option letter for multiple choice, reference text otherwise\n    options: list[str] | None = None  # multiple-choice candidates (None = free-form)\n    references: list[str] = field(default_factory=list)\n    video: str | None = None  # path to a video file\n    frame_paths: list[str] | None = None  # or a list of pre-extracted frame images\n    start: float | None = None  # optional clip bounds in seconds\n    end: float | None = None\n    clip_id: str | None = None  # questions sharing a clip share this id\n\n    @property\n    def is_mc(self) -> bool:\n        return self.options is not None\n\n    @property\n    def source_key(self) -> str:\n        \"\"\"Identifies the visual input (for caching and repeat-clip workloads).\"\"\"\n        base = self.video or \"|\".join(self.frame_paths or [])\n        return f\"{base}#{self.start}-{self.end}\"\n\n    def exists(self) -> bool:\n        if self.video:\n            return Path(self.video).exists()\n        if self.frame_paths:\n            return all(Path(p).exists() for p in self.frame_paths)\n        return False\n\n\ndef stable_hash(s: str) -> int:\n    return int(hashlib.sha1(s.encode()).hexdigest()[:12], 16)\n\n\ndef pilot_subset(samples: list[Sample], fraction: float = 0.30, seed: int = 0) -> list[Sample]:\n    \"\"\"Fixed-seed stratified subset: `fraction` of every task, at least one item each.\n\n    Stratifying by task keeps per-task accuracy estimable on the pilot, and the\n    fixed seed means pilot runs on different days hit exactly the same questions,\n    so pilot results can be compared with each other (and later with full runs).\n    \"\"\"\n    by_task: dict[str, list[Sample]] = defaultdict(list)\n    for s in samples:\n        by_task[s.task].append(s)\n    out: list[Sample] = []\n    for task in sorted(by_task):\n        items = sorted(by_task[task], key=lambda s: s.qid)\n        rng = random.Random(stable_hash(f\"{seed}:{task}\"))\n        k = max(1, round(len(items) * fraction))\n        out.extend(rng.sample(items, k))\n    return sorted(out, key=lambda s: (s.task, s.qid))\n\n\ndef shard(samples: list[Sample], shard_id: int, num_shards: int) -> list[Sample]:\n    \"\"\"Deterministic split for data-parallel runs (one process per GPU).\"\"\"\n    if num_shards <= 1:\n        return samples\n    return [s for i, s in enumerate(samples) if i % num_shards == shard_id]\n\n\n_LOADERS: dict[str, Callable[..., list[Sample]]] = {}\n\n\ndef register(name: str):\n    def deco(fn):\n        _LOADERS[name] = fn\n        return fn\n\n    return deco\n\n\ndef load_samples(\n    dataset: str,\n    subset: str = \"pilot\",\n    *,\n    fraction: float = 0.30,\n    seed: int = 0,\n    tasks: list[str] | None = None,\n    limit: int | None = None,\n    root: str | Path | None = None,\n) -> list[Sample]:\n    \"\"\"Load a dataset, then select `subset` (\"pilot\" = fixed 30%, \"full\" = everything).\"\"\"\n    # Import loaders lazily so `import foveate.data` stays light.\n    from . import lingoqa, mvbench, synthetic  # noqa: F401\n\n    if dataset not in _LOADERS:\n        raise KeyError(f\"unknown dataset {dataset!r}; known: {sorted(_LOADERS)}\")\n    samples = _LOADERS[dataset](root=root, tasks=tasks)\n    if subset == \"pilot\":\n        samples = pilot_subset(samples, fraction=fraction, seed=seed)\n    elif subset != \"full\":\n        raise ValueError(f\"subset must be 'pilot' or 'full', got {subset!r}\")\n    if limit:\n        # Keep the task mix when truncating for smoke tests: round-robin over tasks.\n        by_task: dict[str, list[Sample]] = defaultdict(list)\n        for s in samples:\n            by_task[s.task].append(s)\n        rr: list[Sample] = []\n        queues = [by_task[t] for t in sorted(by_task)]\n        while len(rr) < limit and any(queues):\n            for q in queues:\n                if q and len(rr) < limit:\n                    rr.append(q.pop(0))\n        samples = rr\n    return samples\n",
"src/foveate/data/corrupt.py": "\"\"\"Synthetic distribution shift for driving-style footage: fog, night, motion blur.\n\nModelled loosely on ImageNet-C (Hendrycks & Dietterich, 2019): each corruption\nhas severities 1..5. They are deliberately simple and deterministic (seeded per\nframe) so a shifted run can be reproduced exactly.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom PIL import Image, ImageFilter\n\nCORRUPTIONS = (\"fog\", \"night\", \"blur\")\n\n\ndef _fog(a: np.ndarray, sev: int, rng: np.random.Generator) -> np.ndarray:\n    # Atmospheric scattering model: I = J*t + A*(1-t), with transmission t\n    # falling towards the top of the frame (farther scene = more haze).\n    h, w, _ = a.shape\n    t_min = [0.75, 0.6, 0.45, 0.32, 0.2][sev - 1]\n    depth = np.linspace(1.0, 0.25, h)[:, None]  # top rows ~far\n    t = t_min + (1 - t_min) * (1 - depth)\n    # Low-frequency density noise so the haze is not a perfect gradient.\n    noise = rng.normal(0, 1, (max(2, h // 32), max(2, w // 32))).astype(np.float32)\n    noise = np.asarray(Image.fromarray(noise).resize((w, h), Image.BICUBIC))\n    t = np.clip(t + 0.05 * noise, 0.05, 1.0)[..., None]\n    airlight = 0.85\n    return a * t + airlight * (1 - t)\n\n\ndef _night(a: np.ndarray, sev: int, rng: np.random.Generator) -> np.ndarray:\n    # Gamma darkening, a blue cast, and sensor noise that grows as light drops.\n    gamma = [1.6, 2.0, 2.5, 3.0, 3.6][sev - 1]\n    gain = [0.75, 0.6, 0.5, 0.4, 0.3][sev - 1]\n    out = gain * np.power(a, gamma)\n    out = out * np.array([0.85, 0.9, 1.1], dtype=np.float32)\n    sigma = [0.01, 0.02, 0.03, 0.04, 0.05][sev - 1]\n    return out + rng.normal(0, sigma, a.shape).astype(np.float32)\n\n\ndef _blur(img: Image.Image, sev: int, rng: np.random.Generator) -> np.ndarray:\n    # Horizontal motion blur (camera pan / ego-motion) via a box filter along x,\n    # with length relative to frame width so severity is resolution-independent.\n    a = np.asarray(img, dtype=np.float32) / 255.0\n    frac = [0.01, 0.02, 0.035, 0.05, 0.07][sev - 1]\n    k = max(3, int(round(frac * a.shape[1])) | 1)\n    pad = k // 2\n    p = np.pad(a, ((0, 0), (pad, pad), (0, 0)), mode=\"edge\")\n    c = np.cumsum(p, axis=1, dtype=np.float64)\n    c = np.concatenate([np.zeros_like(c[:, :1]), c], axis=1)\n    out = (c[:, k:] - c[:, :-k]) / k\n    # A touch of defocus as well.\n    out_img = Image.fromarray((np.clip(out, 0, 1) * 255).astype(np.uint8))\n    out_img = out_img.filter(ImageFilter.GaussianBlur(radius=0.3 * sev))\n    return np.asarray(out_img, dtype=np.float32) / 255.0\n\n\ndef parse(spec: str) -> tuple[str, int]:\n    \"\"\"'fog:3' -> ('fog', 3); 'fog' -> ('fog', 3).\"\"\"\n    name, _, sev = spec.partition(\":\")\n    sev_i = int(sev) if sev else 3\n    if name not in CORRUPTIONS:\n        raise ValueError(f\"unknown corruption {name!r}; choose from {CORRUPTIONS}\")\n    if not 1 <= sev_i <= 5:\n        raise ValueError(\"severity must be 1..5\")\n    return name, sev_i\n\n\ndef apply_corruption(img: Image.Image, spec: str, seed: int = 0) -> Image.Image:\n    name, sev = parse(spec)\n    rng = np.random.default_rng(seed)\n    if name == \"blur\":\n        out = _blur(img.convert(\"RGB\"), sev, rng)\n    else:\n        a = np.asarray(img.convert(\"RGB\"), dtype=np.float32) / 255.0\n        out = _fog(a, sev, rng) if name == \"fog\" else _night(a, sev, rng)\n    return Image.fromarray((np.clip(out, 0, 1) * 255).astype(np.uint8))\n",
"src/foveate/data/frames.py": "\"\"\"Frame sampling, decoding, caching and resizing.\n\nAll frame handling lives on the client side (before the model), because frame\ncount and resolution are the two knobs the project controls. Doing it here\nrather than inside the model processor means the Transformers profiler, the\nvLLM grid runner and the serving gateway all see byte-identical inputs.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport logging\nimport threading\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\nfrom PIL import Image\n\nfrom ..tokens import target_size\nfrom . import Sample, stable_hash\n\nlog = logging.getLogger(__name__)\n\n\ndef sample_indices(lo: int, hi: int, n: int) -> list[int]:\n    \"\"\"n indices spread uniformly over [lo, hi], taking the middle of each segment.\n\n    Middle-of-segment sampling avoids always picking the very first frame (often\n    a fade-in) and matches what most video-QA evaluation code does. If n exceeds\n    the number of frames available, indices repeat.\n    \"\"\"\n    if n <= 0:\n        raise ValueError(\"n must be positive\")\n    if hi < lo:\n        hi = lo\n    span = hi - lo + 1\n    return [min(hi, lo + int((i + 0.5) * span / n)) for i in range(n)]\n\n\n@dataclass\nclass VideoInfo:\n    total_frames: int\n    fps: float\n    height: int\n    width: int\n\n    @property\n    def duration(self) -> float:\n        return self.total_frames / self.fps if self.fps else 0.0\n\n\ndef probe_video(path: str) -> VideoInfo:\n    import av\n\n    with av.open(path) as c:\n        st = c.streams.video[0]\n        fps = float(st.average_rate or st.guessed_rate or 25.0)\n        n = st.frames\n        if not n:\n            dur = float(st.duration * st.time_base) if st.duration else float(c.duration or 0) / 1e6\n            n = max(1, int(round(dur * fps)))\n        return VideoInfo(int(n), fps, st.codec_context.height, st.codec_context.width)\n\n\ndef decode_indices(path: str, wanted: Iterable[int]) -> dict[int, np.ndarray]:\n    \"\"\"Decode the wanted frame indices in one sequential pass (robust for any codec).\n\n    Sequential decoding avoids keyframe-seek inaccuracies across webm/mp4/gif.\n    Indices past the real end of the stream (frame counts in headers lie) are\n    filled with the last decoded frame.\n    \"\"\"\n    import av\n\n    wanted = sorted(set(wanted))\n    if not wanted:\n        return {}\n    want_set, last_needed = set(wanted), wanted[-1]\n    out: dict[int, np.ndarray] = {}\n    last = None\n    with av.open(path) as c:\n        st = c.streams.video[0]\n        st.thread_type = \"AUTO\"\n        for i, frame in enumerate(c.decode(st)):\n            if i in want_set:\n                out[i] = frame.to_ndarray(format=\"rgb24\")\n            last = (i, frame)\n            if i >= last_needed:\n                break\n    if last is None:\n        raise RuntimeError(f\"no frames decoded from {path}\")\n    tail = None\n    for idx in wanted:\n        if idx not in out:\n            if tail is None:\n                tail = last[1].to_ndarray(format=\"rgb24\") if last[0] not in out else out[last[0]]\n            out[idx] = tail\n    return out\n\n\n@dataclass\nclass FrameBatch:\n    frames: list[Image.Image]\n    height: int\n    width: int\n    meta: dict  # fps, total_num_frames, frames_indices, duration (for the processor)\n    load_s: float\n    stats: dict = field(default_factory=dict)\n\n    def as_array(self) -> np.ndarray:\n        return np.stack([np.asarray(f) for f in self.frames])\n\n\ndef image_stats(frames: list[Image.Image]) -> dict[str, float]:\n    \"\"\"Cheap, model-free statistics used by the drift monitor (stage 4).\"\"\"\n    arr = np.stack([np.asarray(f.convert(\"RGB\"), dtype=np.float32) for f in frames]) / 255.0\n    luma = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]\n    # Variance of a 4-neighbour Laplacian: low for blurry or foggy frames.\n    lap = (\n        -4 * luma[:, 1:-1, 1:-1]\n        + luma[:, :-2, 1:-1] + luma[:, 2:, 1:-1] + luma[:, 1:-1, :-2] + luma[:, 1:-1, 2:]\n    )\n    mx, mn = arr.max(-1), arr.min(-1)\n    sat = np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0)\n    return {\n        \"brightness\": float(luma.mean()),\n        \"contrast\": float(luma.std()),\n        \"sharpness\": float(lap.var() * 1e3),\n        \"saturation\": float(sat.mean()),\n    }\n\n\nclass FrameStore:\n    \"\"\"Loads n frames at a resolution for a sample, with an on-disk JPEG cache.\n\n    The first request for a video decodes the union of frame indices for every\n    frame count in `frame_counts` in a single pass and caches them at\n    `store_res`, so later configs (other n or lower res) never touch the video\n    decoder again. On Kaggle's 4 vCPUs this turns decoding from the bottleneck\n    into a one-time cost.\n    \"\"\"\n\n    def __init__(\n        self,\n        cache_dir: str | Path | None,\n        frame_counts: Iterable[int] = (2, 4, 8, 16),\n        store_res: int = 896,\n        jpeg_quality: int = 92,\n    ):\n        self.cache_dir = Path(cache_dir) if cache_dir else None\n        self.frame_counts = sorted(set(frame_counts))\n        self.store_res = store_res\n        self.q = jpeg_quality\n        self._locks: dict[str, threading.Lock] = {}\n        self._glock = threading.Lock()\n\n    def _lock(self, key: str) -> threading.Lock:\n        with self._glock:\n            return self._locks.setdefault(key, threading.Lock())\n\n    # -- raw frames at store resolution ------------------------------------\n    def _bounds(self, s: Sample, info: VideoInfo) -> tuple[int, int]:\n        lo, hi = 0, info.total_frames - 1\n        if s.start is not None:\n            lo = max(0, min(hi, int(s.start * info.fps)))\n        if s.end is not None:\n            hi = max(lo, min(hi, int(s.end * info.fps)))\n        return lo, hi\n\n    def _shrink(self, img: Image.Image) -> Image.Image:\n        h, w = img.height, img.width\n        if h * w <= self.store_res**2:\n            return img\n        scale = (self.store_res**2 / (h * w)) ** 0.5\n        return img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.BICUBIC)\n\n    def raw_frames(self, s: Sample, n: int) -> tuple[list[Image.Image], dict]:\n        if s.frame_paths:\n            paths = s.frame_paths\n            idx = sample_indices(0, len(paths) - 1, n)\n            fps = 3.0 if s.dataset == \"mvbench\" else 1.0\n            imgs = [self._shrink(Image.open(paths[i]).convert(\"RGB\")) for i in idx]\n            meta = {\n                \"fps\": fps, \"total_num_frames\": len(paths),\n                \"frames_indices\": idx, \"duration\": len(paths) / fps,\n            }\n            return imgs, meta\n        if not s.video:\n            raise ValueError(f\"sample {s.qid} has no visual input\")\n        key = hashlib.sha1(s.source_key.encode()).hexdigest()[:20]\n        with self._lock(key):\n            info = self._info(key, s.video)\n            lo, hi = self._bounds(s, info)\n            idx = sample_indices(lo, hi, n)\n            imgs = self._from_cache(key, idx)\n            if imgs is None:\n                union = set(idx)\n                for k in self.frame_counts:\n                    union.update(sample_indices(lo, hi, k))\n                decoded = decode_indices(s.video, union)\n                for i, a in decoded.items():\n                    img = self._shrink(Image.fromarray(a))\n                    if self.cache_dir:\n                        p = self.cache_dir / key / f\"{i:06d}.jpg\"\n                        p.parent.mkdir(parents=True, exist_ok=True)\n                        img.save(p, quality=self.q)\n                    decoded[i] = img\n                imgs = [decoded[i] for i in idx]\n        meta = {\n            \"fps\": info.fps, \"total_num_frames\": hi - lo + 1,\n            \"frames_indices\": [i - lo for i in idx], \"duration\": (hi - lo + 1) / info.fps,\n        }\n        return imgs, meta\n\n    def _info(self, key: str, path: str) -> VideoInfo:\n        if self.cache_dir:\n            p = self.cache_dir / key / \"info.txt\"\n            if p.exists():\n                n, fps, h, w = p.read_text().split()\n                return VideoInfo(int(n), float(fps), int(h), int(w))\n            info = probe_video(path)\n            p.parent.mkdir(parents=True, exist_ok=True)\n            p.write_text(f\"{info.total_frames} {info.fps} {info.height} {info.width}\")\n            return info\n        return probe_video(path)\n\n    def _from_cache(self, key: str, idx: list[int]) -> list[Image.Image] | None:\n        if not self.cache_dir:\n            return None\n        paths = [self.cache_dir / key / f\"{i:06d}.jpg\" for i in idx]\n        if not all(p.exists() for p in paths):\n            return None\n        return [Image.open(p).convert(\"RGB\") for p in paths]\n\n    # -- public -------------------------------------------------------------\n    def load(\n        self, s: Sample, n: int, res: int, corruption: str | None = None, with_stats: bool = True\n    ) -> FrameBatch:\n        t0 = time.perf_counter()\n        imgs, meta = self.raw_frames(s, n)\n        if corruption:\n            from .corrupt import apply_corruption\n\n            imgs = [\n                apply_corruption(im, corruption, seed=stable_hash(f\"{s.qid}:{j}:{corruption}\") & 0xFFFFFFFF)\n                for j, im in enumerate(imgs)\n            ]\n        h, w = target_size(imgs[0].height, imgs[0].width, res)\n        imgs = [im.resize((w, h), Image.BICUBIC) if im.size != (w, h) else im for im in imgs]\n        stats = image_stats(imgs) if with_stats else {}\n        return FrameBatch(imgs, h, w, meta, time.perf_counter() - t0, stats)\n",
"src/foveate/data/lingoqa.py": "\"\"\"LingoQA (Marcu et al., ECCV 2024): driving-scene video QA from Wayve.\n\nThe evaluation split has 500 questions over 100 driving clips (two reference\nanswers each, 1,000 rows), with a handful of frames per clip. Answers are free\ntext and are scored by Lingo-Judge, a small classifier released with the\ndataset (see foveate/judge.py). This is the driving demo workload; MVBench is\nthe headline benchmark.\n\nRead the dataset's LICENCE before redistributing anything derived from it.\n\nExpected layout under <data_root>/lingoqa (what `foveate download lingoqa` makes):\n    val.parquet (any *.parquet whose name contains \"val\" also works)\n    images/...  (the extracted images archive)\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\n\nfrom . import Sample, register\n\nlog = logging.getLogger(__name__)\n\nEVAL_FOLDER = \"https://drive.google.com/drive/folders/1oA7W8-Ej_uJEuUxZIjPP5K8hQGGzYsPq\"\n\n\ndef download(root: Path) -> None:\n    import gdown\n\n    out = root / \"lingoqa\"\n    out.mkdir(parents=True, exist_ok=True)\n    gdown.download_folder(EVAL_FOLDER, output=str(out), quiet=False, remaining_ok=True)\n    for z in out.rglob(\"*.zip\"):\n        marker = z.with_suffix(\".done\")\n        if not marker.exists():\n            log.info(\"extracting %s\", z)\n            with zipfile.ZipFile(z) as zf:\n                zf.extractall(z.parent)\n            marker.touch()\n\n\ndef _find_parquet(base: Path) -> Path:\n    cands = sorted(p for p in base.rglob(\"*.parquet\") if \"val\" in p.name.lower())\n    if not cands:\n        cands = sorted(base.rglob(\"*.parquet\"))\n    if not cands:\n        raise FileNotFoundError(f\"no parquet under {base}; run `foveate download lingoqa`\")\n    return cands[0]\n\n\ndef _resolve_images(base: Path, rels, segment_id: str, seg_index: dict[str, list[Path]]):\n    paths = []\n    for r in rels if rels is not None else []:\n        r = str(r)\n        for cand in (base / r, base / \"images\" / r, Path(r)):\n            if cand.exists():\n                paths.append(str(cand))\n                break\n    if paths:\n        return paths\n    return [str(p) for p in seg_index.get(segment_id, [])]\n\n\n@register(\"lingoqa\")\ndef load(root: str | Path | None = None, tasks=None) -> list[Sample]:\n    import pandas as pd\n\n    from ..config import data_root\n\n    base = Path(root or data_root()) / \"lingoqa\"\n    df = pd.read_parquet(_find_parquet(base))\n    # Fallback index: images grouped by their parent folder name (= segment id).\n    seg_index: dict[str, list[Path]] = defaultdict(list)\n    for p in base.rglob(\"*\"):\n        if p.suffix.lower() in {\".jpg\", \".jpeg\", \".png\"}:\n            seg_index[p.parent.name].append(p)\n    for k in seg_index:\n        seg_index[k].sort(key=lambda p: (len(p.stem), p.stem))\n\n    ans_col = \"answers\" if \"answers\" in df.columns else \"answer\"\n    groups: dict[str, dict] = {}\n    for row in df.itertuples(index=False):\n        r = row._asdict()\n        qid = str(r[\"question_id\"])\n        g = groups.setdefault(\n            qid,\n            {\"segment_id\": str(r[\"segment_id\"]), \"question\": r[\"question\"], \"refs\": [],\n             \"images\": r.get(\"images\")},\n        )\n        a = r[ans_col]\n        if isinstance(a, str):\n            g[\"refs\"].append(a)\n        else:\n            g[\"refs\"].extend(str(x) for x in a)\n    samples = []\n    for qid, g in groups.items():\n        frames = _resolve_images(base, g[\"images\"], g[\"segment_id\"], seg_index)\n        if not frames:\n            continue\n        samples.append(\n            Sample(\n                qid=qid,\n                dataset=\"lingoqa\",\n                task=\"driving\",\n                question=str(g[\"question\"]),\n                answer=g[\"refs\"][0],\n                references=list(dict.fromkeys(g[\"refs\"])),\n                frame_paths=frames,\n                clip_id=g[\"segment_id\"],\n            )\n        )\n    log.info(\"LingoQA: %d questions over %d clips\", len(samples), len({s.clip_id for s in samples}))\n    return samples\n",
"src/foveate/data/mvbench.py": "\"\"\"MVBench (Li et al., CVPR 2024): 20 temporal-reasoning tasks x 200 multiple-choice questions.\n\nWhy this dataset: it is in the video table of nearly every recent VLM paper\n(Qwen2.5-VL reports 67.0 for the 3B and 69.6 for the 7B model), so a harness\nbug shows up as a gap to a published number. Clips are short and every answer\nis one option letter, so a full pass over one config is cheap.\n\nLayout expected under <data_root>/mvbench (what `foveate download mvbench` makes):\n    json/<task>.json\n    video/<extracted zip folders>   e.g. video/star/Charades_v1_480/XXXX.mp4\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport os\nimport zipfile\nfrom pathlib import Path\n\nfrom . import LETTERS, Sample, register\n\nlog = logging.getLogger(__name__)\n\nHF_REPO = \"OpenGVLab/MVBench\"\n\n# task json -> (video folder prefix inside video/, input kind, has start/end bounds)\n# Mirrors the official evaluation script's data_list.\nTASKS: dict[str, tuple[str, str, bool]] = {\n    \"action_sequence\": (\"star/Charades_v1_480/\", \"video\", True),\n    \"action_prediction\": (\"star/Charades_v1_480/\", \"video\", True),\n    \"action_antonym\": (\"ssv2_video/\", \"video\", False),\n    \"fine_grained_action\": (\"Moments_in_Time_Raw/videos/\", \"video\", False),\n    \"unexpected_action\": (\"FunQA_test/test/\", \"video\", False),\n    \"object_existence\": (\"clevrer/video_validation/\", \"video\", False),\n    \"object_interaction\": (\"star/Charades_v1_480/\", \"video\", True),\n    \"object_shuffle\": (\"perception/videos/\", \"video\", False),\n    \"moving_direction\": (\"clevrer/video_validation/\", \"video\", False),\n    \"action_localization\": (\"sta/sta_video/\", \"video\", True),\n    \"scene_transition\": (\"scene_qa/video/\", \"video\", False),\n    \"action_count\": (\"perception/videos/\", \"video\", False),\n    \"moving_count\": (\"clevrer/video_validation/\", \"video\", False),\n    \"moving_attribute\": (\"clevrer/video_validation/\", \"video\", False),\n    \"state_change\": (\"perception/videos/\", \"video\", False),\n    \"fine_grained_pose\": (\"nturgbd/\", \"video\", False),  # NTU videos are not in the HF repo\n    \"character_order\": (\"perception/videos/\", \"video\", False),\n    \"egocentric_navigation\": (\"vlnqa/\", \"video\", False),\n    \"episodic_reasoning\": (\"tvqa/frames_fps3_hq/\", \"frame\", True),\n    \"counterfactual_inference\": (\"clevrer/video_validation/\", \"video\", False),\n}\nTVQA_FPS = 3.0\n\n# Zip archives needed per task (perception.zip alone is 11 GB).\nTASK_ZIPS = {\n    \"star\": [\"action_sequence\", \"action_prediction\", \"object_interaction\"],\n    \"ssv2_video\": [\"action_antonym\"],\n    \"Moments_in_Time_Raw\": [\"fine_grained_action\"],\n    \"FunQA_test\": [\"unexpected_action\"],\n    \"clevrer\": [\n        \"object_existence\", \"moving_direction\", \"moving_count\",\n        \"moving_attribute\", \"counterfactual_inference\",\n    ],\n    \"perception\": [\"object_shuffle\", \"action_count\", \"state_change\", \"character_order\"],\n    \"sta\": [\"action_localization\"],\n    \"scene_qa\": [\"scene_transition\"],\n    \"vlnqa\": [\"egocentric_navigation\"],\n    \"tvqa\": [\"episodic_reasoning\"],\n}\n\n# 15 tasks that skip perception.zip (11 GB) and the missing NTU videos:\n# use when disk or download time is tight (e.g. a Kaggle session).\nLITE_TASKS = sorted(\n    t for t in TASKS if t not in TASK_ZIPS[\"perception\"] and t != \"fine_grained_pose\"\n)\nDEFAULT_TASKS = sorted(t for t in TASKS if t != \"fine_grained_pose\")\n\n\ndef resolve_tasks(tasks: list[str] | str | None) -> list[str]:\n    if tasks is None or tasks == \"all\" or tasks == [\"all\"]:\n        return DEFAULT_TASKS\n    if tasks == \"lite\" or tasks == [\"lite\"]:\n        return LITE_TASKS\n    if isinstance(tasks, str):\n        tasks = tasks.split(\",\")\n    unknown = set(tasks) - set(TASKS)\n    if unknown:\n        raise ValueError(f\"unknown MVBench tasks: {sorted(unknown)}\")\n    return list(tasks)\n\n\ndef download(root: Path, tasks: list[str] | str | None = None, keep_zips: bool = False) -> None:\n    \"\"\"Fetch the JSON files plus only the video zips the chosen tasks need, then unzip.\"\"\"\n    from huggingface_hub import hf_hub_download, snapshot_download\n\n    tasks = resolve_tasks(tasks)\n    out = root / \"mvbench\"\n    out.mkdir(parents=True, exist_ok=True)\n    snapshot_download(HF_REPO, repo_type=\"dataset\", local_dir=out, allow_patterns=[\"json/*\"])\n    zips = sorted({z for z, ts in TASK_ZIPS.items() if set(ts) & set(tasks)})\n    vdir = out / \"video\"\n    vdir.mkdir(exist_ok=True)\n    for z in zips:\n        marker = vdir / f\".{z}.done\"\n        if marker.exists():\n            log.info(\"already extracted %s\", z)\n            continue\n        log.info(\"downloading %s.zip\", z)\n        path = hf_hub_download(HF_REPO, f\"video/{z}.zip\", repo_type=\"dataset\", local_dir=out)\n        log.info(\"extracting %s\", path)\n        with zipfile.ZipFile(path) as zf:\n            zf.extractall(vdir)\n        if not keep_zips:\n            os.remove(path)  # halve peak disk use\n        marker.touch()\n\n\ndef _index_files(vdir: Path) -> dict[str, Path]:\n    \"\"\"basename -> path, a fallback when a zip's folder layout differs from the script's.\"\"\"\n    idx: dict[str, Path] = {}\n    for p in vdir.rglob(\"*\"):\n        idx.setdefault(p.name, p)\n    return idx\n\n\n@register(\"mvbench\")\ndef load(root: str | Path | None = None, tasks: list[str] | str | None = None) -> list[Sample]:\n    from ..config import data_root\n\n    base = Path(root or data_root()) / \"mvbench\"\n    jdir, vdir = base / \"json\", base / \"video\"\n    if not jdir.exists():\n        raise FileNotFoundError(f\"{jdir} not found; run `foveate download mvbench` first\")\n    index: dict[str, Path] | None = None\n    samples: list[Sample] = []\n    missing: dict[str, int] = {}\n    for task in resolve_tasks(tasks):\n        prefix, kind, bound = TASKS[task]\n        with open(jdir / f\"{task}.json\") as f:\n            items = json.load(f)\n        for i, it in enumerate(items):\n            cands = list(it[\"candidates\"])\n            if it[\"answer\"] not in cands:\n                continue\n            letter = LETTERS[cands.index(it[\"answer\"])]\n            path = vdir / prefix / it[\"video\"]\n            if not path.exists():\n                if index is None:\n                    index = _index_files(vdir)\n                path = index.get(Path(it[\"video\"]).name, path)\n            if not path.exists():\n                missing[task] = missing.get(task, 0) + 1\n                continue\n            s = Sample(\n                qid=f\"{task}/{i:04d}\",\n                dataset=\"mvbench\",\n                task=task,\n                question=it[\"question\"],\n                answer=letter,\n                options=cands,\n                start=float(it[\"start\"]) if bound and \"start\" in it else None,\n                end=float(it[\"end\"]) if bound and \"end\" in it else None,\n                clip_id=str(path),\n            )\n            if kind == \"frame\":\n                frames = sorted(\n                    (p for p in path.iterdir() if p.suffix.lower() in {\".jpg\", \".jpeg\", \".png\"}),\n                    key=lambda p: (len(p.stem), p.stem),\n                )\n                if s.start is not None and s.end is not None:\n                    lo, hi = int(s.start * TVQA_FPS), int(s.end * TVQA_FPS) + 1\n                    frames = frames[lo:hi] or frames\n                s.frame_paths = [str(p) for p in frames]\n                s.start = s.end = None\n            else:\n                s.video = str(path)\n            samples.append(s)\n    if missing:\n        log.warning(\"MVBench: skipped questions with missing media %s\", missing)\n    log.info(\"MVBench: %d questions over %d tasks\", len(samples), len({s.task for s in samples}))\n    return samples\n",
"src/foveate/data/synthetic.py": "\"\"\"A tiny generated dataset: moving coloured squares on a road-grey background.\n\nUsed by the stage-1 profiler (compute cost does not depend on pixel content,\nso you can profile before any dataset is downloaded) and by CPU tests that run\nthe whole pipeline with a fake backend.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nfrom . import LETTERS, Sample, register\n\nCOLORS = {\"red\": (220, 40, 40), \"green\": (40, 180, 70), \"blue\": (40, 90, 220), \"yellow\": (230, 210, 40)}\n\n\ndef generate(root: Path, n_clips: int = 12, n_frames: int = 32, size=(1280, 720), seed: int = 0):\n    rng = random.Random(seed)\n    out = root / \"synthetic\"\n    items = []\n    for c in range(n_clips):\n        clip = out / f\"clip{c:03d}\"\n        clip.mkdir(parents=True, exist_ok=True)\n        color = rng.choice(list(COLORS))\n        direction = rng.choice([\"left\", \"right\"])\n        for f in range(n_frames):\n            p = clip / f\"{f:04d}.jpg\"\n            if not p.exists():\n                img = Image.new(\"RGB\", size, (90, 90, 95))\n                d = ImageDraw.Draw(img)\n                d.rectangle([0, size[1] * 0.55, size[0], size[1] * 0.6], fill=(230, 230, 230))\n                x = int((f / (n_frames - 1)) * (size[0] - 200))\n                if direction == \"left\":\n                    x = size[0] - 200 - x\n                d.rectangle([x, 300, x + 160, 420], fill=COLORS[color])\n                arr = np.asarray(img).astype(np.int16)\n                arr += np.random.default_rng(c * 1000 + f).integers(-6, 7, arr.shape, dtype=np.int16)\n                Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8)).save(p, quality=90)\n        items.append((clip, color, direction))\n    return items\n\n\n@register(\"synthetic\")\ndef load(root: str | Path | None = None, tasks=None) -> list[Sample]:\n    from ..config import data_root\n\n    root = Path(root or data_root())\n    samples = []\n    for clip, color, direction in generate(root):\n        frames = sorted(str(p) for p in clip.glob(\"*.jpg\"))\n        colors = sorted(COLORS)\n        samples.append(Sample(\n            qid=f\"color/{clip.name}\", dataset=\"synthetic\", task=\"color\",\n            question=\"What color is the moving block?\",\n            options=colors, answer=LETTERS[colors.index(color)],\n            frame_paths=frames, clip_id=clip.name,\n        ))\n        dirs = [\"left\", \"right\"]\n        samples.append(Sample(\n            qid=f\"direction/{clip.name}\", dataset=\"synthetic\", task=\"direction\",\n            question=\"Which direction is the block moving?\",\n            options=dirs, answer=LETTERS[dirs.index(direction)],\n            frame_paths=frames, clip_id=clip.name,\n        ))\n    return samples\n",
"src/foveate/drift.py": "\"\"\"Stage 4: distribution shift. Does the cascade spend more where inputs get harder,\nand can a cheap drift monitor tell the cascade to start higher?\n\nThree measurements, all replayed from the grid cache:\n  1. Per corruption: accuracy, escalation rate and cost of the cascade (with the\n     threshold fit on clean data) next to the cheapest and most expensive static\n     configs. The cascade should escalate more on shifted inputs.\n  2. A simulated stream that switches from clean to shifted halfway. A window\n     detector on model-free image statistics (brightness, contrast, sharpness,\n     saturation) raises an alarm; escalation rate and confidence are tracked too.\n  3. Drift-aware cascade: while the alarm is on, skip the cheapest rung (its\n     answers are mostly escalated anyway). Compare cost and accuracy on the\n     shifted half with the plain cascade.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\n\nimport numpy as np\nimport pandas as pd\n\nfrom .policy import Table, build_table, cascade, evaluate\nfrom .stats import auroc\n\nlog = logging.getLogger(__name__)\n\nSTAT_COLS = [\"img_brightness\", \"img_contrast\", \"img_sharpness\", \"img_saturation\"]\n\n\ndef window_z(ref: np.ndarray, win: np.ndarray) -> float:\n    \"\"\"Largest standardized mean shift across features (a two-sample z statistic).\"\"\"\n    mu, sd = ref.mean(0), ref.std(0) + 1e-9\n    z = (win.mean(0) - mu) / (sd / np.sqrt(len(win)))\n    return float(np.max(np.abs(z)))\n\n\ndef stream_experiment(\n    df: pd.DataFrame, clean: Table, shifted: Table, corruption: str, ladder: list[str],\n    tau: float, window: int = 50, z_alarm: float = 6.0, seed: int = 0,\n) -> dict:\n    rng = np.random.default_rng(seed)\n    common = np.intersect1d(clean.qids, shifted.qids)\n    if len(common) < 2 * window:\n        window = max(5, len(common) // 4)\n    perm = rng.permutation(common)\n    half = len(perm) // 2\n    stream_q = np.concatenate([perm[:half], perm[half:]])\n    is_shift = np.r_[np.zeros(half, bool), np.ones(len(perm) - half, bool)]\n\n    cheap = ladder[0]\n    feats = {}\n    for corr, tab in ((\"clean\", clean), (corruption, shifted)):\n        d = df[(df[\"corruption\"] == corr) & (df[\"config\"] == cheap)].set_index(\"qid\")\n        feats[corr] = d[STAT_COLS]\n    X = np.stack([\n        feats[corruption if sh else \"clean\"].loc[q].to_numpy(float) for q, sh in zip(stream_q, is_shift)\n    ])\n    ref = X[:window]\n\n    # Per-question cascade outcome on the stream, plain and drift-aware.\n    def outcome(tab: Table, q: str, start: int):\n        i = int(np.where(tab.qids == q)[0][0])\n        sub = tab.subset(np.arange(len(tab.qids)) == i)\n        f, c, r = cascade(sub, ladder, tau, start=np.array([start]))\n        return float(sub.correct[0, f[0]]), float(c[0]), int(r[0]), float(sub.conf[0, sub.col(cheap)])\n\n    z_trace, alarm, plain, aware = [], [], [], []\n    on = False\n    for k, (q, sh) in enumerate(zip(stream_q, is_shift)):\n        if k >= window:\n            z = window_z(ref, X[k - window:k])\n            on = z > z_alarm\n        else:\n            z = 0.0\n        z_trace.append(z)\n        alarm.append(on)\n        tab = shifted if sh else clean\n        plain.append(outcome(tab, q, 0))\n        aware.append(outcome(tab, q, 1 if on and len(ladder) > 1 else 0))\n    plain_a, aware_a = np.array(plain), np.array(aware)\n    first_alarm = next((k for k, a in enumerate(alarm) if a and is_shift[k]), None)\n    false_alarms = int(np.sum(np.array(alarm) & ~is_shift))\n\n    def seg(arr, m):\n        return {\"accuracy\": float(arr[m, 0].mean()), \"mean_cost\": float(arr[m, 1].mean()),\n                \"escalation_rate\": float((arr[m, 2] > 1).mean())}\n\n    return {\n        \"corruption\": corruption, \"window\": window, \"z_alarm\": z_alarm,\n        \"detection_delay\": (first_alarm - half) if first_alarm is not None else None,\n        \"false_alarms_on_clean\": false_alarms,\n        \"plain\": {\"clean\": seg(plain_a, ~is_shift), \"shifted\": seg(plain_a, is_shift)},\n        \"drift_aware\": {\"clean\": seg(aware_a, ~is_shift), \"shifted\": seg(aware_a, is_shift)},\n        \"trace\": {\n            \"z\": z_trace, \"alarm\": alarm, \"is_shift\": is_shift.tolist(),\n            \"escalated\": (plain_a[:, 2] > 1).astype(int).tolist(),\n            \"cheap_conf\": plain_a[:, 3].tolist(),\n        },\n    }\n\n\ndef shift_report(df: pd.DataFrame, costs: pd.DataFrame, ladder: list[str], tau: float,\n                 model_ids: dict[str, str]) -> dict:\n    corrs = [c for c in df[\"corruption\"].unique() if c != \"clean\"]\n    if not corrs:\n        raise ValueError(\"no shifted rows; run `foveate grid --corruption fog:3` (or `foveate shift`)\")\n    clean = build_table(df, costs, corruption=\"clean\", model_ids=model_ids)\n    out = {\"tau\": tau, \"ladder\": ladder, \"per_corruption\": [], \"streams\": []}\n    for corr in sorted(corrs):\n        sh = build_table(df, costs, corruption=corr, model_ids=model_ids)\n        lad = [c for c in ladder if c in sh.configs and c in clean.configs]\n        sh.cost = np.array([clean.cost[clean.col(c)] if c in clean.configs else np.nan for c in sh.configs])\n        cm = np.isin(clean.qids, sh.qids)\n        cl = clean.subset(cm)\n        rec = {\"corruption\": corr, \"n\": int(len(sh.qids))}\n        for tag, tab in ((\"clean\", cl), (\"shifted\", sh)):\n            r = evaluate(tab, cascade(tab, lad, tau), \"cascade\")\n            rec[f\"{tag}_cascade_acc\"] = r[\"accuracy\"]\n            rec[f\"{tag}_escalation\"] = r[\"escalation_rate\"]\n            rec[f\"{tag}_cost\"] = r[\"mean_cost\"]\n            rec[f\"{tag}_cheap_acc\"] = float(tab.correct[:, tab.col(lad[0])].mean())\n            rec[f\"{tag}_top_acc\"] = float(tab.correct[:, tab.col(lad[-1])].mean())\n            rec[f\"{tag}_cheap_conf\"] = float(tab.conf[:, tab.col(lad[0])].mean())\n            rec[f\"{tag}_cheap_auroc\"] = auroc(tab.conf[:, tab.col(lad[0])], tab.correct[:, tab.col(lad[0])])\n        out[\"per_corruption\"].append(rec)\n        out[\"streams\"].append(stream_experiment(df, cl, sh, corr, lad, tau))\n    return out\n",
"src/foveate/grid.py": "\"\"\"Stage 2: run every question through every visual config once, and cache it.\n\nEach row stores the answer, three confidence signals, token counts, and image\nstatistics. Every policy in stage 3 (static, rules, cascade, oracle) and the\ndrift study in stage 4 is then evaluated by *replaying* these rows, so trying a\nnew policy costs seconds instead of GPU hours. Cost per config is measured\ntwice: amortised throughput (batched GPU-seconds per question) and batch-1\nlatency from a small probe.\n\nOutput layout (one directory per dataset+subset):\n    results/<dataset>_<subset>/grid/<model>_<F>x<R>[__<corruption>].s<i>of<n>.jsonl\n    results/<dataset>_<subset>/cost/<same stem>.json\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nfrom .config import VisualConfig, cache_root, results_root\nfrom .data import Sample, shard\nfrom .data.frames import FrameStore\nfrom .tokens import visual_tokens\nfrom .utils import capture_env, percentile, read_jsonl, write_jsonl\n\nlog = logging.getLogger(__name__)\n\n\ndef run_dir(dataset: str, subset: str) -> Path:\n    return results_root() / f\"{dataset}_{subset}\"\n\n\ndef stem(cfg: VisualConfig, corruption: str | None) -> str:\n    s = f\"{cfg.model}_{cfg.frames}x{cfg.res}\"\n    if corruption:\n        s += \"__\" + corruption.replace(\":\", \"\")\n    return s\n\n\ndef _done_qids(path: Path) -> set[str]:\n    if not path.exists():\n        return set()\n    return {r[\"qid\"] for r in read_jsonl(path)}\n\n\ndef run_grid(\n    backend,\n    samples: list[Sample],\n    configs: list[VisualConfig],\n    *,\n    dataset: str,\n    subset: str,\n    corruption: str | None = None,\n    shard_id: int = 0,\n    num_shards: int = 1,\n    batch_size: int = 32,\n    probe: int = 16,\n    workers: int = 4,\n    max_new_tokens: int = 48,\n    store_res: int | None = None,\n) -> None:\n    \"\"\"Run `configs` (all for one model) over this shard of `samples`, resumably.\"\"\"\n    if not configs:\n        log.info(\"no configs to run\")\n        return\n    out_dir = run_dir(dataset, subset)\n    my = shard(samples, shard_id, num_shards)\n    store_res = store_res or max(c.res for c in configs)\n    # Cached frames are keyed by their stored resolution so a low-res run can\n    # never feed upscaled frames to a later high-res run.\n    store = FrameStore(cache_root() / f\"r{store_res}\", frame_counts={c.frames for c in configs},\n                       store_res=store_res)\n    env = capture_env()\n    pool = ThreadPoolExecutor(max_workers=workers)\n    # Order configs by frame count so cached frames are reused immediately.\n    for cfg in sorted(configs, key=lambda c: (c.frames, c.res)):\n        name = stem(cfg, corruption)\n        path = out_dir / \"grid\" / f\"{name}.s{shard_id}of{num_shards}.jsonl\"\n        done = _done_qids(path)\n        todo = [s for s in my if s.qid not in done]\n        log.info(\"[%s] %d to run (%d already done) on shard %d/%d\",\n                 name, len(todo), len(done), shard_id, num_shards)\n        load = lambda s, c=cfg: store.load(s, c.frames, c.res, corruption)  # noqa: E731\n        gen_wall = 0.0\n        n_run = 0\n        chunks = [todo[i:i + batch_size] for i in range(0, len(todo), batch_size)]\n        nxt = pool.map(load, chunks[0]) if chunks else None\n        t_cfg = time.perf_counter()\n        for ci, chunk in enumerate(chunks):\n            fbs = list(nxt)\n            # Prefetch the next chunk's frames on CPU threads while the GPU works.\n            nxt = pool.map(load, chunks[ci + 1]) if ci + 1 < len(chunks) else None\n            preds, wall = backend.predict_batch(chunk, fbs, max_new_tokens=max_new_tokens)\n            gen_wall += wall\n            n_run += len(chunk)\n            rows = []\n            for s, fb, p in zip(chunk, fbs, preds):\n                row = {\n                    \"qid\": s.qid, \"dataset\": s.dataset, \"task\": s.task, \"clip_id\": s.clip_id,\n                    \"question\": s.question,\n                    \"config\": cfg.name, \"model\": cfg.model, \"frames\": cfg.frames, \"res\": cfg.res,\n                    \"corruption\": corruption, \"answer\": s.answer,\n                    \"visual_tokens_formula\": visual_tokens(fb.height, fb.width, cfg.frames),\n                    \"frame_hw\": [fb.height, fb.width], \"load_ms\": fb.load_s * 1e3,\n                    **{f\"img_{k}\": v for k, v in fb.stats.items()},\n                    **p,\n                }\n                if s.is_mc:\n                    row[\"correct\"] = bool(p.get(\"pred\") == s.answer)\n                rows.append(row)\n            write_jsonl(path, rows, append=True)\n            if ci == 0 and chunk[0].is_mc:\n                _sanity_check(rows, name)\n            if (ci + 1) % 10 == 0:\n                log.info(\"[%s] %d/%d\", name, n_run, len(todo))\n        # Batch-1 latency probe on the first few questions of the shard.\n        lat = []\n        if probe and my:\n            probe_s = my[:probe]\n            probe_fb = list(pool.map(load, probe_s))\n            lat = backend.latency_probe(probe_s, probe_fb, max_new_tokens=max_new_tokens)\n        cost = {\n            \"config\": cfg.name, \"corruption\": corruption, \"shard\": [shard_id, num_shards],\n            \"n_run\": n_run, \"gen_wall_s\": gen_wall,\n            \"total_wall_s\": time.perf_counter() - t_cfg,\n            \"amortized_ms\": (gen_wall / n_run * 1e3) if n_run else None,\n            \"latency_b1_ms_p50\": percentile(lat, 50) if lat else None,\n            \"latency_b1_ms_p95\": percentile(lat, 95) if lat else None,\n            \"latency_b1_ms\": lat, \"batch_size\": batch_size, \"env\": env,\n            \"backend\": type(backend).__name__,\n        }\n        cpath = out_dir / \"cost\" / f\"{name}.s{shard_id}of{num_shards}.json\"\n        cpath.parent.mkdir(parents=True, exist_ok=True)\n        if n_run or not cpath.exists():\n            cpath.write_text(json.dumps(cost, indent=1))\n    pool.shutdown()\n\n\ndef _sanity_check(rows: list[dict], name: str) -> None:\n    \"\"\"fp16 overflow on GPUs without bf16 shows up as NaN or one constant answer.\"\"\"\n    confs = [r.get(\"conf\") for r in rows]\n    preds = [r.get(\"pred\") for r in rows]\n    if any(c is None or c != c for c in confs):\n        log.warning(\"[%s] NaN confidences in the first batch: try --dtype float32\", name)\n    elif len(rows) >= 16 and len(set(preds)) == 1:\n        log.warning(\"[%s] every answer in the first batch is %r: check dtype/prompt\", name, preds[0])\n    acc = sum(r.get(\"correct\", False) for r in rows) / max(1, len(rows))\n    log.info(\"[%s] first-batch accuracy %.2f (chance is ~0.3 on MVBench)\", name, acc)\n\n\nclass HFGridAdapter:\n    \"\"\"Lets the Transformers backend stand in for vLLM in run_grid (slow, batch 1).\"\"\"\n\n    def __init__(self, hf_backend):\n        self.b = hf_backend\n\n    def predict_batch(self, samples, batches, max_new_tokens=48):\n        t0 = time.perf_counter()\n        rows = [self.b.predict(s, fb, max_new_tokens) for s, fb in zip(samples, batches)]\n        return rows, time.perf_counter() - t0\n\n    def latency_probe(self, samples, batches, max_new_tokens=48):\n        return [self.b.predict(s, fb, max_new_tokens)[\"latency_ms\"] for s, fb in zip(samples, batches)]\n",
"src/foveate/judge.py": "\"\"\"Lingo-Judge scoring for LingoQA's free-form answers.\n\nLingo-Judge (wayveai/Lingo-Judge) is a small text classifier trained by the\nLingoQA authors to predict whether a free-form answer matches the reference;\nthe paper reports it correlates better with human ratings than BLEU, METEOR,\nCIDEr or a GPT-4 judge on this data. A prediction counts as correct when its\nscore against either reference answer exceeds 0.5.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nfrom pathlib import Path\n\nfrom .utils import read_jsonl, write_jsonl\n\nlog = logging.getLogger(__name__)\n\nJUDGE_ID = \"wayveai/Lingo-Judge\"\n\n\ndef judge_inputs(question: str, reference: str, prediction: str) -> str:\n    return f\"[CLS]\\nQuestion: {question}\\nAnswer: {reference}\\nStudent: {prediction}\"\n\n\ndef judge_dir(run_dir: Path, samples_by_qid: dict, device: int | str = 0, batch_size: int = 64):\n    \"\"\"Add `judge_score` and `correct` to every LingoQA grid row that lacks them.\"\"\"\n    from transformers import pipeline\n\n    pipe = pipeline(\"text-classification\", model=JUDGE_ID, device=device)\n    files = sorted((run_dir / \"grid\").glob(\"*.jsonl\"))\n    for f in files:\n        rows = list(read_jsonl(f))\n        todo = [r for r in rows if \"judge_score\" not in r]\n        if not todo:\n            continue\n        log.info(\"judging %d rows in %s\", len(todo), f.name)\n        texts, owners = [], []\n        for r in todo:\n            s = samples_by_qid[r[\"qid\"]]\n            for ref in s.references or [s.answer]:\n                texts.append(judge_inputs(s.question, ref, r.get(\"pred\") or \"\"))\n                owners.append(r)\n        scores = pipe(texts, batch_size=batch_size, truncation=True)\n        best: dict[int, float] = {}\n        for r, sc in zip(owners, scores):\n            best[id(r)] = max(best.get(id(r), 0.0), float(sc[\"score\"]))\n        for r in todo:\n            r[\"judge_score\"] = best[id(r)]\n            r[\"correct\"] = r[\"judge_score\"] > 0.5\n        write_jsonl(f, rows)\n",
"src/foveate/policy.py": "\"\"\"Stage 3: compute-allocation policies, evaluated by replaying the grid cache.\n\nEvery policy maps a question to the sequence of configs it runs. Accuracy is\nthat of the last config run; cost is the sum over every config run (a cascade\npays for its failed cheap attempts too).\n\nPolicies:\n  static      one config for everything (the baselines)\n  rules       question keywords -> config, mapping learned on a calibration split\n  cascade     start cheap, escalate along a ladder while confidence < tau\n  rules+casc  rules pick the starting rung, then the cascade escalates\n  oracle      cheapest config that answers correctly (an unreachable upper bound\n              that shows how much headroom adaptive allocation has)\n\nThresholds and rule mappings are fit on a calibration half and reported on a\nheld-out half, so the headline number is not tuned on its own test data.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport re\nfrom collections import defaultdict\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .data import stable_hash\nfrom .stats import auroc, bootstrap_ci, ece, paired_bootstrap_diff\nfrom .tokens import MODEL_SHAPES, prefill_flops, vit_patches\nfrom .utils import read_jsonl\n\nlog = logging.getLogger(__name__)\n\n# ---------------------------------------------------------------------------\n# Loading the grid cache\n# ---------------------------------------------------------------------------\n\n\ndef load_rows(run_dir: Path, corruption: str | None = None) -> pd.DataFrame:\n    rows = []\n    for f in sorted((run_dir / \"grid\").glob(\"*.jsonl\")):\n        rows.extend(read_jsonl(f))\n    if not rows:\n        raise FileNotFoundError(f\"no grid rows under {run_dir}/grid; run `foveate grid` first\")\n    df = pd.DataFrame(rows)\n    if \"corruption\" not in df:\n        df[\"corruption\"] = None\n    df[\"corruption\"] = df[\"corruption\"].fillna(\"clean\")\n    if corruption is not None:\n        df = df[df[\"corruption\"] == corruption]\n    if \"correct\" not in df or df[\"correct\"].isna().any():\n        missing = df[\"correct\"].isna().sum() if \"correct\" in df else len(df)\n        if missing:\n            log.warning(\"%d rows have no correctness label (LingoQA needs `foveate judge`)\", missing)\n            df = df[df[\"correct\"].notna()] if \"correct\" in df else df.iloc[0:0]\n    return df.drop_duplicates([\"qid\", \"config\", \"corruption\"], keep=\"last\")\n\n\ndef load_costs(run_dir: Path) -> pd.DataFrame:\n    \"\"\"Per (config, corruption): amortised ms/question and batch-1 latency, merged over shards.\"\"\"\n    acc: dict[tuple, dict] = defaultdict(lambda: {\"gen\": 0.0, \"n\": 0, \"lat\": []})\n    for f in sorted((run_dir / \"cost\").glob(\"*.json\")):\n        c = json.loads(f.read_text())\n        k = (c[\"config\"], c.get(\"corruption\") or \"clean\")\n        acc[k][\"gen\"] += c.get(\"gen_wall_s\") or 0.0\n        acc[k][\"n\"] += c.get(\"n_run\") or 0\n        acc[k][\"lat\"].extend(c.get(\"latency_b1_ms\") or [])\n    recs = []\n    for (cfg, corr), a in acc.items():\n        recs.append({\n            \"config\": cfg, \"corruption\": corr,\n            \"amortized_ms\": a[\"gen\"] / a[\"n\"] * 1e3 if a[\"n\"] else np.nan,\n            \"latency_b1_ms\": float(np.median(a[\"lat\"])) if a[\"lat\"] else np.nan,\n            \"latency_b1_p95_ms\": float(np.percentile(a[\"lat\"], 95)) if a[\"lat\"] else np.nan,\n        })\n    return pd.DataFrame(recs)\n\n\ndef flops_cost(df: pd.DataFrame, model_ids: dict[str, str]) -> pd.Series:\n    \"\"\"Model-derived prefill TFLOPs per question (used when timings are missing).\"\"\"\n    out = {}\n    for cfg, g in df.groupby(\"config\"):\n        model = cfg.split(\":\")[0]\n        shape = MODEL_SHAPES.get(model_ids.get(model, model))\n        if shape is None:\n            continue\n        h, w = g[\"frame_hw\"].iloc[0]\n        toks = g[\"prompt_tokens\"].mean()\n        out[cfg] = prefill_flops(shape, toks, vit_patches(h, w, int(g[\"frames\"].iloc[0])))[\"total\"] / 1e12\n    return pd.Series(out)\n\n\n@dataclass\nclass Table:\n    qids: np.ndarray\n    tasks: np.ndarray\n    questions: np.ndarray\n    configs: list[str]\n    correct: np.ndarray  # [Q, C] float (0/1)\n    conf: np.ndarray  # [Q, C]\n    cost: np.ndarray  # [C]\n    cost_name: str\n\n    def col(self, cfg: str) -> int:\n        return self.configs.index(cfg)\n\n    def subset(self, mask: np.ndarray) -> \"Table\":\n        return Table(self.qids[mask], self.tasks[mask], self.questions[mask], self.configs,\n                     self.correct[mask], self.conf[mask], self.cost, self.cost_name)\n\n\ndef build_table(\n    df: pd.DataFrame, costs: pd.DataFrame, cost_metric: str = \"latency_b1_ms\",\n    corruption: str = \"clean\", model_ids: dict[str, str] | None = None,\n    conf_field: str = \"conf\",\n) -> Table:\n    d = df[df[\"corruption\"] == corruption]\n    configs = sorted(d[\"config\"].unique(), key=_cfg_key)\n    piv_c = d.pivot_table(index=\"qid\", columns=\"config\", values=\"correct\", aggfunc=\"last\")\n    piv_p = d.pivot_table(index=\"qid\", columns=\"config\", values=conf_field, aggfunc=\"last\")\n    piv_c = piv_c[configs].dropna()\n    piv_p = piv_p.loc[piv_c.index, configs].fillna(0.0)\n    dropped = d[\"qid\"].nunique() - len(piv_c)\n    if dropped:\n        log.info(\"%d questions are missing from at least one config and were dropped\", dropped)\n    meta = d.drop_duplicates(\"qid\").set_index(\"qid\")\n    # Cost per config: measured if available (clean costs reused for shifted\n    # runs, since corruption does not change the compute), else model FLOPs.\n    cmap = {}\n    c = costs[costs[\"corruption\"] == \"clean\"].set_index(\"config\") if len(costs) else None\n    fl = flops_cost(d, model_ids or {})\n    name = cost_metric\n    for cfg in configs:\n        v = np.nan\n        if c is not None and cfg in c.index and cost_metric in c:\n            v = c.loc[cfg, cost_metric]\n        if np.isnan(v):\n            v = fl.get(cfg, np.nan)\n            name = f\"{cost_metric} (TFLOP fallback for some configs)\"\n        cmap[cfg] = v\n    q = piv_c.index.to_numpy()\n    return Table(\n        qids=q, tasks=meta.loc[q, \"task\"].to_numpy(),\n        questions=meta.loc[q, \"question\"].to_numpy() if \"question\" in meta else np.array([\"\"] * len(q)),\n        configs=configs, correct=piv_c.to_numpy(dtype=float), conf=piv_p.to_numpy(dtype=float),\n        cost=np.array([cmap[c] for c in configs], dtype=float), cost_name=name,\n    )\n\n\ndef _cfg_key(name: str):\n    model, fr = name.split(\":\")\n    f, r = fr.split(\"x\")\n    return (model, int(f), int(r))\n\n\ndef calib_split(t: Table, frac: float = 0.5, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Stratified (by task) deterministic split into calibration / test masks.\"\"\"\n    calib = np.zeros(len(t.qids), dtype=bool)\n    for task in np.unique(t.tasks):\n        idx = np.where(t.tasks == task)[0]\n        idx = sorted(idx, key=lambda i: stable_hash(f\"{seed}:{t.qids[i]}\"))\n        calib[idx[: int(round(len(idx) * frac))]] = True\n    return calib, ~calib\n\n\n# ---------------------------------------------------------------------------\n# Policies. Each returns (final_config_index [Q], cost [Q], n_runs [Q]).\n# ---------------------------------------------------------------------------\n\n\ndef static(t: Table, cfg: str):\n    j = t.col(cfg)\n    n = len(t.qids)\n    return np.full(n, j), np.full(n, t.cost[j]), np.ones(n, dtype=int)\n\n\ndef oracle(t: Table):\n    order = np.argsort(t.cost)\n    final = np.full(len(t.qids), order[0])\n    for i in range(len(t.qids)):\n        for j in order:\n            if t.correct[i, j]:\n                final[i] = j\n                break\n    return final, t.cost[final], np.ones(len(t.qids), dtype=int)\n\n\ndef cascade(t: Table, ladder: list[str], tau: float, start: np.ndarray | None = None):\n    \"\"\"Run ladder rungs in order; stop at the first rung with confidence >= tau.\"\"\"\n    cols = [t.col(c) for c in ladder]\n    n = len(t.qids)\n    start = np.zeros(n, dtype=int) if start is None else start\n    final = np.empty(n, dtype=int)\n    cost = np.zeros(n)\n    runs = np.zeros(n, dtype=int)\n    for i in range(n):\n        for k in range(start[i], len(cols)):\n            j = cols[k]\n            cost[i] += t.cost[j]\n            runs[i] += 1\n            final[i] = j\n            if t.conf[i, j] >= tau:\n                break\n    return final, cost, runs\n\n\nCATEGORIES = {\n    \"count\": r\"\\bhow many\\b|\\bnumber of\\b|\\bcount\\b|\\btimes\\b\",\n    \"temporal\": r\"\\bbefore\\b|\\bafter\\b|\\bthen\\b|\\bfirst\\b|\\blast\\b|\\bsequence\\b|\\border\\b|\"\n                r\"\\bhappen|\\bnext\\b|\\bbegin|\\bend\\b|\\bduring\\b|\\bwhile\\b|\\bchange|\\btransition\",\n    \"motion\": r\"\\bmov(e|ing|ement)\\b|\\bdirection\\b|\\btoward|\\brotat|\\bturn|\\bspeed\\b|\"\n              r\"\\bcollid|\\bcollision\\b|\\bleft\\b|\\bright\\b|\\bbrak|\\blane\\b\",\n    \"detail\": r\"\\bcolou?r\\b|\\btext\\b|\\bread\\b|\\bsign\\b|\\bshape\\b|\\bmaterial\\b|\\bwearing\\b|\"\n              r\"\\bholding\\b|\\blight\\b|\\bletter\\b|\\bsmall\\b\",\n}\n\n\ndef categorize(question: str) -> str:\n    q = question.lower()\n    for name, pat in CATEGORIES.items():\n        if re.search(pat, q):\n            return name\n    return \"other\"\n\n\ndef fit_rules(t: Table, candidates: list[str], tol: float = 0.01) -> dict[str, str]:\n    \"\"\"Per question category: cheapest candidate within `tol` of that category's best accuracy.\"\"\"\n    cats = np.array([categorize(q) for q in t.questions])\n    mapping = {}\n    for cat in sorted(set(cats) | {\"other\"}):\n        m = cats == cat\n        if not m.any():\n            m = np.ones(len(cats), dtype=bool)\n        accs = {c: t.correct[m, t.col(c)].mean() for c in candidates}\n        best = max(accs.values())\n        ok = [c for c in candidates if accs[c] >= best - tol]\n        mapping[cat] = min(ok, key=lambda c: t.cost[t.col(c)])\n    return mapping\n\n\ndef rules(t: Table, mapping: dict[str, str]):\n    cats = [categorize(q) for q in t.questions]\n    final = np.array([t.col(mapping.get(c, mapping[\"other\"])) for c in cats])\n    return final, t.cost[final], np.ones(len(final), dtype=int)\n\n\n# ---------------------------------------------------------------------------\n# Evaluation\n# ---------------------------------------------------------------------------\n\n\ndef evaluate(t: Table, result, name: str, ref: str | None = None) -> dict:\n    final, cost, runs = result\n    correct = t.correct[np.arange(len(final)), final]\n    acc, lo, hi = bootstrap_ci(correct)\n    out = {\n        \"policy\": name, \"n\": int(len(final)), \"accuracy\": acc, \"acc_lo\": lo, \"acc_hi\": hi,\n        \"mean_cost\": float(np.mean(cost)), \"escalation_rate\": float(np.mean(runs > 1)),\n        \"mean_runs\": float(np.mean(runs)),\n    }\n    if ref is not None:\n        rc = t.correct[:, t.col(ref)]\n        d, dlo, dhi = paired_bootstrap_diff(correct, rc)\n        out.update({\"vs_ref\": ref, \"acc_diff\": d, \"acc_diff_lo\": dlo, \"acc_diff_hi\": dhi,\n                    \"cost_saving\": 1 - float(np.mean(cost)) / t.cost[t.col(ref)]})\n    return out\n\n\ndef pareto(points: list[dict]) -> list[dict]:\n    pts = sorted(points, key=lambda p: (p[\"mean_cost\"], -p[\"accuracy\"]))\n    front, best = [], -1.0\n    for p in pts:\n        if p[\"accuracy\"] > best:\n            front.append(p)\n            best = p[\"accuracy\"]\n    return front\n\n\ndef confidence_report(t: Table) -> list[dict]:\n    \"\"\"How well does each config's confidence predict its own correctness?\"\"\"\n    out = []\n    for j, c in enumerate(t.configs):\n        out.append({\"config\": c, \"accuracy\": float(t.correct[:, j].mean()),\n                    \"auroc\": auroc(t.conf[:, j], t.correct[:, j]),\n                    \"ece\": ece(t.conf[:, j], t.correct[:, j]), \"cost\": float(t.cost[j])})\n    return out\n\n\ndef replay(t: Table, ladder: list[str], ref: str | None = None, tol: float = 0.0, seed: int = 0) -> dict:\n    \"\"\"Fit on calibration half, report on test half, and draw full curves.\"\"\"\n    ladder = [c for c in ladder if c in t.configs]\n    if not ladder:\n        raise ValueError(\"none of the ladder configs are in the grid\")\n    ref = ref or max(t.configs, key=lambda c: t.correct[:, t.col(c)].mean())\n    calib_m, test_m = calib_split(t, seed=seed)\n    cal, test = t.subset(calib_m), t.subset(test_m)\n\n    taus = np.unique(np.quantile(t.conf[:, [t.col(c) for c in ladder]], np.linspace(0, 1, 41)))\n    taus = np.concatenate([[0.0], taus, [1.01]])\n\n    # Curves on all data (for the figure).\n    curve = []\n    for tau in taus:\n        r = evaluate(t, cascade(t, ladder, tau), \"cascade\")\n        r[\"tau\"] = float(tau)\n        curve.append(r)\n\n    # Threshold: cheapest tau whose calibration accuracy reaches the reference's minus tol.\n    target = cal.correct[:, cal.col(ref)].mean() - tol\n    cands = [(tau, evaluate(cal, cascade(cal, ladder, tau), \"c\")) for tau in taus]\n    ok = [(tau, r) for tau, r in cands if r[\"accuracy\"] >= target]\n    tau_star = min(ok, key=lambda x: x[1][\"mean_cost\"])[0] if ok else taus[-1]\n\n    mapping = fit_rules(cal, sorted(set(ladder) | {ref}), tol=0.01)\n    start_map = {cat: (ladder.index(c) if c in ladder else 0) for cat, c in mapping.items()}\n    start_test = np.array([start_map.get(categorize(q), 0) for q in test.questions])\n    start_cal = np.array([start_map.get(categorize(q), 0) for q in cal.questions])\n    cands_rc = [(tau, evaluate(cal, cascade(cal, ladder, tau, start_cal), \"c\")) for tau in taus]\n    ok_rc = [(tau, r) for tau, r in cands_rc if r[\"accuracy\"] >= target]\n    tau_rc = min(ok_rc, key=lambda x: x[1][\"mean_cost\"])[0] if ok_rc else taus[-1]\n\n    policies = [evaluate(test, static(test, c), f\"static {c}\", ref) for c in t.configs]\n    policies.append(evaluate(test, rules(test, mapping), \"rules\", ref))\n    policies.append(evaluate(test, cascade(test, ladder, tau_star), f\"cascade (tau={tau_star:.3f})\", ref))\n    policies.append(evaluate(test, cascade(test, ladder, tau_rc, start_test),\n                             f\"rules+cascade (tau={tau_rc:.3f})\", ref))\n    policies.append(evaluate(test, oracle(test), \"oracle (upper bound)\", ref))\n    # Ablations: drop one rung at a time from the cascade.\n    ablations = []\n    for drop in ladder[:-1]:\n        lad = [c for c in ladder if c != drop]\n        ablations.append(evaluate(test, cascade(test, lad, tau_star), f\"cascade without {drop}\", ref))\n\n    per_task = []\n    for task in np.unique(t.tasks):\n        m = t.tasks == task\n        row = {\"task\": task, \"n\": int(m.sum())}\n        for c in t.configs:\n            row[c] = float(t.correct[m, t.col(c)].mean())\n        per_task.append(row)\n\n    return {\n        \"cost_metric\": t.cost_name, \"reference\": ref, \"ladder\": ladder, \"tau\": float(tau_star),\n        \"tau_rules_cascade\": float(tau_rc), \"rules\": mapping, \"n_calib\": int(calib_m.sum()),\n        \"n_test\": int(test_m.sum()), \"policies\": policies, \"ablations\": ablations,\n        \"frontier\": [p[\"policy\"] for p in pareto(policies)], \"curve\": curve,\n        \"confidence\": confidence_report(t), \"per_task\": per_task,\n        \"costs\": dict(zip(t.configs, map(float, t.cost))),\n    }\n",
"src/foveate/profile.py": "\"\"\"Stage 1: where does the time go? A batch-1 sweep over frames x resolution.\n\nFor each setting the Transformers backend reports, per request:\n    load (decode + resize, CPU) | preprocess (processor, CPU) | H2D copy |\n    vision encoder | LLM prefill | decode (TPOT x tokens)\nplus visual-token counts and peak memory. Every setting gets warm-up runs that\nare discarded (first calls pay for CUDA context, cuBLAS heuristics and memory\nallocation), then `repeats` timed runs; we keep medians and p95.\n\nThe second sweep fixes the visual input and varies the output length, which\nseparates the prefill cost (paid once) from the decode cost (paid per token).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .data import Sample\nfrom .data.frames import FrameStore\nfrom .tokens import decode_floor_seconds, gpu_spec, prefill_flops, shape_for, vit_patches\nfrom .utils import capture_env, gpu_name, write_jsonl\n\nlog = logging.getLogger(__name__)\n\n\ndef _run_setting(backend, store, sample, n, res, out_tokens, warmup, repeats, extra) -> list[dict]:\n    fb = store.load(sample, n, res, with_stats=False)\n    for _ in range(warmup):\n        backend.profile_one(sample, fb, out_tokens=out_tokens)\n    rows = []\n    for r in range(repeats):\n        fb = store.load(sample, n, res, with_stats=False)  # re-time the load as well\n        t = backend.profile_one(sample, fb, out_tokens=out_tokens)\n        rows.append({\n            \"frames\": n, \"res\": res, \"repeat\": r, \"frame_hw\": [fb.height, fb.width],\n            \"load_ms\": fb.load_s * 1e3, **asdict(t), **extra,\n        })\n    return rows\n\n\ndef run_profile(\n    backend,\n    model_id: str,\n    samples: list[Sample],\n    frames: list[int],\n    resolutions: list[int],\n    out_tokens: int,\n    decode_sweep: list[int],\n    warmup: int,\n    repeats: int,\n    out_path: Path,\n    cache_dir: Path | None = None,\n) -> list[dict]:\n    store = FrameStore(cache_dir, frame_counts=frames, store_res=max(resolutions))\n    sample = samples[0]\n    env = capture_env()\n    rows: list[dict] = []\n    for res in resolutions:\n        for n in frames:\n            try:\n                rr = _run_setting(backend, store, sample, n, res, out_tokens, warmup, repeats,\n                                  {\"sweep\": \"visual\"})\n            except RuntimeError as e:  # most likely CUDA OOM at the largest settings\n                log.warning(\"frames=%d res=%d failed: %s\", n, res, str(e).splitlines()[0])\n                _free_cuda()\n                rows.append({\"frames\": n, \"res\": res, \"sweep\": \"visual\", \"error\": str(e)[:200]})\n                continue\n            rows.extend(rr)\n            med = np.median([x[\"gpu_total_ms\"] for x in rr])\n            log.info(\"frames=%2d res=%3d tokens=%5d  total %.0f ms (vision %.0f, prefill %.0f, \"\n                     \"decode %.0f)\", n, res, rr[0][\"visual_tokens\"], med,\n                     np.median([x[\"vision_ms\"] for x in rr]),\n                     np.median([x[\"prefill_ms\"] for x in rr]),\n                     np.median([x[\"decode_ms\"] for x in rr]))\n    # Decode sweep at a mid-sized input.\n    n_mid, r_mid = sorted(frames)[len(frames) // 2], sorted(resolutions)[len(resolutions) // 2]\n    for ot in decode_sweep:\n        rows.extend(_run_setting(backend, store, sample, n_mid, r_mid, ot, warmup, repeats,\n                                 {\"sweep\": \"decode\"}))\n    for r in rows:\n        r[\"model\"] = model_id\n        r[\"gpu\"] = gpu_name()\n    write_jsonl(out_path, rows)\n    (out_path.with_suffix(\".env.json\")).write_text(__import__(\"json\").dumps(env, indent=1))\n    return rows\n\n\ndef _free_cuda():\n    try:\n        import gc\n\n        import torch\n\n        gc.collect()\n        torch.cuda.empty_cache()\n    except ImportError:\n        pass\n\n\ndef summarize(rows: list[dict], model_id: str, peak_tflops: float | None = None,\n              mem_bw: float | None = None) -> list[dict]:\n    \"\"\"Median per setting, plus achieved TFLOP/s against the GPU's spec-sheet peak.\"\"\"\n    import pandas as pd\n\n    df = pd.DataFrame([r for r in rows if \"error\" not in r])\n    if df.empty:\n        return []\n    spec = gpu_spec(df[\"gpu\"].iloc[0]) or {}\n    peak = peak_tflops or spec.get(\"tflops\")\n    bw = mem_bw or spec.get(\"mem_bw\")\n    shape = shape_for(model_id)\n    out = []\n    for (sweep, n, res, ot), g in df.groupby([\"sweep\", \"frames\", \"res\", \"out_tokens\"]):\n        h, w = g[\"frame_hw\"].iloc[0]\n        rec = {\n            \"sweep\": sweep, \"frames\": int(n), \"res\": int(res), \"out_tokens\": int(ot),\n            \"visual_tokens\": int(g[\"visual_tokens\"].iloc[0]),\n            \"prompt_tokens\": int(g[\"prompt_tokens\"].iloc[0]),\n        }\n        for col in [\"load_ms\", \"preprocess_ms\", \"h2d_ms\", \"vision_ms\", \"prefill_ms\",\n                    \"ttft_gpu_ms\", \"decode_ms\", \"tpot_ms\", \"gpu_total_ms\", \"peak_mem_gb\"]:\n            rec[col] = float(g[col].median())\n        rec[\"gpu_total_p95_ms\"] = float(g[\"gpu_total_ms\"].quantile(0.95))\n        rec[\"e2e_ms\"] = rec[\"load_ms\"] + rec[\"preprocess_ms\"] + rec[\"gpu_total_ms\"]\n        if shape:\n            fl = prefill_flops(shape, rec[\"prompt_tokens\"], vit_patches(h, w, int(n)))\n            rec[\"vision_tflops_achieved\"] = fl[\"vit\"] / (rec[\"vision_ms\"] / 1e3) / 1e12\n            rec[\"prefill_tflops_achieved\"] = fl[\"llm\"] / (rec[\"prefill_ms\"] / 1e3) / 1e12\n            if peak:\n                rec[\"vision_mfu\"] = rec[\"vision_tflops_achieved\"] / peak\n                rec[\"prefill_mfu\"] = rec[\"prefill_tflops_achieved\"] / peak\n            if bw:\n                rec[\"tpot_floor_ms\"] = decode_floor_seconds(shape, bw) * 1e3\n        out.append(rec)\n    return out\n",
"src/foveate/prompts.py": "\"\"\"Prompt construction and answer scoring shared by every backend.\"\"\"\n\nfrom __future__ import annotations\n\nimport math\nimport re\n\nfrom .data import LETTERS, Sample\n\nMC_INSTRUCTION = \"Answer with the option's letter from the given choices directly.\"\nOPEN_INSTRUCTION = \"Answer the question in one short sentence.\"\n\n\ndef question_text(s: Sample) -> str:\n    if s.is_mc:\n        opts = \"\\n\".join(f\"({LETTERS[i]}) {o}\" for i, o in enumerate(s.options or []))\n        return f\"Question: {s.question}\\nOptions:\\n{opts}\\n{MC_INSTRUCTION}\"\n    return f\"{s.question}\\n{OPEN_INSTRUCTION}\"\n\n\ndef chat_messages(s: Sample, video_item: dict | None = None) -> list[dict]:\n    \"\"\"HF-style chat messages with one video placeholder followed by the question.\"\"\"\n    content = [video_item or {\"type\": \"video\"}, {\"type\": \"text\", \"text\": question_text(s)}]\n    return [{\"role\": \"user\", \"content\": content}]\n\n\ndef option_letters(s: Sample) -> list[str]:\n    return list(LETTERS[: len(s.options or [])])\n\n\ndef softmax_over_options(letter_logprobs: dict[str, float], letters: list[str]) -> dict[str, float]:\n    \"\"\"Renormalise next-token log-probs over the valid option letters only.\n\n    This is \"logit-based\" multiple-choice scoring: the prediction is the most\n    probable valid letter, and the renormalised probability of that letter is\n    the confidence the cascade thresholds. A letter missing from the top-k list\n    gets a floor one nat below the smallest returned value.\n    \"\"\"\n    if letter_logprobs:\n        floor = min(letter_logprobs.values()) - 1.0\n    else:\n        floor = 0.0\n    lps = [letter_logprobs.get(L, floor) for L in letters]\n    m = max(lps)\n    z = sum(math.exp(lp - m) for lp in lps)\n    return {L: math.exp(lp - m) / z for L, lp in zip(letters, lps)}\n\n\ndef mc_summary(probs: dict[str, float]) -> dict:\n    \"\"\"Prediction plus three confidence signals: max prob, top-2 margin, entropy.\"\"\"\n    ranked = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)\n    p1 = ranked[0][1]\n    p2 = ranked[1][1] if len(ranked) > 1 else 0.0\n    ent = -sum(p * math.log(p) for p in probs.values() if p > 0)\n    return {\"pred\": ranked[0][0], \"conf\": p1, \"margin\": p1 - p2, \"entropy\": ent}\n\n\n_LETTER_RE = re.compile(r\"^\\s*\\(?([A-H])\\b\")\n\n\ndef parse_letter(text: str, letters: list[str]) -> str | None:\n    \"\"\"Fallback parser for generated text like 'B', '(B)', 'B. a car'.\"\"\"\n    m = _LETTER_RE.match(text or \"\")\n    if m and m.group(1) in letters:\n        return m.group(1)\n    return None\n\n\ndef clean_token(tok: str) -> str:\n    \"\"\"Strip tokenizer artefacts (leading space markers) from a decoded token.\"\"\"\n    return tok.replace(\"Ġ\", \"\").replace(\"▁\", \"\").strip().lstrip(\"(\")\n",
"src/foveate/report.py": "\"\"\"Figures and a Markdown summary from whatever results exist so far.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nfrom pathlib import Path\n\nimport numpy as np\n\nlog = logging.getLogger(__name__)\n\n# Categorical slots in fixed order (validated palette), plus recessive greys.\nC = [\"#2a78d6\", \"#eb6834\", \"#1baf7a\", \"#eda100\", \"#e87ba4\", \"#008300\", \"#6250d6\", \"#e34948\"]\nINK, INK2, GRID = \"#0b0b0b\", \"#52514e\", \"#e6e5e0\"\n\n\ndef _style(ax, title: str, xlabel: str, ylabel: str):\n    ax.set_title(title, loc=\"left\", fontsize=11, color=INK)\n    ax.set_xlabel(xlabel, color=INK2)\n    ax.set_ylabel(ylabel, color=INK2)\n    ax.grid(True, color=GRID, linewidth=0.8)\n    ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"):\n        ax.spines[s].set_visible(False)\n    for s in (\"left\", \"bottom\"):\n        ax.spines[s].set_color(\"#bdbcb6\")\n    ax.tick_params(colors=INK2)\n\n\ndef _plt():\n    import matplotlib\n\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n\n    return plt\n\n\n# ---------------------------------------------------------------------------\n\n\ndef plot_profile(summary: list[dict], out: Path) -> list[Path]:\n    plt = _plt()\n    vis = [r for r in summary if r[\"sweep\"] == \"visual\"]\n    paths = []\n    if vis:\n        vis = sorted(vis, key=lambda r: (r[\"res\"], r[\"frames\"]))\n        labels = [f\"{r['frames']}f @ {r['res']}\" for r in vis]\n        parts = [(\"load\", \"load_ms\"), (\"preprocess\", \"preprocess_ms\"), (\"vision encoder\", \"vision_ms\"),\n                 (\"LLM prefill\", \"prefill_ms\"), (\"decode\", \"decode_ms\")]\n        fig, ax = plt.subplots(figsize=(8, 0.32 * len(vis) + 1.6))\n        left = np.zeros(len(vis))\n        for k, (name, col) in enumerate(parts):\n            vals = np.array([r.get(col, 0) or 0 for r in vis])\n            ax.barh(labels, vals, left=left, color=C[k], label=name, height=0.7,\n                    edgecolor=\"white\", linewidth=1)\n            left += vals\n        _style(ax, \"Where batch-1 latency goes (median ms)\", \"milliseconds\", \"\")\n        ax.legend(ncol=5, fontsize=8, frameon=False, loc=\"lower center\", bbox_to_anchor=(0.5, 1.02))\n        fig.tight_layout()\n        p = out / \"profile_breakdown.png\"\n        fig.savefig(p, dpi=150)\n        plt.close(fig)\n        paths.append(p)\n\n        fig, ax = plt.subplots(figsize=(6.5, 4.2))\n        resolutions = sorted({r[\"res\"] for r in vis})\n        markers = [\"o\", \"s\", \"^\", \"D\", \"v\"]\n        for k, res in enumerate(resolutions):\n            pts = [r for r in vis if r[\"res\"] == res]\n            ax.plot([r[\"visual_tokens\"] for r in pts], [r[\"ttft_gpu_ms\"] for r in pts],\n                    marker=markers[k % 5], color=C[k % 8], linewidth=1.5, markersize=6,\n                    label=f\"res {res}\")\n        x = np.array([r[\"visual_tokens\"] for r in vis], float)\n        y = np.array([r[\"ttft_gpu_ms\"] for r in vis], float)\n        if len(x) > 2:\n            b, a = np.polyfit(x, y, 1)\n            r2 = 1 - ((y - (a + b * x)) ** 2).sum() / ((y - y.mean()) ** 2).sum()\n            xs = np.linspace(0, x.max(), 50)\n            ax.plot(xs, a + b * xs, color=INK2, linestyle=\"--\", linewidth=1,\n                    label=f\"fit: {b:.3f} ms/token, R²={r2:.3f}\")\n        _style(ax, \"Time to first token grows with visual tokens\", \"visual tokens\", \"GPU TTFT (ms)\")\n        ax.legend(frameon=False, fontsize=8)\n        fig.tight_layout()\n        p = out / \"ttft_vs_tokens.png\"\n        fig.savefig(p, dpi=150)\n        plt.close(fig)\n        paths.append(p)\n    dec = sorted([r for r in summary if r[\"sweep\"] == \"decode\"], key=lambda r: r[\"out_tokens\"])\n    if dec:\n        fig, ax = plt.subplots(figsize=(6.5, 4))\n        ot = [r[\"out_tokens\"] for r in dec]\n        ax.plot(ot, [r[\"ttft_gpu_ms\"] for r in dec], marker=\"o\", color=C[0], linewidth=1.5,\n                label=\"TTFT (prefill, paid once)\")\n        ax.plot(ot, [r[\"decode_ms\"] for r in dec], marker=\"s\", color=C[1], linewidth=1.5,\n                label=\"decode (paid per token)\")\n        _style(ax, \"Prefill vs decode as output length grows\", \"output tokens\", \"ms\")\n        ax.legend(frameon=False, fontsize=8)\n        fig.tight_layout()\n        p = out / \"prefill_vs_decode.png\"\n        fig.savefig(p, dpi=150)\n        plt.close(fig)\n        paths.append(p)\n    return paths\n\n\ndef plot_frontier(rep: dict, out: Path, tag: str) -> list[Path]:\n    plt = _plt()\n    paths = []\n    fig, ax = plt.subplots(figsize=(7, 4.6))\n    curve = rep[\"curve\"]\n    ax.plot([c[\"mean_cost\"] for c in curve], [c[\"accuracy\"] for c in curve], color=C[0],\n            linewidth=2, label=\"cascade (all thresholds, all questions)\", zorder=2)\n    for p in rep[\"policies\"]:\n        name = p[\"policy\"]\n        if name.startswith(\"static\"):\n            ax.scatter(p[\"mean_cost\"], p[\"accuracy\"], s=40, color=\"#8a8984\", zorder=3,\n                       edgecolor=\"white\", linewidth=1.5)\n            ax.annotate(name.replace(\"static \", \"\"), (p[\"mean_cost\"], p[\"accuracy\"]),\n                        textcoords=\"offset points\", xytext=(5, -10), fontsize=7, color=INK2)\n    styles = {\"cascade (\": (C[0], \"o\"), \"rules+cascade\": (C[2], \"D\"), \"rules\": (C[1], \"s\"),\n              \"oracle\": (\"white\", \"*\")}\n    for p in rep[\"policies\"]:\n        for key, (col, m) in styles.items():\n            if p[\"policy\"].startswith(key):\n                ax.scatter(p[\"mean_cost\"], p[\"accuracy\"], s=110 if m == \"*\" else 70, color=col,\n                           marker=m, edgecolor=INK if m == \"*\" else \"white\", linewidth=1.2,\n                           zorder=4, label=f\"{p['policy']} (held-out half)\")\n                break\n    _style(ax, f\"Accuracy vs cost: {tag}\", f\"mean cost per question ({rep['cost_metric']})\", \"accuracy\")\n    ax.legend(frameon=False, fontsize=7, loc=\"lower right\")\n    fig.tight_layout()\n    p = out / f\"frontier_{tag}.png\"\n    fig.savefig(p, dpi=150)\n    plt.close(fig)\n    paths.append(p)\n\n    pt = rep.get(\"per_task\") or []\n    if len(pt) > 1:\n        cfgs = list(rep[\"costs\"])\n        mat = np.array([[r[c] for c in cfgs] for r in pt])\n        fig, ax = plt.subplots(figsize=(1.0 + 0.75 * len(cfgs), 0.3 * len(pt) + 1.8))\n        im = ax.imshow(mat, cmap=\"Blues\", vmin=0, vmax=1, aspect=\"auto\")\n        ax.set_xticks(range(len(cfgs)), cfgs, rotation=45, ha=\"right\", fontsize=7)\n        ax.set_yticks(range(len(pt)), [r[\"task\"] for r in pt], fontsize=7)\n        for i in range(mat.shape[0]):\n            for j in range(mat.shape[1]):\n                ax.text(j, i, f\"{mat[i, j]:.2f}\", ha=\"center\", va=\"center\", fontsize=6,\n                        color=\"white\" if mat[i, j] > 0.6 else INK)\n        ax.set_title(\"Per-task accuracy by config: which tasks need frames, which need pixels\",\n                     loc=\"left\", fontsize=10)\n        fig.colorbar(im, ax=ax, fraction=0.03)\n        fig.tight_layout()\n        p = out / f\"per_task_{tag}.png\"\n        fig.savefig(p, dpi=150)\n        plt.close(fig)\n        paths.append(p)\n    return paths\n\n\ndef plot_shift(rep: dict, out: Path, tag: str) -> list[Path]:\n    plt = _plt()\n    paths = []\n    pc = rep[\"per_corruption\"]\n    fig, ax = plt.subplots(figsize=(6.5, 4))\n    x = np.arange(len(pc))\n    w = 0.38\n    ax.bar(x - w / 2 - 0.01, [r[\"clean_escalation\"] for r in pc], w, color=C[0], label=\"clean\")\n    ax.bar(x + w / 2 + 0.01, [r[\"shifted_escalation\"] for r in pc], w, color=C[1], label=\"shifted\")\n    ax.set_xticks(x, [r[\"corruption\"] for r in pc])\n    _style(ax, \"Cascade escalation rate, clean vs shifted inputs\", \"\", \"share of questions escalated\")\n    ax.legend(frameon=False, fontsize=8)\n    fig.tight_layout()\n    p = out / f\"shift_escalation_{tag}.png\"\n    fig.savefig(p, dpi=150)\n    plt.close(fig)\n    paths.append(p)\n    for st in rep[\"streams\"]:\n        tr = st[\"trace\"]\n        n = len(tr[\"z\"])\n        k = max(5, st[\"window\"] // 2)\n        esc = np.convolve(tr[\"escalated\"], np.ones(k) / k, mode=\"same\")\n        fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)\n        axes[0].plot(range(n), tr[\"z\"], color=C[0], linewidth=1.5, label=\"image-statistics drift (max |z|)\")\n        axes[0].axhline(st[\"z_alarm\"], color=INK2, linestyle=\"--\", linewidth=1, label=\"alarm threshold\")\n        _style(axes[0], f\"Stream switches to {st['corruption']} at the dashed line\", \"\", \"drift z\")\n        axes[1].plot(range(n), esc, color=C[1], linewidth=1.5, label=\"escalation rate (moving avg)\")\n        _style(axes[1], \"\", \"request index\", \"escalation rate\")\n        for a in axes:\n            a.axvline(n // 2, color=\"#8a8984\", linestyle=\":\", linewidth=1)\n            a.legend(frameon=False, fontsize=8, loc=\"upper left\")\n        fig.tight_layout()\n        p = out / f\"shift_stream_{tag}_{st['corruption'].replace(':', '')}.png\"\n        fig.savefig(p, dpi=150)\n        plt.close(fig)\n        paths.append(p)\n    return paths\n\n\n# ---------------------------------------------------------------------------\n\n\ndef _fmt(v, nd=3):\n    if v is None or (isinstance(v, float) and np.isnan(v)):\n        return \"–\"\n    if isinstance(v, float):\n        return f\"{v:.{nd}f}\"\n    return str(v)\n\n\ndef md_table(rows: list[dict], cols: list[str], headers: list[str] | None = None) -> str:\n    headers = headers or cols\n    lines = [\"| \" + \" | \".join(headers) + \" |\", \"|\" + \"---|\" * len(cols)]\n    for r in rows:\n        lines.append(\"| \" + \" | \".join(_fmt(r.get(c)) for c in cols) + \" |\")\n    return \"\\n\".join(lines)\n\n\ndef build_report(results: Path) -> Path:\n    figs = results / \"figures\"\n    figs.mkdir(parents=True, exist_ok=True)\n    md = [\"# Foveate results\", \"\", \"Generated by `foveate report`. Every number below comes from \"\n          \"result files in this folder; nothing is typed in by hand.\", \"\"]\n\n    for p in sorted((results / \"profile\").glob(\"*.summary.json\")):\n        summ = json.loads(p.read_text())\n        imgs = plot_profile(summ, figs)\n        md += [f\"## Stage 1: profile ({p.stem.replace('.summary', '')})\", \"\"]\n        md += [md_table([r for r in summ if r[\"sweep\"] == \"visual\"],\n                        [\"frames\", \"res\", \"visual_tokens\", \"load_ms\", \"preprocess_ms\", \"vision_ms\",\n                         \"prefill_ms\", \"decode_ms\", \"tpot_ms\", \"e2e_ms\", \"peak_mem_gb\",\n                         \"prefill_mfu\"]), \"\"]\n        md += [f\"![]({i.relative_to(results)})\" for i in imgs] + [\"\"]\n\n    for run in sorted(d for d in results.iterdir() if d.is_dir() and (d / \"replay\").exists()):\n        rp = run / \"replay\" / \"summary.json\"\n        if rp.exists():\n            rep = json.loads(rp.read_text())\n            imgs = plot_frontier(rep, figs, run.name)\n            md += [f\"## Stages 2-3: policies on {run.name}\", \"\",\n                   f\"Cost metric: {rep['cost_metric']}. Reference config: `{rep['reference']}`. \"\n                   f\"Thresholds and rules fit on {rep['n_calib']} calibration questions, \"\n                   f\"reported on {rep['n_test']} held-out questions. Ladder: \"\n                   + \" → \".join(f\"`{c}`\" for c in rep[\"ladder\"]) + \".\", \"\"]\n            md += [md_table(rep[\"policies\"], [\"policy\", \"accuracy\", \"acc_lo\", \"acc_hi\", \"mean_cost\",\n                                              \"cost_saving\", \"acc_diff\", \"acc_diff_lo\", \"acc_diff_hi\",\n                                              \"escalation_rate\"]), \"\"]\n            md += [\"Confidence quality per config (AUROC: does confidence separate right from \"\n                   \"wrong answers? ECE: is it calibrated?):\", \"\"]\n            md += [md_table(rep[\"confidence\"], [\"config\", \"accuracy\", \"auroc\", \"ece\", \"cost\"]), \"\"]\n            if rep.get(\"ablations\"):\n                md += [\"Ablations (cascade with one rung removed):\", \"\"]\n                md += [md_table(rep[\"ablations\"], [\"policy\", \"accuracy\", \"mean_cost\", \"acc_diff\",\n                                                   \"cost_saving\"]), \"\"]\n            md += [f\"Learned rules: `{json.dumps(rep['rules'])}`\", \"\"]\n            md += [f\"![]({i.relative_to(results)})\" for i in imgs] + [\"\"]\n        sp = run / \"replay\" / \"shift.json\"\n        if sp.exists():\n            rep = json.loads(sp.read_text())\n            imgs = plot_shift(rep, figs, run.name)\n            md += [f\"## Stage 4: distribution shift on {run.name}\", \"\"]\n            md += [md_table(rep[\"per_corruption\"],\n                            [\"corruption\", \"clean_cascade_acc\", \"shifted_cascade_acc\",\n                             \"clean_escalation\", \"shifted_escalation\", \"clean_cost\", \"shifted_cost\",\n                             \"clean_cheap_acc\", \"shifted_cheap_acc\", \"clean_cheap_auroc\",\n                             \"shifted_cheap_auroc\"]), \"\"]\n            for st in rep[\"streams\"]:\n                md += [f\"Stream with {st['corruption']}: alarm delay {st['detection_delay']} \"\n                       f\"requests, {st['false_alarms_on_clean']} false alarms on clean. On the \"\n                       f\"shifted half the drift-aware cascade cost \"\n                       f\"{_fmt(st['drift_aware']['shifted']['mean_cost'])} vs \"\n                       f\"{_fmt(st['plain']['shifted']['mean_cost'])} (plain), accuracy \"\n                       f\"{_fmt(st['drift_aware']['shifted']['accuracy'])} vs \"\n                       f\"{_fmt(st['plain']['shifted']['accuracy'])}.\", \"\"]\n            md += [f\"![]({i.relative_to(results)})\" for i in imgs] + [\"\"]\n\n    serving = sorted((results / \"serving\").glob(\"*.summary.json\")) if (results / \"serving\").exists() else []\n    if serving:\n        rows = [json.loads(p.read_text()) | {\"run\": p.stem.replace(\".summary\", \"\")} for p in serving]\n        md += [\"## Stage 5: serving under load\", \"\"]\n        md += [md_table(rows, [\"run\", \"policy\", \"requests\", \"throughput_rps\", \"p50_ms\", \"p95_ms\",\n                               \"p99_ms\", \"ttft_p50_ms\", \"accuracy\", \"escalation_rate\"]), \"\"]\n    out = results / \"REPORT.md\"\n    out.write_text(\"\\n\".join(md))\n    log.info(\"wrote %s\", out)\n    return out\n",
"src/foveate/serve/__init__.py": "",
"src/foveate/serve/gateway.py": "\"\"\"Stage 5: a small policy-aware gateway in front of vLLM's OpenAI-compatible servers.\n\n    client ──HTTP──> gateway (this file) ──> vllm serve 3B  (port 8001)\n                     │ picks frames/res        └> vllm serve 7B-AWQ (port 8002)\n                     │ runs the cascade\n                     └ logs telemetry (JSONL)\n\nThe gateway owns the policy: it samples and resizes frames, sends them as a\n`data:video/jpeg` URL (a comma-separated list of base64 JPEG frames, which\nvLLM decodes without re-sampling), reads the option-letter log-probs, and\nescalates to the next rung when confidence is below tau. vLLM owns batching,\nKV-cache paging, prefix caching and the multimodal encoder cache.\n\nRun:  foveate serve --dataset mvbench --subset pilot --backends 3b=http://localhost:8001 \\\n          7b=http://localhost:8002 --policy-file results/mvbench_pilot/replay/policy.json\n\"\"\"\n\nfrom __future__ import annotations\n\nimport asyncio\nimport base64\nimport io\nimport json\nimport logging\nimport math\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nfrom ..config import VisualConfig, cache_root\nfrom ..data import Sample\nfrom ..data.frames import FrameBatch, FrameStore\nfrom ..prompts import (\n    clean_token,\n    mc_summary,\n    option_letters,\n    parse_letter,\n    question_text,\n    softmax_over_options,\n)\n\nlog = logging.getLogger(__name__)\n\n\ndef video_data_url(fb: FrameBatch, quality: int = 90) -> str:\n    parts = []\n    for im in fb.frames:\n        buf = io.BytesIO()\n        im.save(buf, format=\"JPEG\", quality=quality)\n        parts.append(base64.b64encode(buf.getvalue()).decode())\n    return \"data:video/jpeg;base64,\" + \",\".join(parts)\n\n\ndef sampled_fps(fb: FrameBatch) -> float:\n    m = fb.meta\n    dur = m.get(\"duration\") or 0\n    return len(fb.frames) / dur if dur else 1.0\n\n\nclass Gateway:\n    def __init__(\n        self,\n        samples: dict[str, Sample],\n        backends: dict[str, str],\n        model_ids: dict[str, str],\n        ladder: list[VisualConfig],\n        tau: float,\n        telemetry: Path,\n        frame_workers: int = 4,\n        store_res: int = 672,\n    ):\n        import httpx\n\n        self.samples = samples\n        self.backends = backends\n        self.model_ids = model_ids\n        self.ladder = ladder\n        self.tau = tau\n        self.telemetry = telemetry\n        telemetry.parent.mkdir(parents=True, exist_ok=True)\n        self.store = FrameStore(cache_root() / f\"r{store_res}\",\n                                frame_counts={c.frames for c in ladder}, store_res=store_res)\n        self.pool = ThreadPoolExecutor(max_workers=frame_workers)\n        self.client = httpx.AsyncClient(timeout=httpx.Timeout(600.0),\n                                        limits=httpx.Limits(max_connections=512))\n        self._tlock = asyncio.Lock()\n\n    async def _call(self, s: Sample, cfg: VisualConfig) -> dict:\n        loop = asyncio.get_running_loop()\n        t0 = time.perf_counter()\n        fb = await loop.run_in_executor(self.pool, self.store.load, s, cfg.frames, cfg.res, None, False)\n        url = await loop.run_in_executor(self.pool, video_data_url, fb)\n        t_prep = time.perf_counter()\n        body = {\n            \"model\": self.model_ids[cfg.model],\n            \"messages\": [{\"role\": \"user\", \"content\": [\n                {\"type\": \"video_url\", \"video_url\": {\"url\": url}},\n                {\"type\": \"text\", \"text\": question_text(s)},\n            ]}],\n            \"temperature\": 0.0,\n            \"max_tokens\": 1 if s.is_mc else 48,\n            \"logprobs\": True,\n            \"top_logprobs\": 20,\n            \"stream\": True,\n            \"stream_options\": {\"include_usage\": True},\n            \"media_io_kwargs\": {\"video\": {\"fps\": sampled_fps(fb)}},\n        }\n        ttft = None\n        text, lp_first, n_tok, usage = \"\", None, 0, {}\n        async with self.client.stream(\"POST\", f\"{self.backends[cfg.model]}/v1/chat/completions\",\n                                      json=body) as r:\n            r.raise_for_status()\n            async for line in r.aiter_lines():\n                if not line.startswith(\"data: \") or line.endswith(\"[DONE]\"):\n                    continue\n                chunk = json.loads(line[6:])\n                if chunk.get(\"usage\"):\n                    usage = chunk[\"usage\"]\n                for ch in chunk.get(\"choices\", []):\n                    delta = ch.get(\"delta\", {}).get(\"content\")\n                    if delta:\n                        if ttft is None:\n                            ttft = time.perf_counter() - t_prep\n                        text += delta\n                    lps = (ch.get(\"logprobs\") or {}).get(\"content\") or []\n                    for item in lps:\n                        n_tok += 1\n                        if lp_first is None:\n                            lp_first = item\n        t_end = time.perf_counter()\n        out = {\"config\": cfg.name, \"prep_ms\": (t_prep - t0) * 1e3,\n               \"ttft_ms\": (ttft or (t_end - t_prep)) * 1e3, \"backend_ms\": (t_end - t_prep) * 1e3,\n               \"prompt_tokens\": usage.get(\"prompt_tokens\"), \"text\": text}\n        if s.is_mc:\n            letters = option_letters(s)\n            letter_lp: dict[str, float] = {}\n            for cand in (lp_first or {}).get(\"top_logprobs\", []):\n                L = clean_token(cand.get(\"token\", \"\"))\n                if L in letters:\n                    letter_lp[L] = max(letter_lp.get(L, -math.inf), cand[\"logprob\"])\n            if letter_lp:\n                out.update(mc_summary(softmax_over_options(letter_lp, letters)))\n            else:\n                out.update({\"pred\": parse_letter(text, letters), \"conf\": 0.0})\n        else:\n            out.update({\"pred\": text.strip(), \"conf\": 1.0})\n        return out\n\n    async def ask(self, qid: str, policy: str = \"cascade\", tau: float | None = None) -> dict:\n        s = self.samples[qid]\n        t0 = time.perf_counter()\n        if policy == \"cascade\":\n            rungs, thr = self.ladder, self.tau if tau is None else tau\n        else:  # \"static:3b:16x448\"\n            rungs, thr = [VisualConfig.parse(policy.split(\":\", 1)[1])], -1.0\n        calls = []\n        for cfg in rungs:\n            res = await self._call(s, cfg)\n            calls.append(res)\n            if res.get(\"conf\", 0.0) >= thr:\n                break\n        final = calls[-1]\n        rec = {\n            \"qid\": qid, \"task\": s.task, \"policy\": policy, \"tau\": thr,\n            \"pred\": final.get(\"pred\"), \"answer\": s.answer,\n            \"correct\": (final.get(\"pred\") == s.answer) if s.is_mc else None,\n            \"conf\": final.get(\"conf\"), \"rungs\": [c[\"config\"] for c in calls],\n            \"ttft_ms\": sum(c[\"prep_ms\"] + c[\"backend_ms\"] for c in calls[:-1])\n            + calls[-1][\"prep_ms\"] + calls[-1][\"ttft_ms\"],\n            \"latency_ms\": (time.perf_counter() - t0) * 1e3, \"calls\": calls,\n        }\n        async with self._tlock:\n            with self.telemetry.open(\"a\") as f:\n                f.write(json.dumps(rec) + \"\\n\")\n        return rec\n\n\ndef build_app(gw: Gateway):\n    from fastapi import Body, FastAPI, HTTPException\n\n    app = FastAPI(title=\"foveate gateway\")\n\n    @app.post(\"/ask\")\n    async def ask(\n        qid: str = Body(...),\n        policy: str = Body(\"cascade\"),\n        tau: float | None = Body(None),\n    ):\n        \"\"\"Answer one question; policy is \"cascade\" or \"static:MODEL:FxR\".\"\"\"\n        if qid not in gw.samples:\n            raise HTTPException(404, f\"unknown qid {qid}\")\n        return await gw.ask(qid, policy, tau)\n\n    @app.get(\"/health\")\n    async def health():\n        return {\"ok\": True, \"ladder\": [c.name for c in gw.ladder], \"tau\": gw.tau,\n                \"questions\": len(gw.samples)}\n\n    return app\n",
"src/foveate/serve/loadgen.py": "\"\"\"Open-loop load generator: Poisson arrivals against the gateway.\n\nOpen loop (arrivals do not wait for responses) is what real traffic looks like\nand is the only way to see queueing: a closed-loop client slows down when the\nserver does, hiding exactly the tail latency we want to measure.\n\nWorkload knobs:\n  rate        mean arrivals per second (Poisson, seeded)\n  repeat      fraction of requests that reuse a recently seen clip with a\n              different question (exercises vLLM's prefix + encoder caches)\n\"\"\"\n\nfrom __future__ import annotations\n\nimport asyncio\nimport json\nimport random\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom ..data import Sample\n\n\ndef make_trace(samples: list[Sample], n: int, rate: float, repeat: float, seed: int = 0) -> list[dict]:\n    rng = random.Random(seed)\n    by_clip: dict[str, list[Sample]] = defaultdict(list)\n    for s in samples:\n        by_clip[s.clip_id or s.qid].append(s)\n    recent: list[str] = []\n    t = 0.0\n    trace = []\n    for _ in range(n):\n        t += rng.expovariate(rate)\n        reuse = [c for c in recent if len(by_clip[c]) > 1]\n        if reuse and rng.random() < repeat:\n            s = rng.choice(by_clip[rng.choice(reuse)])\n        else:\n            s = rng.choice(samples)\n        recent = (recent + [s.clip_id or s.qid])[-32:]\n        trace.append({\"t\": t, \"qid\": s.qid})\n    return trace\n\n\nasync def run_trace(url: str, trace: list[dict], policy: str, out: Path, timeout: float = 600.0) -> dict:\n    import httpx\n\n    results = []\n    async with httpx.AsyncClient(timeout=timeout, limits=httpx.Limits(max_connections=1024)) as client:\n        t0 = time.perf_counter()\n\n        async def one(item):\n            delay = item[\"t\"] - (time.perf_counter() - t0)\n            if delay > 0:\n                await asyncio.sleep(delay)\n            sent = time.perf_counter()\n            try:\n                r = await client.post(f\"{url}/ask\", json={\"qid\": item[\"qid\"], \"policy\": policy})\n                r.raise_for_status()\n                rec = r.json()\n                rec[\"client_latency_ms\"] = (time.perf_counter() - sent) * 1e3\n                rec[\"ok\"] = True\n            except Exception as e:  # keep going; failures are part of the result\n                rec = {\"qid\": item[\"qid\"], \"ok\": False, \"error\": str(e)[:200],\n                       \"client_latency_ms\": (time.perf_counter() - sent) * 1e3}\n            rec[\"sent_s\"] = sent - t0\n            results.append(rec)\n\n        await asyncio.gather(*(one(it) for it in trace))\n        wall = time.perf_counter() - t0\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with out.open(\"w\") as f:\n        for r in results:\n            r.pop(\"calls\", None)\n            f.write(json.dumps(r) + \"\\n\")\n    return summarize(results, wall, policy)\n\n\ndef summarize(results: list[dict], wall: float, policy: str) -> dict:\n    ok = [r for r in results if r.get(\"ok\")]\n    lat = np.array([r[\"client_latency_ms\"] for r in ok]) if ok else np.array([np.nan])\n    ttft = np.array([r.get(\"ttft_ms\", np.nan) for r in ok]) if ok else np.array([np.nan])\n    corr = [r[\"correct\"] for r in ok if r.get(\"correct\") is not None]\n    esc = [len(r.get(\"rungs\", [])) > 1 for r in ok]\n    return {\n        \"policy\": policy, \"requests\": len(results), \"ok\": len(ok), \"wall_s\": wall,\n        \"throughput_rps\": len(ok) / wall if wall else float(\"nan\"),\n        \"p50_ms\": float(np.nanpercentile(lat, 50)), \"p95_ms\": float(np.nanpercentile(lat, 95)),\n        \"p99_ms\": float(np.nanpercentile(lat, 99)), \"mean_ms\": float(np.nanmean(lat)),\n        \"ttft_p50_ms\": float(np.nanpercentile(ttft, 50)), \"ttft_p95_ms\": float(np.nanpercentile(ttft, 95)),\n        \"accuracy\": float(np.mean(corr)) if corr else None,\n        \"escalation_rate\": float(np.mean(esc)) if esc else None,\n    }\n",
"src/foveate/stats.py": "\"\"\"Statistics used everywhere: bootstrap intervals, paired tests, calibration.\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef bootstrap_ci(x, n_boot: int = 2000, alpha: float = 0.05, seed: int = 0, stat=np.mean):\n    \"\"\"Percentile bootstrap interval for a statistic of one sample.\"\"\"\n    x = np.asarray(x, dtype=float)\n    if len(x) == 0:\n        return float(\"nan\"), float(\"nan\"), float(\"nan\")\n    rng = np.random.default_rng(seed)\n    idx = rng.integers(0, len(x), size=(n_boot, len(x)))\n    boots = stat(x[idx], axis=1)\n    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])\n    return float(stat(x)), float(lo), float(hi)\n\n\ndef paired_bootstrap_diff(a, b, n_boot: int = 2000, alpha: float = 0.05, seed: int = 0):\n    \"\"\"Mean(a - b) with a bootstrap interval, resampling *questions* (pairs).\n\n    Pairing matters: two configs answering the same questions are strongly\n    correlated, so the interval on their difference is much tighter than the\n    two separate intervals suggest.\n    \"\"\"\n    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)\n    return bootstrap_ci(d, n_boot=n_boot, alpha=alpha, seed=seed)\n\n\ndef auroc(scores, labels) -> float:\n    \"\"\"Probability a random correct answer has higher confidence than a random wrong one.\"\"\"\n    s = np.asarray(scores, dtype=float)\n    y = np.asarray(labels, dtype=bool)\n    pos, neg = s[y], s[~y]\n    if len(pos) == 0 or len(neg) == 0:\n        return float(\"nan\")\n    order = np.argsort(np.concatenate([pos, neg]), kind=\"mergesort\")\n    ranks = np.empty(len(order))\n    allv = np.concatenate([pos, neg])[order]\n    # average ranks for ties\n    i = 0\n    r = np.arange(1, len(allv) + 1, dtype=float)\n    while i < len(allv):\n        j = i\n        while j + 1 < len(allv) and allv[j + 1] == allv[i]:\n            j += 1\n        r[i:j + 1] = (i + j + 2) / 2\n        i = j + 1\n    ranks[order] = r\n    return float((ranks[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))\n\n\ndef ece(conf, correct, n_bins: int = 10) -> float:\n    \"\"\"Expected calibration error with equal-width bins.\"\"\"\n    c = np.asarray(conf, dtype=float)\n    y = np.asarray(correct, dtype=float)\n    bins = np.minimum((c * n_bins).astype(int), n_bins - 1)\n    total = 0.0\n    for b in range(n_bins):\n        m = bins == b\n        if m.any():\n            total += m.mean() * abs(y[m].mean() - c[m].mean())\n    return float(total)\n\n\ndef spearman(x, y) -> float:\n    x, y = np.asarray(x, float), np.asarray(y, float)\n    if len(x) < 3:\n        return float(\"nan\")\n    rx, ry = np.argsort(np.argsort(x)), np.argsort(np.argsort(y))\n    return float(np.corrcoef(rx, ry)[0, 1])\n\n\ndef linear_fit(x, y) -> dict:\n    \"\"\"Least squares y = a + b x, with R^2 (used for latency vs visual tokens).\"\"\"\n    x, y = np.asarray(x, float), np.asarray(y, float)\n    b, a = np.polyfit(x, y, 1)\n    pred = a + b * x\n    ss_res = ((y - pred) ** 2).sum()\n    ss_tot = ((y - y.mean()) ** 2).sum()\n    return {\"intercept\": float(a), \"slope\": float(b), \"r2\": float(1 - ss_res / ss_tot) if ss_tot else float(\"nan\")}\n",
"src/foveate/tokens.py": "\"\"\"Visual-token and FLOP arithmetic for Qwen2.5-VL style models.\n\nQwen2.5-VL cuts each frame into 14x14 patches, merges every 2x2 block of\npatches into one LLM token (so one token covers 28x28 pixels) and, for video,\nfuses every 2 consecutive frames into one temporal patch. Everything the\nprofiler measures should line up with these formulas; when it does not, that\ngap is a finding (see theory/01_anatomy_of_vlm_inference.md).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nPATCH = 14\nMERGE = 2\nFACTOR = PATCH * MERGE  # 28 px per token side\nTEMPORAL_PATCH = 2\n\n\ndef smart_resize(\n    height: int,\n    width: int,\n    factor: int = FACTOR,\n    min_pixels: int = 4 * FACTOR * FACTOR,\n    max_pixels: int = 16384 * FACTOR * FACTOR,\n) -> tuple[int, int]:\n    \"\"\"Same rounding rule as Qwen2-VL's image processor.\n\n    Returns (h, w), both multiples of `factor`, with h*w inside\n    [min_pixels, max_pixels] and the aspect ratio kept as close as possible.\n    \"\"\"\n    if height <= 0 or width <= 0:\n        raise ValueError(f\"bad size {height}x{width}\")\n    if max(height, width) / min(height, width) > 200:\n        raise ValueError(f\"aspect ratio too extreme: {height}x{width}\")\n    h_bar = max(factor, round(height / factor) * factor)\n    w_bar = max(factor, round(width / factor) * factor)\n    if h_bar * w_bar > max_pixels:\n        beta = math.sqrt((height * width) / max_pixels)\n        h_bar = max(factor, math.floor(height / beta / factor) * factor)\n        w_bar = max(factor, math.floor(width / beta / factor) * factor)\n    elif h_bar * w_bar < min_pixels:\n        beta = math.sqrt(min_pixels / (height * width))\n        h_bar = math.ceil(height * beta / factor) * factor\n        w_bar = math.ceil(width * beta / factor) * factor\n    return h_bar, w_bar\n\n\ndef target_size(height: int, width: int, res: int) -> tuple[int, int]:\n    \"\"\"Size a frame is resized to for a `res` setting.\n\n    `res` is a pixel budget: the frame keeps its aspect ratio and gets about\n    res*res pixels, rounded to multiples of 28. res=448 on a 16:9 frame gives\n    588x336 (= 21x12 tokens = 252 tokens per frame pair).\n    \"\"\"\n    return smart_resize(height, width, max_pixels=res * res, min_pixels=FACTOR * FACTOR)\n\n\ndef tokens_per_frame(height: int, width: int) -> int:\n    \"\"\"LLM tokens for one already-resized frame (or one frame pair in video mode).\"\"\"\n    if height % FACTOR or width % FACTOR:\n        raise ValueError(f\"{height}x{width} is not a multiple of {FACTOR}\")\n    return (height // FACTOR) * (width // FACTOR)\n\n\ndef visual_tokens(height: int, width: int, n_frames: int, mode: str = \"video\") -> int:\n    \"\"\"Visual tokens the LLM sees for n frames of a resized size.\n\n    mode=\"video\": frames are fused in pairs (odd counts are padded by\n    repeating the last frame, as the HF processor does).\n    mode=\"images\": every frame is its own image; a single image is internally\n    duplicated to fill the temporal patch, so it costs one frame pair.\n    \"\"\"\n    per = tokens_per_frame(height, width)\n    if mode == \"video\":\n        return math.ceil(n_frames / TEMPORAL_PATCH) * per\n    if mode == \"images\":\n        return n_frames * per\n    raise ValueError(mode)\n\n\ndef vit_patches(height: int, width: int, n_frames: int, mode: str = \"video\") -> int:\n    \"\"\"Patches the vision encoder processes (4x the LLM tokens, before merging).\"\"\"\n    return visual_tokens(height, width, n_frames, mode) * MERGE * MERGE\n\n\n@dataclass(frozen=True)\nclass ModelShape:\n    \"\"\"The few numbers needed for back-of-envelope cost models.\"\"\"\n\n    name: str\n    llm_params: float  # language model parameters (incl. embeddings is fine at this precision)\n    vit_params: float\n    layers: int\n    kv_heads: int\n    head_dim: int\n    weight_bytes: float  # bytes per weight as loaded (2 for fp16, ~0.55 for AWQ int4)\n\n\nMODEL_SHAPES = {\n    \"Qwen/Qwen2.5-VL-3B-Instruct\": ModelShape(\n        \"qwen2.5-vl-3b\", 3.09e9, 0.67e9, 36, 2, 128, 2.0\n    ),\n    \"Qwen/Qwen2.5-VL-7B-Instruct\": ModelShape(\n        \"qwen2.5-vl-7b\", 7.6e9, 0.67e9, 28, 4, 128, 2.0\n    ),\n    \"Qwen/Qwen2.5-VL-7B-Instruct-AWQ\": ModelShape(\n        \"qwen2.5-vl-7b-awq\", 7.6e9, 0.67e9, 28, 4, 128, 0.55\n    ),\n}\n\n\ndef shape_for(model_id: str) -> ModelShape | None:\n    return MODEL_SHAPES.get(model_id)\n\n\ndef kv_bytes_per_token(shape: ModelShape, dtype_bytes: int = 2) -> int:\n    \"\"\"K and V, for every layer, for every KV head (GQA keeps this small).\"\"\"\n    return 2 * shape.layers * shape.kv_heads * shape.head_dim * dtype_bytes\n\n\ndef prefill_flops(shape: ModelShape, llm_tokens: int, vit_patch_count: int) -> dict[str, float]:\n    \"\"\"~2 FLOPs per parameter per token for the matmuls (attention FLOPs ignored:\n    they are small at these lengths, and Qwen's ViT uses windowed attention).\"\"\"\n    llm = 2.0 * shape.llm_params * llm_tokens\n    vit = 2.0 * shape.vit_params * vit_patch_count\n    return {\"llm\": llm, \"vit\": vit, \"total\": llm + vit}\n\n\ndef decode_floor_seconds(shape: ModelShape, mem_bw_gbs: float) -> float:\n    \"\"\"Lower bound per decode step at batch 1: every weight is read once.\"\"\"\n    total_params = shape.llm_params  # the ViT does not run during decode\n    return total_params * shape.weight_bytes / (mem_bw_gbs * 1e9)\n\n\n# Dense fp16/bf16 tensor-core peak and memory bandwidth from vendor spec sheets.\n# They are ceilings, not expectations; override with --peak-tflops / --mem-bw.\nGPU_SPECS = {\n    \"T4\": {\"tflops\": 65.0, \"mem_bw\": 320.0},\n    \"L4\": {\"tflops\": 121.0, \"mem_bw\": 300.0},\n    \"A10G\": {\"tflops\": 70.0, \"mem_bw\": 600.0},\n    \"A10\": {\"tflops\": 125.0, \"mem_bw\": 600.0},\n    \"RTX 4090\": {\"tflops\": 165.0, \"mem_bw\": 1008.0},\n    \"A100\": {\"tflops\": 312.0, \"mem_bw\": 1555.0},\n    \"H100\": {\"tflops\": 989.0, \"mem_bw\": 3350.0},\n    \"P100\": {\"tflops\": 19.0, \"mem_bw\": 732.0},\n}\n\n\ndef gpu_spec(gpu_name: str | None) -> dict[str, float] | None:\n    if not gpu_name:\n        return None\n    up = gpu_name.upper()\n    # Longest key first so \"A10G\" wins over \"A10\".\n    for key in sorted(GPU_SPECS, key=len, reverse=True):\n        if key.upper() in up:\n            return GPU_SPECS[key]\n    return None\n",
"src/foveate/utils.py": "\"\"\"Small shared helpers: JSONL I/O, environment capture, logging.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport os\nimport platform\nimport subprocess\nimport sys\nimport time\nfrom importlib import metadata\nfrom pathlib import Path\nfrom typing import Any, Iterable, Iterator\n\nlog = logging.getLogger(\"foveate\")\n\n\ndef setup_logging(verbose: bool = False) -> None:\n    logging.basicConfig(\n        level=logging.DEBUG if verbose else logging.INFO,\n        format=\"%(asctime)s %(levelname)s %(name)s: %(message)s\",\n        datefmt=\"%H:%M:%S\",\n    )\n\n\ndef write_jsonl(path: str | Path, rows: Iterable[dict], append: bool = False) -> int:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    n = 0\n    with path.open(\"a\" if append else \"w\") as f:\n        for r in rows:\n            f.write(json.dumps(r, default=_json_default) + \"\\n\")\n            n += 1\n    return n\n\n\ndef read_jsonl(path: str | Path) -> Iterator[dict]:\n    with Path(path).open() as f:\n        for line in f:\n            line = line.strip()\n            if line:\n                yield json.loads(line)\n\n\ndef _json_default(o: Any):\n    try:\n        import numpy as np\n\n        if isinstance(o, np.integer):\n            return int(o)\n        if isinstance(o, np.floating):\n            return float(o)\n        if isinstance(o, np.ndarray):\n            return o.tolist()\n    except ImportError:\n        pass\n    if isinstance(o, Path):\n        return str(o)\n    raise TypeError(f\"not JSON serializable: {type(o)}\")\n\n\ndef _pkg_version(name: str) -> str | None:\n    try:\n        return metadata.version(name)\n    except metadata.PackageNotFoundError:\n        return None\n\n\ndef capture_env() -> dict[str, Any]:\n    \"\"\"Everything needed to reproduce a number: GPU, driver, library versions.\"\"\"\n    env: dict[str, Any] = {\n        \"time_utc\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n        \"python\": sys.version.split()[0],\n        \"platform\": platform.platform(),\n        \"kaggle\": bool(os.environ.get(\"KAGGLE_KERNEL_RUN_TYPE\")),\n        \"packages\": {\n            p: _pkg_version(p)\n            for p in [\"torch\", \"transformers\", \"vllm\", \"foveate\", \"av\", \"numpy\", \"accelerate\"]\n        },\n    }\n    try:\n        import torch\n\n        env[\"cuda_available\"] = torch.cuda.is_available()\n        if torch.cuda.is_available():\n            env[\"torch_cuda\"] = torch.version.cuda\n            env[\"gpus\"] = [\n                {\n                    \"name\": torch.cuda.get_device_name(i),\n                    \"capability\": \".\".join(map(str, torch.cuda.get_device_capability(i))),\n                    \"memory_gb\": round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1),\n                }\n                for i in range(torch.cuda.device_count())\n            ]\n    except ImportError:\n        env[\"cuda_available\"] = False\n    try:\n        out = subprocess.run(\n            [\"nvidia-smi\", \"--query-gpu=driver_version\", \"--format=csv,noheader\"],\n            capture_output=True, text=True, timeout=10,\n        )\n        if out.returncode == 0:\n            env[\"driver\"] = out.stdout.strip().splitlines()[0]\n    except (FileNotFoundError, subprocess.TimeoutExpired):\n        pass\n    return env\n\n\ndef gpu_name() -> str | None:\n    try:\n        import torch\n\n        if torch.cuda.is_available():\n            return torch.cuda.get_device_name(0)\n    except ImportError:\n        pass\n    return None\n\n\ndef default_dtype() -> str:\n    \"\"\"bf16 where the GPU supports it (Ampere+), fp16 otherwise (T4, V100).\"\"\"\n    try:\n        import torch\n\n        if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:\n            return \"bfloat16\"\n    except ImportError:\n        pass\n    return \"float16\"\n\n\ndef percentile(values: list[float], q: float) -> float:\n    import numpy as np\n\n    return float(np.percentile(np.asarray(values, dtype=float), q)) if values else float(\"nan\")\n",
"src/foveate/configs/default.yaml": "# Foveate experiment configuration.\n# A \"visual config\" is written MODEL:FRAMESxRES, e.g. 3b:8x448 means the 3B model,\n# 8 uniformly sampled frames, each resized to about 448*448 pixels (aspect kept).\n\nmodels:\n  3b:\n    id: Qwen/Qwen2.5-VL-3B-Instruct\n    quantization: null\n  7b:\n    # AWQ 4-bit build: fits one 16 GB T4 next to its KV cache, and is the\n    # \"bigger model\" rung of the cascade.\n    id: Qwen/Qwen2.5-VL-7B-Instruct-AWQ\n    quantization: awq\n\n# Configs evaluated on every question (stage 2). Cost grows with\n# ceil(frames/2) * (res/28)^2 visual tokens, see src/foveate/tokens.py.\ngrid:\n  mvbench:\n    3b: [4x224, 8x224, 16x224, 4x448, 8x448, 16x448, 8x672]\n    7b: [8x448, 16x448]\n  lingoqa:\n    3b: [2x224, 4x224, 2x448, 4x448, 4x672]\n    7b: [4x448]\n  # 24 generated questions: a one-minute smoke test of the GPU stack.\n  synthetic:\n    3b: [2x224, 8x448]\n    7b: [8x448]\n\n# Escalation order for the cascade (stage 3). Each rung must be in the grid.\nladder:\n  mvbench: [\"3b:4x224\", \"3b:8x448\", \"3b:16x448\", \"7b:16x448\"]\n  lingoqa: [\"3b:2x224\", \"3b:4x448\", \"7b:4x448\"]\n  synthetic: [\"3b:2x224\", \"3b:8x448\", \"7b:8x448\"]\n\n# Distribution shift (stage 4): corruptions applied to every frame.\nshift:\n  corruptions: [fog, night, blur]\n  severity: 3\n  # Configs re-run on the shifted copies: the ladder minus its middle rung, to\n  # keep the shift study to about a third of the grid's GPU time.\n  configs:\n    mvbench: [\"3b:4x224\", \"3b:16x448\", \"7b:16x448\"]\n    lingoqa: [\"3b:2x224\", \"3b:4x448\", \"7b:4x448\"]\n    synthetic: [\"3b:2x224\", \"3b:8x448\"]\n\n# Stage 1 profiler sweep (Transformers backend, batch 1).\nprofile:\n  frames: [1, 2, 4, 8, 16]\n  res: [224, 448, 672, 896]\n  out_tokens: 32\n  decode_sweep_out_tokens: [1, 16, 64, 256]\n  repeats: 10\n  warmup: 3\n\npilot_fraction: 0.30\nseed: 0\n",
"scripts/run_all.sh": "#!/usr/bin/env bash\n# Run every stage end to end.  MODE=pilot (default, fixed 30%) or MODE=full.\n#   MODE=pilot GPUS=2 TASKS=all bash scripts/run_all.sh\nset -euo pipefail\nMODE=${MODE:-pilot}\nGPUS=${GPUS:-$(nvidia-smi -L 2>/dev/null | wc -l)}\nGPUS=${GPUS:-1}\nTASKS=${TASKS:-all}           # all | lite (skips the 11 GB Perception Test zip)\nRUN_LINGOQA=${RUN_LINGOQA:-1}\nRUN_SERVING=${RUN_SERVING:-1}\necho \"mode=$MODE gpus=$GPUS tasks=$TASKS\"\n\nfoveate env > results_env.json || true\n\n# Download model weights once, before parallel workers race for them\npython -c \"from huggingface_hub import snapshot_download as d; [d(m) for m in ['Qwen/Qwen2.5-VL-3B-Instruct','Qwen/Qwen2.5-VL-7B-Instruct-AWQ']]\"\n\n# One-minute smoke test of the GPU stack on 24 generated questions\nfoveate grid --dataset synthetic --full --model 3b --probe 2\n\n# Stage 1: profile (one GPU is enough; synthetic frames, no dataset needed)\nfoveate profile --model 3b --source synthetic\n\n# Stage 2: quality grid on MVBench, both models\nfoveate download mvbench --tasks \"$TASKS\"\nfoveate grid --dataset mvbench --$MODE --tasks \"$TASKS\" --model 3b --gpus \"$GPUS\"\nfoveate grid --dataset mvbench --$MODE --tasks \"$TASKS\" --model 7b --gpus \"$GPUS\"\n\n# Stage 4 data: shifted copies (fog, night, blur) for the ladder configs\nfoveate shift --dataset mvbench --$MODE --tasks \"$TASKS\" --model 3b --gpus \"$GPUS\"\nfoveate shift --dataset mvbench --$MODE --tasks \"$TASKS\" --model 7b --gpus \"$GPUS\"\n\n# Stages 3-4: replay policies and shift from the cache (CPU, seconds)\nfoveate replay --dataset mvbench --$MODE --tasks \"$TASKS\"\n\n# Driving demo: LingoQA (needs Google Drive access; skipped on failure)\nif [[ \"$RUN_LINGOQA\" == \"1\" ]]; then\n  if foveate download lingoqa; then\n    foveate grid --dataset lingoqa --$MODE --model 3b --gpus \"$GPUS\"\n    foveate grid --dataset lingoqa --$MODE --model 7b --gpus \"$GPUS\"\n    foveate judge --dataset lingoqa --$MODE\n    foveate replay --dataset lingoqa --$MODE\n  else\n    echo \"LingoQA download failed; see README (manual download)\"\n  fi\nfi\n\n# Stage 5: serving under load\nif [[ \"$RUN_SERVING\" == \"1\" ]]; then\n  MODE=$MODE TASKS=$TASKS bash scripts/serve_and_load.sh\nfi\n\nfoveate report\necho \"done: see ${FOVEATE_RESULTS:-results}/REPORT.md\"\n",
"scripts/serve_and_load.sh": "#!/usr/bin/env bash\n# Stage 5: start one vLLM server per GPU, the Foveate gateway, then run the load test.\nset -euo pipefail\nMODE=${MODE:-pilot}\nTASKS=${TASKS:-all}\nRATES=${RATES:-\"0.5 1 2\"}\nN=${N:-200}\nMAXRES=${MAXRES:-672}\nRES=${FOVEATE_RESULTS:-results}\nmkdir -p \"$RES/serving\"\nMMKW=\"{\\\"min_pixels\\\": 784, \\\"max_pixels\\\": $((MAXRES * MAXRES))}\"\nGPU7B=$(( $(nvidia-smi -L | wc -l) > 1 ? 1 : 0 ))\nMEM=$([[ $GPU7B == 0 ]] && echo 0.42 || echo 0.88)   # share one GPU if only one exists\n\nCUDA_VISIBLE_DEVICES=0 nohup vllm serve Qwen/Qwen2.5-VL-3B-Instruct --port 8001 \\\n  --dtype half --max-model-len 8192 --gpu-memory-utilization $MEM \\\n  --limit-mm-per-prompt '{\"video\": 1, \"image\": 0}' --mm-processor-kwargs \"$MMKW\" \\\n  --enable-prefix-caching > $RES/serving/vllm_3b.log 2>&1 &\nP1=$!\nCUDA_VISIBLE_DEVICES=$GPU7B nohup vllm serve Qwen/Qwen2.5-VL-7B-Instruct-AWQ --port 8002 \\\n  --dtype half --quantization awq --max-model-len 8192 --gpu-memory-utilization $MEM \\\n  --limit-mm-per-prompt '{\"video\": 1, \"image\": 0}' --mm-processor-kwargs \"$MMKW\" \\\n  --enable-prefix-caching > $RES/serving/vllm_7b.log 2>&1 &\nP2=$!\ntrap 'kill $P1 $P2 ${P3:-} 2>/dev/null || true' EXIT\n\nfor port in 8001 8002; do\n  echo \"waiting for vLLM on :$port\"\n  for i in $(seq 1 180); do curl -sf localhost:$port/health >/dev/null && break; sleep 5; done\n  curl -sf localhost:$port/health >/dev/null || { echo \"vLLM :$port failed; see $RES/serving/*.log\"; exit 1; }\ndone\n\nPOLICY=$RES/mvbench_${MODE}/replay/policy.json\nnohup foveate serve --dataset mvbench --$MODE --tasks \"$TASKS\" --policy-file \"$POLICY\" \\\n  > $RES/serving/gateway.log 2>&1 &\nP3=$!\nfor i in $(seq 1 60); do curl -sf localhost:8000/health >/dev/null && break; sleep 2; done\n\nREF=$(python -c \"import json;print(json.load(open('$RES/mvbench_${MODE}/replay/summary.json'))['reference'])\")\nfor rate in $RATES; do\n  foveate loadtest --dataset mvbench --$MODE --tasks \"$TASKS\" --policy cascade --rate $rate --n $N\n  foveate loadtest --dataset mvbench --$MODE --tasks \"$TASKS\" --policy \"static:$REF\" --rate $rate --n $N\ndone\n",
"scripts/setup_gpu.sh": "#!/usr/bin/env bash\n# Install Foveate plus a vLLM build that matches this machine's NVIDIA driver.\n# Usage: bash scripts/setup_gpu.sh            (from the repo root)\nset -euo pipefail\n\n# CUDA version the driver supports, e.g. \"12.8\" or \"13.0\".\nDRV_CUDA=$(nvidia-smi 2>/dev/null | sed -n 's/.*CUDA Version: \\([0-9]*\\.[0-9]*\\).*/\\1/p' | head -1)\necho \"driver supports CUDA ${DRV_CUDA:-unknown}\"\nmajor=${DRV_CUDA%%.*}; minor=${DRV_CUDA##*.}\nif [[ -z \"${DRV_CUDA}\" ]]; then\n  echo \"no NVIDIA GPU found; installing CPU-only extras\"; VLLM_SPEC=\"\"\nelif (( major >= 13 )); then\n  VLLM_SPEC=\"vllm==0.32.0\"          # tested API target (torch cu130 wheels)\nelif (( major == 12 && minor >= 8 )); then\n  VLLM_SPEC=\"vllm==0.11.0\"          # torch 2.8 / cu128 wheels\nelse\n  VLLM_SPEC=\"vllm==0.9.2\"           # torch 2.7 / cu126 wheels\nfi\n# Override with: VLLM_OVERRIDE=\"vllm==X.Y.Z\" bash scripts/setup_gpu.sh\nVLLM_SPEC=${VLLM_OVERRIDE:-$VLLM_SPEC}\n\nif [[ -n \"${VLLM_SPEC}\" ]]; then\n  echo \"installing ${VLLM_SPEC}\"\n  pip install -q \"${VLLM_SPEC}\" || echo \"WARNING: vLLM install failed; the grid will fall back to Transformers\"\nfi\npip install -q -e \".[data,hf,serve,dev]\"\npython - <<'PY'\nimport importlib, torch\nprint(\"torch\", torch.__version__, \"cuda ok:\", torch.cuda.is_available(),\n      \"gpus:\", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])\ntry:\n    v = importlib.import_module(\"vllm\"); print(\"vllm\", v.__version__)\nexcept Exception as e:\n    print(\"vllm not importable:\", e)\nPY\n",
"theory/01_anatomy_of_vlm_inference.md": "# 01 · Anatomy of VLM inference (Stage 1: the profiler)\n\n> **One sentence to remember:** for a short answer about a video, the cost is set\n> almost entirely by how many visual tokens you create, and you create them with two\n> knobs: how many frames you send and how many pixels each frame has.\n\nThis note explains what `foveate profile` measures and why the numbers come out the\nway they do. All arithmetic uses Qwen2.5-VL-3B on a Kaggle **T4** (16 GB, fp16).\nWhere a number is a back-of-envelope estimate, it says so. Your profiler output is\nthe ground truth that confirms or corrects it.\n\n---\n\n## 1. The life of one request\n\n```\n video file ──► decode + sample frames ──► resize ──► processor (patchify, normalise)     CPU\n                                                         │\n                                       H2D copy ◄────────┘\n                                          │\n                  ┌───────────────────────▼──────────────────────┐\n                  │ vision encoder (ViT, ~0.67B params)           │  GPU, runs once\n                  │ 14×14 patches → features → 2×2 merge → tokens │\n                  └───────────────────────┬──────────────────────┘\n                                          ▼\n                  ┌──────────────────────────────────────────────┐\n                  │ LLM prefill: all visual + text tokens at once │  GPU, compute-bound\n                  │ writes the KV cache  ──────────► TTFT ends     │\n                  └───────────────────────┬──────────────────────┘\n                                          ▼\n                  ┌──────────────────────────────────────────────┐\n                  │ decode: 1 token per step, reads all weights   │  GPU, memory-bound\n                  │ + KV cache every step ──────────► TPOT         │\n                  └──────────────────────────────────────────────┘\n```\n\nThe profiler times each box separately (see `src/foveate/backends/hf_backend.py`):\n\n| Stage | How it is timed |\n|---|---|\n| load (decode + resize) | `time.perf_counter` around `FrameStore.load` |\n| preprocess | `perf_counter` around the HF processor call |\n| H2D | CUDA events around `.to(device)` |\n| vision encoder | CUDA events from forward **pre/post hooks** on `model.visual` |\n| prefill | first generate step minus H2D minus vision |\n| decode | events from a do-nothing **logits processor**, which runs once per generated token |\n\n---\n\n## 2. Where visual tokens come from\n\nQwen2.5-VL cuts each frame into **14×14-pixel patches**, then an MLP **merges every\n2×2 block of patches into one LLM token**. So one token covers **28×28 pixels**. For\nvideo it also fuses **every 2 consecutive frames** into one temporal patch (a 3D\nconvolution with temporal size 2), which halves the token count again.\n\n$$\n\\text{visual tokens} = \\left\\lceil \\frac{F}{2} \\right\\rceil \\cdot \\frac{H}{28} \\cdot \\frac{W}{28}\n$$\n\n| Setting | Visual tokens |\n|---|---|\n| 4 frames, 224×224 | 2 × 64 = **128** |\n| 8 frames, 448×448 | 4 × 256 = **1,024** |\n| 16 frames, 448×448 | 8 × 256 = **2,048** |\n| 8 frames, 672×672 | 4 × 576 = **2,304** |\n| 16 frames, 672×672 | 8 × 576 = **4,608** |\n\n**Doubling resolution (per side) multiplies tokens by 4; doubling frames multiplies\nthem by 2.** Resolution is the expensive knob. `tokens.py` implements the exact\nrounding rule (`smart_resize`), and `tests/test_tokens.py` checks the table above.\n\n> **Trend check.** Every frontier API now exposes this knob to users. OpenAI's image\n> input has a `detail: low` mode that costs a fixed 85 tokens; Gemini has a\n> `media_resolution` setting. Inside models, token reduction is an active research\n> line: FastV (ECCV 2024) drops low-attention visual tokens after layer 2, and\n> VisionZip and PyramidDrop (both CVPR 2025) prune or merge tokens. NVIDIA's NVILA\n> calls the recipe \"scale then compress\". Foveate works *outside* the model (frames\n> and pixels), so it composes with any of those.\n\n---\n\n## 3. Prefill is compute-bound\n\nA transformer forward pass costs about **2 FLOPs per parameter per token** for the\nmatrix multiplies. Prefill processes all prompt tokens in parallel, so the GPU's\ntensor cores stay busy.\n\nFor 16 frames at 448 (2,048 visual tokens + ~100 text tokens):\n\n- **LLM:** 2 × 3.09e9 × 2,150 ≈ **13.3 TFLOP**\n- **Vision encoder:** it runs on the patches *before* the 2×2 merge, so 4× as many:\n  2 × 0.67e9 × 8,192 ≈ **11.0 TFLOP**\n\nA T4 peaks at **65 TFLOP/s** in fp16 on paper. Real kernels reach roughly 30 to 50%\nof that, so this request needs about **0.75 to 1.2 s** of prefill+vision on a T4.\n\n**The non-obvious finding to look for:** on a 3B VLM, the vision encoder is nearly\n**half** of the prefill compute (11.0 vs 13.3 TFLOP), because it sees 4× more\npositions than the LLM. On the 7B model the LLM is 2.5× bigger while the ViT is the\nsame, so the vision share drops to about a quarter. Small VLMs are vision-bound\nearlier than people expect. The profiler's `vision_mfu` and `prefill_mfu` columns\n(achieved TFLOP/s ÷ spec peak) let you check this.\n\n---\n\n## 4. Decode is memory-bound\n\nDecode generates **one token per step**. Each step multiplies a single vector by every\nweight matrix, so the GPU does little math per byte loaded. The limit is how fast it\ncan stream the weights from memory.\n\n$$\n\\text{TPOT}_{\\min} \\approx \\frac{\\text{weight bytes}}{\\text{memory bandwidth}}\n= \\frac{3.09\\text{e}9 \\times 2\\ \\text{B}}{320\\ \\text{GB/s}} \\approx 19\\ \\text{ms}\n$$\n\nThat is a ceiling of about 50 tokens/s for one request on a T4. **Batching** is how\nservers beat it: 32 requests in one step share a single read of the weights. This is\nthe whole reason continuous batching exists (see note 05).\n\nFor multiple-choice questions Foveate generates **one token**, so decode is almost\nfree and the question becomes purely a prefill question. The profiler's decode sweep\n(1, 16, 64, 256 output tokens) shows the crossover point where decode starts to\ndominate.\n\n### Roofline in one line\n\nArithmetic intensity = FLOPs per byte moved. The T4's ridge point is\n65e12 / 320e9 ≈ **203 FLOP/byte**. Prefill on thousands of tokens sits far above it\n(compute-bound); batch-1 decode sits near **1 FLOP/byte** (memory-bound).\n\n---\n\n## 5. KV cache\n\nEach token stores a key and a value vector per layer per KV head:\n\n$$\n\\text{KV bytes/token} = 2 \\times L \\times H_{kv} \\times d_{head} \\times 2\\ \\text{B}\n$$\n\nQwen2.5-VL-3B: 2 × 36 × 2 × 128 × 2 = **36 KB per token** (grouped-query attention\nwith only 2 KV heads keeps it small). A 2,048-token video request needs about 75 MB.\nKV is not the bottleneck at batch 1. It *is* the bottleneck when vLLM packs dozens of\nvideo requests into one T4, which is when PagedAttention matters (note 05).\n\n---\n\n## 6. Measuring correctly (the part most benchmarks get wrong)\n\n1. **CUDA is asynchronous.** A kernel launch returns before the GPU finishes, so\n   `time.time()` around a model call measures Python, not the GPU. Use\n   `torch.cuda.Event(enable_timing=True)`, which is recorded inside the GPU stream, and\n   synchronize once *after* the timed region.\n2. **Warm up.** The first calls pay for CUDA context creation, cuBLAS algorithm\n   selection and the caching allocator growing. The profiler discards warm-up runs.\n3. **Report medians and p95, not means.** Latency distributions have right tails.\n4. **Record the environment.** GPU, driver, CUDA, torch, transformers and vLLM\n   versions go in `*.env.json` next to every result.\n5. **Say where each number came from.** Stage timings come from Transformers\n   (because we own the model object). Serving numbers (TTFT under load, p95) come from\n   vLLM. They are different engines and will not match exactly.\n\n---\n\n## 7. T4-specific caveats (Kaggle)\n\n- **No bf16.** Turing GPUs (compute capability 7.5) support fp16 and fp32 only.\n  Qwen2.5-VL was trained in bf16, and fp16 has a much smaller range (max ≈ 65,504), so\n  activations can overflow. The grid runner checks the first batch for NaNs or a single\n  repeated answer. If you see that, rerun with `--dtype float32 --tp 2` (the 3B model\n  in fp32 needs ~15 GB of weights, which fits only split across both T4s).\n- **No FlashAttention-2** (it needs Ampere or newer). Transformers uses PyTorch SDPA,\n  and vLLM picks its Triton attention backend on Turing.\n- **Two GPUs, used as data parallel.** `--gpus 2` runs one process per T4, each on half\n  the questions. That beats tensor parallelism here because the T4s talk over PCIe,\n  and a 3B model fits on one card anyway.\n\n---\n\n## 8. What to look for in your Stage 1 results\n\n- `ttft_vs_tokens.png`: TTFT should be close to **linear in visual tokens** (R² near 1).\n  The slope is your GPU's \"ms per visual token\". A curve bending upward means attention's\n  quadratic term is starting to show.\n- `profile_breakdown.png`: at low token counts, CPU **load + preprocess** can rival the\n  GPU time. That is a real finding: on Kaggle's 4 vCPUs the video decoder can become the\n  bottleneck, which is why Foveate caches decoded frames.\n- `prefill_mfu`: if it is far below 0.3, something is inefficient (padding, small\n  kernels, fp32 fallback).\n\n**Read next:** the Qwen2.5-VL technical report (arXiv 2502.13923), sections on the\nvision encoder and dynamic resolution; the PyTorch profiler docs; then note 02.\n",
"theory/02_benchmarks_confidence_and_statistics.md": "# 02 · Benchmarks, confidence and honest statistics (Stage 2: the quality grid)\n\n> **One sentence to remember:** run every question through every config once, cache\n> the answer, the confidence and the cost, and you can evaluate any policy later in\n> seconds instead of GPU hours.\n\n---\n\n## 1. Why MVBench is the main benchmark\n\nThe ask was a dataset that is widely used and fast to run. MVBench (Li et al., CVPR\n2024) is both:\n\n- **Widely used.** It sits in the video table of almost every recent open VLM report.\n  The Qwen2.5-VL report gives **67.0** for the 3B model and **69.6** for the 7B. That\n  gives you a published number to sanity-check your harness against.\n- **Fast.** 20 tasks × 200 questions, short clips, and every answer is one option\n  letter, so a question costs one prefill and *one* decode step.\n- **Temporal by design.** Its tasks (action sequence, moving direction, object\n  shuffle, counterfactual inference...) were built from existing datasets so that a\n  single frame should not be enough. That makes the frames knob meaningful.\n\nTwo honest caveats:\n\n1. **Your accuracy will be lower than 67.0.** Qwen evaluated with up to 768 frames at\n   2 fps. Foveate deliberately uses 4 to 16 frames, because the project is about the\n   cost/accuracy trade-off. Being 5 to 10 points under at 16 frames is expected; being\n   30 points under means a bug.\n2. **One task is missing.** `fine_grained_pose` uses NTU RGB+D videos, which are not\n   redistributed on the Hub. Foveate runs the other 19. `--tasks lite` drops 4 more\n   (the 11 GB Perception Test zip) when disk or time is tight.\n\n**LingoQA** (Wayve, ECCV 2024) is the driving demo: 500 questions on 100 driving\nclips, free-form answers scored by Lingo-Judge, a small classifier the authors trained\nbecause BLEU-style metrics correlate badly with human judgement on this data.\n\n> **Trend check.** Benchmarks leak shortcuts. \"Revealing Single Frame Bias for\n> Video-and-Language Learning\" (Lei et al., 2022) showed many video QA sets are mostly\n> solvable from one frame, and MMStar (2024) showed many image benchmarks are partly\n> solvable with no image at all. Foveate's per-task heatmap tests this directly: a task\n> whose accuracy does not move from 4 to 16 frames is not really temporal for this model.\n\n---\n\n## 2. Scoring multiple choice from logits\n\nInstead of generating text and parsing it, Foveate reads the next-token\nlog-probabilities of the option letters and renormalises over the valid ones:\n\n$$\np(\\text{A}) = \\frac{e^{\\ell_A}}{\\sum_{k \\in \\text{options}} e^{\\ell_k}}\n$$\n\nThe prediction is the most probable letter. This removes parsing errors, costs one\ndecode step, and gives three **confidence signals** for free:\n\n| Signal | Definition | Intuition |\n|---|---|---|\n| `conf` | max option probability | how sure the model is |\n| `margin` | top-1 minus top-2 probability | how close the runner-up is |\n| `entropy` | −Σ p log p over options | how spread out the belief is |\n\nFor free-form answers (LingoQA) the signal is the **geometric-mean token probability**\nof the generated answer, exp(mean log-prob). It is weaker: a fluent wrong sentence can\nbe confident.\n\n---\n\n## 3. Is the confidence any good? AUROC and ECE\n\nThe cascade in note 03 only works if confidence separates right answers from wrong\nones. Two numbers in `REPORT.md` measure that per config:\n\n- **AUROC**: the probability that a randomly chosen correct answer has higher\n  confidence than a randomly chosen wrong one. 0.5 is useless; 0.8 is good.\n  This is what the cascade needs, since it only *ranks* questions by confidence.\n- **ECE** (expected calibration error): bin answers by confidence and compare each\n  bin's average confidence with its accuracy. Low ECE means \"70% sure\" really is right\n  70% of the time. The cascade does not need calibration, only ranking, but\n  calibration matters if you show confidences to users.\n\nModern instruction-tuned models are usually **overconfident**: RLHF and instruction\ntuning sharpen the output distribution. Expect ECE well above zero and AUROC between\n0.65 and 0.8.\n\n---\n\n## 4. The pilot subset: 30% that you can trust\n\n`--subset pilot` picks a **fixed-seed, task-stratified 30%** (≈1,140 MVBench questions,\n150 LingoQA questions). Why these choices:\n\n- **Stratified** so every task keeps ~60 questions and per-task accuracy is still\n  estimable.\n- **Fixed seed** so pilot runs on different days hit the same questions and can be\n  compared directly.\n- **How precise is a pilot?** The standard error of an accuracy p on n questions is\n  √(p(1−p)/n). At p = 0.6 and n = 1,140 that is ±1.5 points (±2.9 for a 95%\n  interval). Good enough to rank configs that differ by 3+ points; run `--full` for\n  the headline numbers (±1.6 points at 95%).\n\n---\n\n## 5. Bootstrap intervals and *paired* comparisons\n\nEvery accuracy in the report carries a **bootstrap 95% interval**: resample questions\nwith replacement 2,000 times, recompute, and take the 2.5th and 97.5th percentiles.\n\nThe important subtlety is **pairing**. Two configs answer the *same* questions, and a\nhard question is hard for both. So the interval on their *difference* is computed by\nresampling questions once and differencing within each resample\n(`paired_bootstrap_diff`). That interval is often 2 to 3× tighter than you would guess\nby comparing two separate intervals, which is exactly what lets a pilot detect a\n2-point difference.\n\n---\n\n## 6. Cost: two kinds of milliseconds\n\nThe grid records cost per config in two ways (`results/<run>/cost/*.json`):\n\n| Metric | What it is | Use it for |\n|---|---|---|\n| `latency_b1_ms` | median batch-1 end-to-end time from a small probe | what one user waits |\n| `amortized_ms` | batched GPU wall time ÷ questions | what the GPU pays (throughput cost) |\n\nThey can disagree: batching helps cheap configs more than expensive ones, because\ncheap configs leave the GPU underused at batch 1. `foveate replay --cost amortized_ms`\nswitches the policy analysis to throughput cost.\n\n---\n\n## 7. Why caching the grid is the key engineering move\n\nThe grid is a matrix of `questions × configs`, each cell holding (correct, confidence,\ntokens). Any policy (a fixed config, a rule, a cascade, a drift response) is a\nfunction that walks this matrix. So:\n\n- trying a new policy costs **seconds**,\n- every policy is evaluated on **exactly the same answers** (a paired comparison), and\n- live serving (note 05) only has to confirm that replayed numbers survive real load.\n\nThis \"simulate from a log\" pattern is how production teams evaluate routing and\ncaching changes before shipping them.\n\n**Read next:** the MVBench paper (arXiv 2311.17005), section 3 on task construction;\nGuo et al. 2017, \"On Calibration of Modern Neural Networks\"; then note 03.\n",
"theory/03_adaptive_compute_and_cascades.md": "# 03 · Adaptive compute and cascades (Stage 3: policies)\n\n> **One sentence to remember:** most questions are easy, so answer everything cheaply\n> first and pay for more frames, pixels or parameters only when the cheap answer looks\n> unsure.\n\n---\n\n## 1. The optimisation problem\n\nFor each question *x*, a policy π chooses which configs to run. We want\n\n$$\n\\min_{\\pi}\\ \\mathbb{E}_x\\big[\\text{cost}(\\pi, x)\\big]\n\\quad \\text{s.t.} \\quad\n\\text{accuracy}(\\pi) \\ge \\text{accuracy}(\\text{reference}) - \\delta\n$$\n\nwhere the reference is the best fixed config and δ is the accuracy you are willing to\ngive up (`--tol`, default 0). Foveate compares five kinds of policy, all by replaying\nthe grid cache:\n\n| Policy | Decides with | Cost |\n|---|---|---|\n| static | nothing: one config for all | cost of that config |\n| rules | question keywords → config | one config per question |\n| cascade | confidence after each rung | sum of every rung it ran |\n| rules + cascade | keywords pick the first rung, confidence escalates | sum of rungs run |\n| oracle | the answer key | cheapest correct config (unreachable bound) |\n\nThe **oracle** is not a policy you can deploy. It tells you the *headroom*: if the\noracle saves 70% and your cascade saves 30%, the gap is a better confidence signal\naway.\n\n---\n\n## 2. The cascade's cost model\n\nWith a cheap rung of cost c₁, an expensive rung of cost c₂ and escalation rate p\n(the share of questions whose cheap confidence is below τ):\n\n$$\n\\mathbb{E}[\\text{cost}] = c_1 + p\\,c_2\n$$\n\nIt beats always running the expensive rung when\n\n$$\nc_1 + p\\,c_2 < c_2 \\iff p < 1 - \\frac{c_1}{c_2}\n$$\n\nWorked example from the default ladder: 4 frames at 224 is 128 visual tokens, and 16\nframes at 448 is 2,048. If latency scales with tokens, c₁/c₂ ≈ 0.06, so the cascade\nstill saves money even if it escalates **94%** of questions. Cheap rungs this cheap\nare almost free insurance. The real constraint is accuracy: escalating on the wrong\nquestions, or *not* escalating on confidently-wrong ones, is what loses points.\n\n**Failure mode to watch:** confidently wrong cheap answers. If the cheap rung answers\n\"A\" with 0.9 confidence and is wrong, no threshold rescues it. The AUROC column of the\ncheap config bounds how good any cascade can be.\n\n---\n\n## 3. Picking τ without fooling yourself\n\nIf you sweep τ on the test questions and report the best point, you have tuned on the\ntest set and the number is optimistic. Foveate splits the questions into a\n**calibration half** and a **held-out half** (stratified by task, fixed seed):\n\n1. Sweep τ over quantiles of the confidence on the calibration half.\n2. Pick the cheapest τ whose calibration accuracy reaches the reference's accuracy\n   minus δ.\n3. Report accuracy and cost **on the held-out half**, with a paired-bootstrap interval\n   on the accuracy difference versus the reference.\n\nThe full curve over all τ (drawn on all questions) appears in the frontier figure for\nintuition. The held-out points are the claims.\n\n> **Going further: guarantees.** Conformal risk control (Angelopoulos et al., 2022)\n> turns step 2 into a statement like \"with probability 1−α, held-out accuracy loss is\n> at most δ\" under exchangeability. It is a natural upgrade once the basic cascade\n> works.\n\n---\n\n## 4. Rules: using the question text\n\nThe question already says something about what evidence it needs:\n\n| Category | Keyword cues | Hypothesis |\n|---|---|---|\n| temporal | before, after, then, sequence, order, happen | needs more **frames** |\n| motion | moving, direction, toward, turn, collide, lane | needs more **frames** |\n| count | how many, number of, times | frames and pixels |\n| detail | colour, text, sign, shape, light, small | needs more **pixels** |\n| other | anything else | default |\n\nFoveate does not hard-code the mapping. `fit_rules` learns, per category on the\ncalibration half, the **cheapest config within 1 point of that category's best**. The\nlearned mapping is printed in the report. If \"temporal\" maps to 16 frames at 224 and\n\"detail\" maps to 8 frames at 672, the hypothesis holds; if not, that is a finding too.\n\nRules decide before seeing any answer (no wasted cheap call). Cascades decide after\nseeing one (better information, wasted cheap call). **Rules + cascade** uses the\nrule to pick the starting rung, then escalates.\n\n---\n\n## 5. Ablations\n\nThe report's ablation table removes one rung at a time from the cascade (keeping τ\nfixed). If dropping the middle rung costs no accuracy, the ladder is too long; if\ndropping the 7B rung costs a lot, model size is doing work that frames and pixels\ncannot.\n\n---\n\n## 6. Where this sits in current research and industry\n\nAdaptive compute is one of the main ideas in inference right now, at every level:\n\n- **Model cascades and routers.** FrugalGPT (Chen, Zaharia & Zou, 2023) cascades API\n  models by a learned answer scorer; RouteLLM (LMSYS, 2024) trains routers between a\n  strong and a weak LLM from preference data. Foveate applies the idea to the *visual\n  input* (frames, pixels) as well as model size.\n- **Early exit inside a model.** CALM (Schuster et al., NeurIPS 2022) stops computing\n  layers once a token's prediction is confident. Same principle, finer grain.\n- **Test-time compute scaling.** Reasoning models (OpenAI o1, DeepSeek-R1) spend more\n  tokens on harder prompts, and Snell et al. (2024) showed compute-optimal allocation\n  by difficulty beats uniform allocation. A cascade is the cheapest version of\n  \"difficulty-aware compute\".\n- **Video-specific.** Query-aware frame selection and token pruning for long video are\n  an active line. The novelty warning from the project plan stands: Foveate's claim is\n  not \"adaptive frames is new\", it is a **measured** cost/accuracy trade-off on a real\n  serving stack, with held-out evaluation and an honest oracle gap.\n\n---\n\n## 7. What a good Stage 3 result looks like\n\nA sentence you can defend, for example:\n\n> \"On the MVBench held-out half, the rules+cascade policy matched the best fixed\n> config's accuracy (difference −0.4 points, 95% CI [−1.5, +0.7]) at 38% lower mean\n> batch-1 latency, escalating 41% of questions.\"\n\nThe numbers in that sentence are placeholders. Every number in your README must come\nfrom `results/REPORT.md`. If the cascade does *not* beat the best static config, say\nso and use the confidence AUROC and oracle gap to explain why. That is also a result.\n\n**Read next:** FrugalGPT (arXiv 2305.05176); RouteLLM (arXiv 2406.18665); then note 04.\n",
"theory/04_distribution_shift_and_drift.md": "# 04 · Distribution shift and drift (Stage 4: your original idea, made measurable)\n\n> **One sentence to remember:** a drift detector tells you the inputs changed, not that\n> the answers got worse; the cascade is what turns \"inputs got harder\" into \"spend more\n> compute on this request\".\n\n---\n\n## 1. What kind of shift are we simulating?\n\nMachine learning distinguishes a few kinds of shift between training/reference data\nand live traffic:\n\n| Shift | What changes | Driving example |\n|---|---|---|\n| **covariate** | p(x), the inputs | night, fog, rain, motion blur, a new camera |\n| **label / prior** | p(y) | more pedestrians in a city than on a highway |\n| **concept** | p(y given x) | a traffic rule changes the right answer |\n\nFoveate simulates **covariate shift** with three corruptions in the style of\nImageNet-C (Hendrycks & Dietterich, ICLR 2019), at severities 1 to 5\n(`data/corrupt.py`):\n\n- **fog**: the atmospheric scattering model I = J·t + A·(1−t), with transmission t\n  falling toward the top of the frame where the scene is farther away;\n- **night**: gamma darkening, a blue cast, and sensor noise that grows as light drops;\n- **blur**: horizontal motion blur (ego-motion) plus slight defocus.\n\nThey are deterministic (seeded per frame), so a shifted run is reproducible. Real\nadverse-condition data (night and rain splits in driving datasets) is the natural next\nstep.\n\n---\n\n## 2. Three things to measure\n\n`foveate shift` reruns the ladder configs on corrupted frames; `foveate replay` then\nreports:\n\n1. **Per corruption:** cheap-rung accuracy, top-rung accuracy, cascade accuracy,\n   **escalation rate** and cost, clean versus shifted, using the τ fitted on clean data.\n   The hoped-for pattern: escalation rises on shifted inputs, accuracy holds better\n   than the cheap config, and cost rises only there.\n2. **A stream that switches from clean to shifted halfway**, with a drift detector\n   watching it (section 3).\n3. **A drift-aware cascade**: while the alarm is on, skip the cheapest rung, because\n   most of its answers would be escalated anyway. This saves the wasted cheap call\n   on every request during the shift.\n\n---\n\n## 3. A detector that costs nothing\n\nFoveate's detector uses four **model-free image statistics** computed while loading\nframes: brightness, contrast, sharpness (variance of the Laplacian) and saturation.\nFor a sliding window of the last W requests it computes a two-sample z statistic\nagainst a clean reference window:\n\n$$\nz = \\max_{j} \\frac{\\left|\\bar{x}^{\\text{window}}_j - \\mu^{\\text{ref}}_j\\right|}{\\sigma^{\\text{ref}}_j / \\sqrt{W}}\n$$\n\nand raises an alarm when z exceeds a threshold (default 6, which is conservative\nbecause the four features are tested together). The report gives **detection delay**\n(requests after the switch before the alarm) and **false alarms** on the clean half.\n\nWhy not embeddings? A vision-embedding detector (mean shift or MMD on ViT features) is\nmore general, and is the right upgrade. But it needs the model's hidden states, which\nvLLM does not return. The image statistics run on the CPU for free and are enough for\nweather and lighting shifts. \"Failing Loudly\" (Rabanser, Günnemann & Lipton, NeurIPS\n2019) found that simple two-sample tests on reduced representations are strong\nbaselines for shift detection, which supports starting simple.\n\n---\n\n## 4. The three traps in \"drift-aware serving\"\n\n1. **Drift is not failure.** Fog can shift every statistic and the model can still\n   answer correctly. Only labels show accuracy, which is why Foveate reports accuracy\n   on shifted copies of labelled questions rather than trusting the detector.\n2. **Wrong time scale.** A window statistic reacts after W requests. Per-request\n   decisions (escalate this one) must come from per-request signals such as confidence.\n   The detector's job is the slow loop: change the *default* starting rung.\n3. **Confidence degrades under shift too.** Ovadia et al. (NeurIPS 2019, \"Can You\n   Trust Your Model's Uncertainty?\") showed calibration gets worse as shift grows. The\n   report's `shifted_cheap_auroc` column checks this for your model. If AUROC drops\n   from 0.75 to 0.6 under fog, the cascade's threshold picks the wrong questions\n   exactly when you need it most, and the drift-aware start rung becomes more valuable.\n\n---\n\n## 5. Correlations worth reporting\n\nThese come straight out of the shift results:\n\n- **Corruption severity vs. accuracy drop** per config: do more frames or more\n  pixels make the model more robust? (Blur often hurts low resolution less, since the\n  detail was already gone.)\n- **Escalation-rate increase vs. accuracy drop** across corruptions: if they move\n  together, confidence is tracking difficulty.\n- **Drift z vs. cheap-rung confidence** over the stream: does the model-free signal\n  lead the model-based one?\n\n> **Trend check.** Physical-AI teams treat this as \"long-tail\" or \"out-of-distribution\"\n> scenario handling: rare weather, lighting and sensor conditions dominate failure\n> analysis. Validation and safety teams increasingly use VLMs to *mine* such scenarios\n> from fleet logs, which is exactly where inference cost per clip matters. Keep the\n> claim modest: recorded-clip QA under synthetic corruption is a robustness study,\n> not evidence of on-road safety.\n\n**Read next:** ImageNet-C (arXiv 1903.12261); Failing Loudly (arXiv 1810.11953);\nOvadia et al. (arXiv 1906.02530); then note 05.\n",
"theory/05_serving_vlms_under_load.md": "# 05 · Serving VLMs under load (Stage 5: the gateway and load test)\n\n> **One sentence to remember:** under load, p95 latency is mostly *waiting*, so a\n> policy that makes the average request cheaper also shortens everyone's queue.\n\n---\n\n## 1. The system on Kaggle's two T4s\n\n```\n load generator ──HTTP──► foveate gateway (CPU) ──► vllm serve Qwen2.5-VL-3B      GPU 0, :8001\n (Poisson arrivals)        frames, resize, cascade └► vllm serve Qwen2.5-VL-7B-AWQ GPU 1, :8002\n                           telemetry JSONL\n```\n\nThe split of responsibilities is deliberate:\n\n- **vLLM** owns everything per-token: continuous batching, PagedAttention, prefix\n  caching, the multimodal encoder cache.\n- **The gateway** owns everything per-request that vLLM does not know about: how many\n  frames, which resolution, which model, when to escalate.\n\nPlacing one model per GPU turns \"route to the bigger model\" into \"route to the other\nGPU\", which is how multi-model serving fleets work.\n\n---\n\n## 2. What vLLM does for you (and why each piece exists)\n\n**Continuous batching** (Orca, OSDI 2022). A naive server batches requests and waits\nfor the slowest to finish. Continuous batching re-forms the batch at *every decode\nstep*: finished sequences leave, new ones join. Since batch-1 decode is memory-bound\n(note 01), adding requests to a step is nearly free until compute saturates.\n\n**PagedAttention** (vLLM, SOSP 2023). KV cache is stored in fixed-size blocks, like\nvirtual-memory pages, so memory is not reserved for each request's maximum length.\nThis roughly doubled to quadrupled batch sizes in the paper, and batch size is\nthroughput.\n\n**Chunked prefill** (Sarathi-Serve, OSDI 2024). A 4,600-token video prefill would stall\nevery other request's decode. Splitting prefill into chunks and mixing them with\ndecode steps keeps time-between-tokens smooth for everyone.\n\n**Prefix caching** (also SGLang's RadixAttention, 2024). If two requests share a\nprompt prefix, the KV cache for that prefix is reused. Foveate's prompt puts the\n**video before the question**, so \"same clip, different question\" requests share the\nentire visual prefix. For multimodal inputs vLLM hashes the media content, so\nidentical frames hit the cache. The load generator's `--repeat` knob controls how\noften this happens.\n\n**Encoder cache / processor cache.** vLLM also caches vision-encoder outputs and\npreprocessed media between requests, so a repeated clip can skip the ViT, the second\nbiggest cost from note 01.\n\n> **Trend check.** The frontier of serving research is *disaggregation*: run prefill\n> and decode on different GPUs (DistServe and Splitwise, both 2024), and for\n> multimodal models, split the encoder off too (\"encode-prefill-decode\", EPD,\n> 2024-25). The vLLM sessions at PyTorch Conference 2026 were about KV-cache transfer\n> and offloading for exactly this. Foveate's per-stage profile (note 01) is the\n> measurement you would need to decide whether encoder disaggregation pays off.\n\n---\n\n## 3. Queueing: why p95 explodes\n\nModel a server as a queue with arrival rate λ and service rate μ (requests per\nsecond it can finish). Utilisation is ρ = λ/μ. For the simplest queue (M/M/1), mean\ntime in system is\n\n$$\nT = \\frac{1}{\\mu - \\lambda} = \\frac{1/\\mu}{1 - \\rho}\n$$\n\nAt ρ = 0.5 requests take 2× their service time; at ρ = 0.9, 10×. Tails are worse than\nmeans. Two consequences for this project:\n\n1. **A cheaper average request lowers ρ for everyone.** If the cascade cuts mean\n   service cost by 35%, μ rises, ρ falls, and the p95 improvement under load can be\n   much larger than 35%. This is the \"under load\" result Stage 5 exists to show.\n2. **Escalations add variance.** A cascade request that escalates pays twice. Mean\n   cost falls but the cost distribution gets a fatter right tail, which pushes p99\n   up. Report p50, p95 *and* p99.\n\n**Little's law** (L = λ·W) is the sanity check: the average number of requests in\nflight equals arrival rate times average latency. If your telemetry violates it,\nsomething is mismeasured.\n\n---\n\n## 4. Running a fair load test\n\n- **Open loop, Poisson arrivals.** A closed-loop client (send, wait, send) slows down\n  when the server slows down, hiding the queue. `foveate loadtest` schedules every\n  arrival in advance from a seeded exponential distribution.\n- **Same trace for every policy.** Static and cascade policies see the identical\n  sequence of questions at identical times.\n- **Sweep the rate.** One rate tells you little. Run a low rate (latency is pure\n  service time), a medium one, and one near saturation, and plot p95 against rate for\n  both policies. The cascade's curve should bend upward later.\n- **Goodput.** DistServe defines goodput as requests per second that *meet* a latency\n  SLO (for example p95 TTFT < 2 s). It is a better headline than raw throughput,\n  because throughput can rise while every user waits longer.\n- **Warm the servers** with a few requests first, and keep the gateway's frame cache\n  warm too, or the first minute measures disk reads.\n\n---\n\n## 5. T4 serving notes\n\n- The 3B model in fp16 is ~7.5 GB of weights, leaving ~6 GB of KV cache on a 16 GB T4\n  at `--gpu-memory-utilization 0.9`. At 36 KB/token that is ~170k tokens, about 80\n  concurrent 2,048-token video requests in theory, fewer in practice because of\n  activation memory and the encoder cache.\n- The 7B-AWQ model is ~6 GB of int4 weights; its KV is 56 KB/token (28 layers, 4 KV\n  heads).\n- `--enforce-eager` skips CUDA-graph capture: faster start-up and less memory, slower\n  decode. Use it if start-up fails on memory.\n- Kaggle notebooks can reach `localhost` ports, so the whole stack runs inside one\n  notebook with servers in background processes.\n\n---\n\n## 6. What a good Stage 5 result looks like\n\nA table from `REPORT.md` with, for the same Poisson trace at two or three rates:\nthroughput, p50/p95/p99 latency, p95 TTFT, accuracy and escalation rate for \"always\n16×448 on 3B\" versus the cascade. The claim you can defend is the one the table\nsupports, including the honest case where escalation variance hurts p99.\n\n**Read next:** the vLLM paper (arXiv 2309.06180), sections 1 to 4; Sarathi-Serve\n(arXiv 2403.02310); DistServe (arXiv 2401.09670), especially its definition of goodput.\n"
}

for rel, text in FILES.items():
    path = os.path.join(WORK, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(text)
os.chdir(WORK)
print(f"wrote {len(FILES)} files to {WORK}")

In [ ]:
# Installs a vLLM build matching the GPU driver, then foveate itself (~5 min).
!bash scripts/setup_gpu.sh
!foveate env

In [ ]:
# Download weights once, before two GPU workers race for them.
from huggingface_hub import snapshot_download
for m in ["Qwen/Qwen2.5-VL-3B-Instruct"] + (["Qwen/Qwen2.5-VL-7B-Instruct-AWQ"] if RUN_7B else []):
    snapshot_download(m)
print("weights cached")

## Smoke test
24 generated questions through the 3B model on one GPU. If this fails, fix it before the long runs (see README > Troubleshooting). The log line `first-batch accuracy` should be well above chance.

In [ ]:
!foveate grid --dataset synthetic --full --model 3b --probe 2

## 1 · Profile: where does the time go?

Batch-1 Transformers runs with CUDA events around the vision encoder and every decode
step, over frames × resolution on synthetic frames (compute does not depend on pixel
content). Read `theory/01_anatomy_of_vlm_inference.md` alongside.

In [ ]:
if RUN_PROFILE:
    !foveate profile --model 3b --source synthetic

In [ ]:
import json, pandas as pd
R = os.environ["FOVEATE_RESULTS"]
p = f"{R}/profile/3b_synthetic.summary.json"
if os.path.exists(p):
    df = pd.DataFrame(json.load(open(p)))
    cols = ["sweep","frames","res","out_tokens","visual_tokens","load_ms","preprocess_ms","vision_ms","prefill_ms","decode_ms","tpot_ms","e2e_ms","peak_mem_gb","vision_mfu","prefill_mfu"]
    display(df[[c for c in cols if c in df]].round(2))

## 2 · Quality grid on MVBench
Every question through every config once; answers, confidences, tokens and costs are cached under `results/mvbench_<mode>/`. See `theory/02`.

In [ ]:
!foveate download mvbench --tasks {TASKS}

In [ ]:
!foveate grid --dataset mvbench --{MODE} --tasks {TASKS} --model 3b --gpus {GPUS}

In [ ]:
if RUN_7B:
    !foveate grid --dataset mvbench --{MODE} --tasks {TASKS} --model 7b --gpus {GPUS}

## 3 + 4 · Shifted copies, then replay every policy
`shift` reruns the ladder on fog / night / blur frames. `replay` evaluates static configs, learned rules, the confidence cascade, rules+cascade and the oracle on a held-out half, plus the drift study. See `theory/03` and `theory/04`.

In [ ]:
if RUN_SHIFT:
    !foveate shift --dataset mvbench --{MODE} --tasks {TASKS} --model 3b --gpus {GPUS}
    if RUN_7B:
        !foveate shift --dataset mvbench --{MODE} --tasks {TASKS} --model 7b --gpus {GPUS}

In [ ]:
!foveate replay --dataset mvbench --{MODE} --tasks {TASKS}
rep = json.load(open(f"{R}/mvbench_{MODE}/replay/summary.json"))
print("reference:", rep["reference"], "| ladder:", " -> ".join(rep["ladder"]), "| tau:", round(rep["tau"], 3))
display(pd.DataFrame(rep["policies"])[["policy","accuracy","acc_lo","acc_hi","mean_cost","cost_saving","acc_diff","acc_diff_lo","acc_diff_hi","escalation_rate"]].round(3))
display(pd.DataFrame(rep["confidence"]).round(3))

## Driving demo · LingoQA
500 driving-scene questions with free-form answers scored by Lingo-Judge. Google Drive sometimes rate-limits the download; the notebook skips this part if it fails.

In [ ]:
if RUN_LINGOQA:
    ok = subprocess.run(["foveate", "download", "lingoqa"]).returncode == 0
    if ok:
        !foveate grid --dataset lingoqa --{MODE} --model 3b --gpus {GPUS}
        if RUN_7B:
            !foveate grid --dataset lingoqa --{MODE} --model 7b --gpus {GPUS}
        !foveate judge --dataset lingoqa --{MODE}
        !foveate replay --dataset lingoqa --{MODE}
    else:
        print("LingoQA download failed: see README > Troubleshooting")

## 5 · Serving under load
One vLLM server per T4 (3B on GPU 0, 7B-AWQ on GPU 1), the Foveate gateway in front, and an open-loop Poisson load test comparing the cascade with the best fixed config. See `theory/05`.

In [ ]:
if RUN_SERVING:
    os.environ["MODE"], os.environ["TASKS"] = MODE, TASKS
    !bash scripts/serve_and_load.sh

## Report

In [ ]:
!foveate report
from IPython.display import Image, Markdown
display(Markdown(open(f"{R}/REPORT.md").read().split("![")[0][:20000]))
import glob
for f in sorted(glob.glob(f"{R}/figures/*.png")):
    print(os.path.basename(f)); display(Image(filename=f, width=720))

In [ ]:
shutil.make_archive("/kaggle/working/foveate_results", "zip", R) if os.path.isdir("/kaggle") else None
print("download foveate_results.zip from the Output tab; copy results/ into your GitHub repo")